In [ ]:
!pip install -q MDAnalysis MDTraj torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# Cell 1: Common model loader for MethodA / MethodB / MethodC
# Folder layout:
# base_root/
#   MethodA/WT/...
#   MethodA/NAM/...
#   MethodB/WT/...
#   MethodB/NAM/...
#   MethodC/WT/...
#   MethodC/NAM/...
# ============================================================

import os
import json
from dataclasses import dataclass
from typing import Optional, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


base_root = "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS"

METHOD_DIRS = {
    "A": "MethodA",
    "B": "MethodB",
    "C": "MethodC",
    "MethodA": "MethodA",
    "MethodB": "MethodB",
    "MethodC": "MethodC",
}


@dataclass
class RunPaths:
    method: str
    base_dir: str
    data_file: str
    model_file: str
    analysis_dir: str


def build_paths(method, system, rep_id, analysis_subdir=None):
    method_dir = METHOD_DIRS[method]
    system = system.upper()

    base_dir = os.path.join(base_root, method_dir)
    sys_dir = os.path.join(base_dir, system)

    data_file = os.path.join(
        sys_dir,
        f"trajectory_graphs_phi_psi_rmsf_{system}_rep{rep_id}.pt"
    )

    model_file = os.path.join(
        sys_dir,
        f"edgegat_dynamics_best_{system}_rep{rep_id}.pt"
    )

    analysis_dir = os.path.join(base_dir, "ANALYSIS")
    if analysis_subdir is not None:
        analysis_dir = os.path.join(analysis_dir, analysis_subdir, f"{system}_rep{rep_id}")

    os.makedirs(analysis_dir, exist_ok=True)

    return RunPaths(
        method=method_dir,
        base_dir=base_dir,
        data_file=data_file,
        model_file=model_file,
        analysis_dir=analysis_dir,
    )


# ============================================================
# EdgeGAT convolution variants
# ============================================================

class EdgeGATConv_MethodA(MessagePassing):
    """
    Method A:
    Edge attributes enter attention score only.

    alpha_ij = f(x_i, x_j, e_ij)
    message = alpha_ij * x_j
    """

    def __init__(self, in_channels, out_channels, edge_dim=3, heads=1,
                 concat=True, dropout=0.0, bias=True):
        super().__init__(aggr="add", node_dim=0)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        self.heads = heads
        self.concat = concat
        self.dropout = dropout

        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.edge_lin = nn.Linear(edge_dim, heads * out_channels, bias=False)

        self.att = nn.Parameter(torch.empty(heads, 3 * out_channels))
        self.leaky_relu = nn.LeakyReLU(0.2)

        if bias:
            self.bias = nn.Parameter(
                torch.Tensor(heads * out_channels if concat else out_channels)
            )
        else:
            self.register_parameter("bias", None)

        self._last_alpha = None
        self._alpha_mask = None

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.edge_lin.weight)
        nn.init.xavier_uniform_(self.att)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def set_alpha_mask(self, mask: Optional[torch.Tensor]):
        self._alpha_mask = mask

    def clear_alpha_mask(self):
        self._alpha_mask = None

    def get_last_alpha(self):
        if self._last_alpha is None:
            raise RuntimeError("Alpha not cached. Run a forward pass first.")
        return self._last_alpha

    def forward(self, x, edge_index, edge_attr):
        x = self.lin(x)
        edge_attr = self.edge_lin(edge_attr)
        return self.propagate(edge_index=edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr, index):
        x_i = x_i.view(-1, self.heads, self.out_channels)
        x_j = x_j.view(-1, self.heads, self.out_channels)
        edge_attr = edge_attr.view(-1, self.heads, self.out_channels)

        cat = torch.cat([x_i, x_j, edge_attr], dim=-1)
        alpha = torch.einsum("ehc,hc->eh", cat, self.att)
        alpha = self.leaky_relu(alpha)
        alpha = softmax(alpha, index)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        self._last_alpha = alpha.detach()

        if self._alpha_mask is not None:
            m = self._alpha_mask
            if m.ndim == 1:
                m = m.view(-1, 1)
            alpha = alpha * m

        return x_j * alpha.unsqueeze(-1)

    def update(self, aggr_out):
        if self.concat:
            aggr_out = aggr_out.view(-1, self.heads * self.out_channels)
        else:
            aggr_out = aggr_out.mean(dim=1)

        if self.bias is not None:
            aggr_out = aggr_out + self.bias

        return aggr_out


class EdgeGATConv_MethodB(MessagePassing):
    """
    Method B:
    Edge attributes enter attention score and message content.

    alpha_ij = f(x_i, x_j, e_ij)
    message = alpha_ij * (x_j + e_ij)
    """

    def __init__(self, in_channels, out_channels, edge_dim=3, heads=1,
                 concat=True, dropout=0.0, bias=True):
        super().__init__(aggr="add", node_dim=0)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        self.heads = heads
        self.concat = concat
        self.dropout = dropout

        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.edge_lin = nn.Linear(edge_dim, heads * out_channels, bias=False)

        self.att = nn.Parameter(torch.empty(heads, 3 * out_channels))
        self.leaky_relu = nn.LeakyReLU(0.2)

        if bias:
            self.bias = nn.Parameter(
                torch.Tensor(heads * out_channels if concat else out_channels)
            )
        else:
            self.register_parameter("bias", None)

        self._last_alpha = None
        self._alpha_mask = None

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.edge_lin.weight)
        nn.init.xavier_uniform_(self.att)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def set_alpha_mask(self, mask: Optional[torch.Tensor]):
        self._alpha_mask = mask

    def clear_alpha_mask(self):
        self._alpha_mask = None

    def get_last_alpha(self):
        if self._last_alpha is None:
            raise RuntimeError("Alpha not cached. Run a forward pass first.")
        return self._last_alpha

    def forward(self, x, edge_index, edge_attr):
        x = self.lin(x)
        edge_attr = self.edge_lin(edge_attr)
        return self.propagate(edge_index=edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr, index):
        x_i = x_i.view(-1, self.heads, self.out_channels)
        x_j = x_j.view(-1, self.heads, self.out_channels)
        edge_attr = edge_attr.view(-1, self.heads, self.out_channels)

        cat = torch.cat([x_i, x_j, edge_attr], dim=-1)
        alpha = torch.einsum("ehc,hc->eh", cat, self.att)
        alpha = self.leaky_relu(alpha)
        alpha = softmax(alpha, index)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        self._last_alpha = alpha.detach()

        if self._alpha_mask is not None:
            m = self._alpha_mask
            if m.ndim == 1:
                m = m.view(-1, 1)
            alpha = alpha * m

        msg = x_j + edge_attr
        return msg * alpha.unsqueeze(-1)

    def update(self, aggr_out):
        if self.concat:
            aggr_out = aggr_out.view(-1, self.heads * self.out_channels)
        else:
            aggr_out = aggr_out.mean(dim=1)

        if self.bias is not None:
            aggr_out = aggr_out + self.bias

        return aggr_out


class EdgeGATConv_MethodC(MessagePassing):
    """
    Method C:
    Edge attributes enter attention score and gate message channels.

    alpha_ij = f(x_i, x_j, e_ij)
    gate_ij = sigmoid(W_e e_ij)
    message = alpha_ij * (x_j * gate_ij)
    """

    def __init__(self, in_channels, out_channels, edge_dim=3, heads=1,
                 concat=True, dropout=0.0, bias=True):
        super().__init__(aggr="add", node_dim=0)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        self.heads = heads
        self.concat = concat
        self.dropout = dropout

        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)

        self.edge_lin = nn.Linear(edge_dim, heads * out_channels, bias=False)
        self.edge_gate = nn.Linear(edge_dim, heads * out_channels, bias=True)

        self.att = nn.Parameter(torch.empty(heads, 3 * out_channels))
        self.leaky_relu = nn.LeakyReLU(0.2)

        if bias:
            self.bias = nn.Parameter(
                torch.Tensor(heads * out_channels if concat else out_channels)
            )
        else:
            self.register_parameter("bias", None)

        self._last_alpha = None
        self._alpha_mask = None

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.edge_lin.weight)
        nn.init.xavier_uniform_(self.edge_gate.weight)
        nn.init.zeros_(self.edge_gate.bias)
        nn.init.xavier_uniform_(self.att)

        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def set_alpha_mask(self, mask: Optional[torch.Tensor]):
        self._alpha_mask = mask

    def clear_alpha_mask(self):
        self._alpha_mask = None

    def get_last_alpha(self):
        if self._last_alpha is None:
            raise RuntimeError("Alpha not cached. Run a forward pass first.")
        return self._last_alpha

    def forward(self, x, edge_index, edge_attr):
        x = self.lin(x)
        edge_att = self.edge_lin(edge_attr)
        edge_gate = self.edge_gate(edge_attr)

        return self.propagate(
            edge_index=edge_index,
            x=x,
            edge_att=edge_att,
            edge_gate=edge_gate,
        )

    def message(self, x_i, x_j, edge_att, edge_gate, index):
        x_i = x_i.view(-1, self.heads, self.out_channels)
        x_j = x_j.view(-1, self.heads, self.out_channels)
        edge_att = edge_att.view(-1, self.heads, self.out_channels)
        edge_gate = edge_gate.view(-1, self.heads, self.out_channels)

        cat = torch.cat([x_i, x_j, edge_att], dim=-1)
        alpha = torch.einsum("ehc,hc->eh", cat, self.att)
        alpha = self.leaky_relu(alpha)
        alpha = softmax(alpha, index)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        self._last_alpha = alpha.detach()

        if self._alpha_mask is not None:
            m = self._alpha_mask
            if m.ndim == 1:
                m = m.view(-1, 1)
            alpha = alpha * m

        gate = torch.sigmoid(edge_gate)
        msg = x_j * gate

        return msg * alpha.unsqueeze(-1)

    def update(self, aggr_out):
        if self.concat:
            aggr_out = aggr_out.view(-1, self.heads * self.out_channels)
        else:
            aggr_out = aggr_out.mean(dim=1)

        if self.bias is not None:
            aggr_out = aggr_out + self.bias

        return aggr_out


# ============================================================
# Full model
# ============================================================

class EdgeGATDynamics(nn.Module):
    def __init__(
        self,
        method="A",
        in_features=6,
        hidden_dim=128,
        out_features=3,
        heads=4,
        edge_dim=3,
        conv_bias=True,
        model_bias=True,
    ):
        super().__init__()

        method_key = method.replace("Method", "").upper()

        conv_map = {
            "A": EdgeGATConv_MethodA,
            "B": EdgeGATConv_MethodB,
            "C": EdgeGATConv_MethodC,
        }

        if method_key not in conv_map:
            raise ValueError(f"Unknown method: {method}")

        Conv = conv_map[method_key]

        self.method = method_key

        self.gat1 = Conv(in_features, hidden_dim, edge_dim=edge_dim, heads=heads, concat=True, bias=conv_bias)
        self.gat2 = Conv(hidden_dim * heads, hidden_dim, edge_dim=edge_dim, heads=heads, concat=True, bias=conv_bias)
        self.gat3 = Conv(hidden_dim * heads, hidden_dim, edge_dim=edge_dim, heads=heads, concat=True, bias=conv_bias)
        self.gat4 = Conv(hidden_dim * heads, hidden_dim, edge_dim=edge_dim, heads=1, concat=False, bias=conv_bias)

        self.fc = nn.Linear(hidden_dim, out_features)

        if model_bias:
            self.bias = nn.Parameter(torch.ones(1))
        else:
            self.register_parameter("bias", None)

    def clear_masks(self):
        for layer in [self.gat1, self.gat2, self.gat3, self.gat4]:
            layer.clear_alpha_mask()

    def set_masks_all_layers(self, mask_vec):
        for layer in [self.gat1, self.gat2, self.gat3, self.gat4]:
            layer.set_alpha_mask(mask_vec)

    def forward(self, x, edge_index, edge_attr):
        x = self.gat1(x, edge_index, edge_attr)
        x = F.relu(x)

        x = self.gat2(x, edge_index, edge_attr)
        x = F.relu(x)

        x = self.gat3(x, edge_index, edge_attr)
        x = F.relu(x)

        x = self.gat4(x, edge_index, edge_attr)

        out = self.fc(x)

        if self.bias is not None:
            out = out * self.bias

        return out


# ============================================================
# Architecture inference
# ============================================================

@dataclass
class Arch:
    in_features: int
    hidden_dim: int
    out_features: int
    heads: int
    edge_dim: int
    conv_bias: bool
    model_bias: bool


def infer_arch_from_state_dict(sd: Dict[str, torch.Tensor], method="A") -> Arch:
    required = ["fc.weight", "gat1.att", "gat1.lin.weight"]
    for k in required:
        if k not in sd:
            raise KeyError(f"Missing key {k}")

    fc_w = sd["fc.weight"]
    out_features = int(fc_w.shape[0])
    hidden_dim = int(fc_w.shape[1])

    att = sd["gat1.att"]
    heads = int(att.shape[0])

    in_features = int(sd["gat1.lin.weight"].shape[1])

    if "gat1.edge_lin.weight" in sd:
        edge_dim = int(sd["gat1.edge_lin.weight"].shape[1])
    elif "gat1.edge_gate.weight" in sd:
        edge_dim = int(sd["gat1.edge_gate.weight"].shape[1])
    else:
        edge_dim = 3

    conv_bias = "gat1.bias" in sd
    model_bias = ("bias" in sd) and (sd["bias"].numel() == 1)

    return Arch(
        in_features=in_features,
        hidden_dim=hidden_dim,
        out_features=out_features,
        heads=heads,
        edge_dim=edge_dim,
        conv_bias=conv_bias,
        model_bias=model_bias,
    )


# ============================================================
# Loader
# ============================================================

def load_model_and_data(method, system, rep_id, device=None, analysis_subdir=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    device = torch.device(device)

    paths = build_paths(
        method=method,
        system=system,
        rep_id=rep_id,
        analysis_subdir=analysis_subdir,
    )

    if not os.path.exists(paths.data_file):
        raise FileNotFoundError(paths.data_file)

    if not os.path.exists(paths.model_file):
        raise FileNotFoundError(paths.model_file)

    ckpt = torch.load(paths.model_file, map_location="cpu")
    sd = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt

    arch = infer_arch_from_state_dict(sd, method=method)

    model = EdgeGATDynamics(
        method=method,
        in_features=arch.in_features,
        hidden_dim=arch.hidden_dim,
        out_features=arch.out_features,
        heads=arch.heads,
        edge_dim=arch.edge_dim,
        conv_bias=arch.conv_bias,
        model_bias=arch.model_bias,
    ).to(device)

    model.load_state_dict(sd, strict=True)
    model.eval()

    data_list = torch.load(paths.data_file, map_location="cpu", weights_only=False)

    print("Loaded:")
    print(" method     :", paths.method)
    print(" system     :", system.upper())
    print(" replica    :", rep_id)
    print(" model file :", paths.model_file)
    print(" data file  :", paths.data_file)
    print(" analysis   :", paths.analysis_dir)
    print(" arch       :", arch)

    return model, data_list, paths, arch, device


# quick test example:
# model, data_list, paths, arch, device = load_model_and_data("A", "WT", 1, analysis_subdir="RMSD")

In [ ]:
# ============================================================
# Cell 2: RMSD evaluation for MethodA / MethodB / MethodC
# Requires Cell 1 common loader to be already executed
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt


def kabsch_align(P, Q, eps=1e-12):
    """
    Align P onto Q using Kabsch algorithm.
    P, Q: torch tensors [N, 3]
    Returns aligned P.
    """
    P_centroid = P.mean(dim=0, keepdim=True)
    Q_centroid = Q.mean(dim=0, keepdim=True)

    P_centered = P - P_centroid
    Q_centered = Q - Q_centroid

    C = P_centered.T @ Q_centered
    V, S, Wt = torch.linalg.svd(C)

    d = torch.sign(torch.det(V @ Wt))
    D = torch.diag(torch.tensor([1.0, 1.0, d], device=P.device, dtype=P.dtype))

    U = V @ D @ Wt
    P_aligned = P_centered @ U + Q_centroid

    return P_aligned


def kabsch_rmsd(P, Q, eps=1e-12):
    """
    Kabsch-aligned RMSD between P and Q.
    """
    P_aligned = kabsch_align(P, Q, eps=eps)
    diff = P_aligned - Q
    return torch.sqrt((diff * diff).sum(dim=1).mean() + eps)


def to_device_graph(d, device):
    d = d.to(device)
    d.x = d.x.float()
    d.edge_attr = d.edge_attr.float()
    d.edge_index = d.edge_index.long()
    return d


def evaluate_rmsd_one_replica(method, system, rep_id, device="cuda", make_plot=True):
    """
    Computes RMSD metrics for one method/system/replica.
    """

    model, data_cpu, paths, arch, device_obj = load_model_and_data(
        method=method,
        system=system,
        rep_id=rep_id,
        device=device,
        analysis_subdir="RMSD",
    )

    out_dir = paths.analysis_dir
    os.makedirs(out_dir, exist_ok=True)

    n_frames = len(data_cpu)
    if n_frames < 2:
        raise ValueError("Need at least 2 frames for prediction RMSD.")

    ref = data_cpu[0].x[:, :3].float().to(device_obj)

    rows = []

    model.eval()

    with torch.no_grad():
        for t in range(n_frames - 1):
            d0 = to_device_graph(data_cpu[t], device_obj)
            d1 = to_device_graph(data_cpu[t + 1], device_obj)

            pred = model(d0.x, d0.edge_index, d0.edge_attr)
            true_next = d1.x[:, :3]

            pred_vs_true = kabsch_rmsd(pred, true_next).item()
            true_vs_ref = kabsch_rmsd(true_next, ref).item()
            pred_vs_ref = kabsch_rmsd(pred, ref).item()

            rows.append({
                "frame": t,
                "target_frame": t + 1,
                "pred_vs_true_rmsd": pred_vs_true,
                "true_vs_first_true_rmsd": true_vs_ref,
                "pred_vs_first_true_rmsd": pred_vs_ref,
            })

    df = pd.DataFrame(rows)

    csv_path = os.path.join(out_dir, "rmsd_framewise.csv")
    txt_path = os.path.join(out_dir, "rmsd_summary.txt")

    df.to_csv(csv_path, index=False)

    summary = {
        "method": paths.method,
        "system": system.upper(),
        "rep_id": rep_id,
        "n_frames": n_frames,
        "n_predictions": len(df),

        "pred_vs_true_mean": df["pred_vs_true_rmsd"].mean(),
        "pred_vs_true_sd": df["pred_vs_true_rmsd"].std(ddof=1),
        "pred_vs_true_median": df["pred_vs_true_rmsd"].median(),
        "pred_vs_true_min": df["pred_vs_true_rmsd"].min(),
        "pred_vs_true_max": df["pred_vs_true_rmsd"].max(),

        "true_vs_first_mean": df["true_vs_first_true_rmsd"].mean(),
        "true_vs_first_sd": df["true_vs_first_true_rmsd"].std(ddof=1),
        "true_vs_first_median": df["true_vs_first_true_rmsd"].median(),

        "pred_vs_first_mean": df["pred_vs_first_true_rmsd"].mean(),
        "pred_vs_first_sd": df["pred_vs_first_true_rmsd"].std(ddof=1),
        "pred_vs_first_median": df["pred_vs_first_true_rmsd"].median(),
    }

    with open(txt_path, "w") as f:
        f.write(f"RMSD evaluation: {paths.method} {system.upper()} rep{rep_id}\n")
        f.write(f"Model: {paths.model_file}\n")
        f.write(f"Data : {paths.data_file}\n\n")

        for k, v in summary.items():
            f.write(f"{k}: {v}\n")

    pd.DataFrame([summary]).to_csv(
        os.path.join(out_dir, "rmsd_summary.csv"),
        index=False,
    )

    if make_plot:
        plt.figure(figsize=(8, 4.8))
        plt.plot(df["target_frame"], df["pred_vs_true_rmsd"], label="Predicted vs true")
        plt.plot(df["target_frame"], df["true_vs_first_true_rmsd"], label="True vs first true")
        plt.plot(df["target_frame"], df["pred_vs_first_true_rmsd"], label="Predicted vs first true")
        plt.xlabel("Frame")
        plt.ylabel("Kabsch RMSD")
        plt.title(f"{paths.method} {system.upper()} rep{rep_id} RMSD")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "rmsd_plot.png"), dpi=300)
        plt.close()

    print(f"DONE RMSD: {paths.method} {system.upper()} rep{rep_id}")
    print("Output:", out_dir)

    return summary


def run_rmsd_all_methods(methods=("A", "B", "C"), systems=("WT", "NAM"), reps=(1, 2, 3), device="cuda"):
    """
    Runs RMSD for all methods/systems/replicas and saves combined summaries.
    """

    all_rows = []

    for method in methods:
        for system in systems:
            for rep in reps:
                summary = evaluate_rmsd_one_replica(
                    method=method,
                    system=system,
                    rep_id=rep,
                    device=device,
                    make_plot=True,
                )
                all_rows.append(summary)

    all_df = pd.DataFrame(all_rows)

    for method in methods:
        method_dir = os.path.join(base_root, METHOD_DIRS[method], "ANALYSIS", "RMSD")
        os.makedirs(method_dir, exist_ok=True)

        sub = all_df[all_df["method"] == METHOD_DIRS[method]].copy()

        sub.to_csv(
            os.path.join(method_dir, "rmsd_summary_all_replicas.csv"),
            index=False,
        )

        by_system = (
            sub.groupby("system")
            .agg(
                pred_vs_true_mean=("pred_vs_true_mean", "mean"),
                pred_vs_true_sd_across_reps=("pred_vs_true_mean", "std"),
                true_vs_first_mean=("true_vs_first_mean", "mean"),
                true_vs_first_sd_across_reps=("true_vs_first_mean", "std"),
                pred_vs_first_mean=("pred_vs_first_mean", "mean"),
                pred_vs_first_sd_across_reps=("pred_vs_first_mean", "std"),
                n_reps=("rep_id", "count"),
            )
            .reset_index()
        )

        by_system.to_csv(
            os.path.join(method_dir, "rmsd_summary_by_system.csv"),
            index=False,
        )

    # global comparison across all methods
    comparison_dir = os.path.join(base_root, "RMSD_COMPARISON")
    os.makedirs(comparison_dir, exist_ok=True)

    all_df.to_csv(
        os.path.join(comparison_dir, "rmsd_summary_all_methods_all_replicas.csv"),
        index=False,
    )

    global_summary = (
        all_df.groupby(["method", "system"])
        .agg(
            pred_vs_true_mean=("pred_vs_true_mean", "mean"),
            pred_vs_true_sd_across_reps=("pred_vs_true_mean", "std"),
            true_vs_first_mean=("true_vs_first_mean", "mean"),
            true_vs_first_sd_across_reps=("true_vs_first_mean", "std"),
            pred_vs_first_mean=("pred_vs_first_mean", "mean"),
            pred_vs_first_sd_across_reps=("pred_vs_first_mean", "std"),
            n_reps=("rep_id", "count"),
        )
        .reset_index()
    )

    global_summary.to_csv(
        os.path.join(comparison_dir, "rmsd_summary_by_method_system.csv"),
        index=False,
    )

    print("\nALL RMSD DONE")
    print("Comparison output:", comparison_dir)

    return all_df, global_summary

In [ ]:
all_rmsd_df, rmsd_summary = run_rmsd_all_methods(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    device="cuda",
)

Loaded:
 method     : MethodA
 system     : WT
 replica    : 1
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgegat_dynamics_best_WT_rep1.pt
 data file  : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/trajectory_graphs_phi_psi_rmsf_WT_rep1.pt
 analysis   : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/RMSD/WT_rep1
 arch       : Arch(in_features=6, hidden_dim=128, out_features=3, heads=4, edge_dim=3, conv_bias=True, model_bias=True)
DONE RMSD: MethodA WT rep1
Output: /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/RMSD/WT_rep1
Loaded:
 method     : MethodA
 system     : WT
 replica    : 2
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgegat_dynamics_best_WT_rep2.pt
 data file  : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/trajectory_graphs_phi_psi_rmsf_WT_rep2.pt
 analysis   : /content/drive/

In [ ]:
# ============================================================
# Cell 3: Pairwise Distance evaluation for MethodA / MethodB / MethodC
# Requires Cell 1 (Loader) and Cell 2 (Kabsch) to be already executed
# ============================================================

import os
import pandas as pd
import torch

def evaluate_node_distances(method, system, rep_id, node_a, node_b, device="cuda"):
    """
    Computes the distance between node_a and node_b in both true and predicted
    coordinates for a single method/system/replica.
    """

    # Load using your common loader.
    # This automatically puts the output in base_dir/ANALYSIS/NODE_DISTANCES/system_repX
    model, data_cpu, paths, arch, device_obj = load_model_and_data(
        method=method,
        system=system,
        rep_id=rep_id,
        device=device,
        analysis_subdir="NODE_DISTANCES",
    )

    out_dir = paths.analysis_dir
    os.makedirs(out_dir, exist_ok=True)

    n_frames = len(data_cpu)
    if n_frames < 2:
        raise ValueError("Need at least 2 frames for prediction.")

    rows = []
    model.eval()

    with torch.no_grad():
        for t in range(n_frames - 1):
            d0 = to_device_graph(data_cpu[t], device_obj)
            d1 = to_device_graph(data_cpu[t + 1], device_obj)

            # Predict next frame
            pred = model(d0.x, d0.edge_index, d0.edge_attr)
            true_next = d1.x[:, :3]

            # Align pred to true_next
            # (Optional for internal distances, but useful if you export coords)
            pred_aligned = kabsch_align(pred, true_next)

            # Calculate Euclidean distance for the specified nodes
            dist_true = torch.norm(true_next[node_a] - true_next[node_b]).item()
            dist_pred = torch.norm(pred_aligned[node_a] - pred_aligned[node_b]).item()

            # Optional: Distance in the previous frame (t) as a baseline
            dist_current = torch.norm(d0.x[node_a, :3] - d0.x[node_b, :3]).item()

            rows.append({
                "frame": t,
                "target_frame": t + 1,
                "dist_current": dist_current,
                "dist_true": dist_true,
                "dist_pred": dist_pred,
                "absolute_error": abs(dist_pred - dist_true)
            })

    df = pd.DataFrame(rows)

    # Save to text file (tab-separated)
    txt_path = os.path.join(out_dir, f"distance_nodes_{node_a}_{node_b}.txt")
    df.to_csv(txt_path, sep="\t", index=False)

    print(f"DONE Distances: {paths.method} {system.upper()} rep{rep_id} (Nodes {node_a}, {node_b})")
    print(f"Saved to: {txt_path}")

    return df

def run_distances_all_methods(node_a, node_b, methods=("A", "B", "C"), systems=("WT", "NAM"), reps=(1, 2, 3), device="cuda"):
    """
    Runs distance calculations for all methods/systems/replicas and
    compiles a global summary of the mean absolute error.
    """
    summary_rows = []

    for method in methods:
        for system in systems:
            for rep in reps:
                df = evaluate_node_distances(
                    method=method,
                    system=system,
                    rep_id=rep,
                    node_a=node_a,
                    node_b=node_b,
                    device=device
                )

                # Append summary metrics for this replica
                summary_rows.append({
                    "method": method,
                    "system": system,
                    "rep_id": rep,
                    "mean_true_dist": df["dist_true"].mean(),
                    "mean_pred_dist": df["dist_pred"].mean(),
                    "mean_absolute_error": df["absolute_error"].mean()
                })

    # Save a global comparison summary
    comparison_dir = os.path.join(base_root, "DISTANCES_COMPARISON")
    os.makedirs(comparison_dir, exist_ok=True)

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(comparison_dir, f"distance_summary_nodes_{node_a}_{node_b}.csv")
    summary_df.to_csv(summary_csv, index=False)

    print("\nALL DISTANCES DONE")
    print("Comparison output:", comparison_dir)

    return summary_df

In [ ]:
# Assuming your node 50 is index 50, and node 149 is index 149
distance_summary = run_distances_all_methods(
    node_a=49,
    node_b=148,
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    device="cuda"
)

# View the summary of mean absolute errors
print(distance_summary)

Loaded:
 method     : MethodA
 system     : WT
 replica    : 1
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgegat_dynamics_best_WT_rep1.pt
 data file  : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/trajectory_graphs_phi_psi_rmsf_WT_rep1.pt
 analysis   : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/NODE_DISTANCES/WT_rep1
 arch       : Arch(in_features=6, hidden_dim=128, out_features=3, heads=4, edge_dim=3, conv_bias=True, model_bias=True)
DONE Distances: MethodA WT rep1 (Nodes 49, 148)
Saved to: /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/NODE_DISTANCES/WT_rep1/distance_nodes_49_148.txt
Loaded:
 method     : MethodA
 system     : WT
 replica    : 2
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgegat_dynamics_best_WT_rep2.pt
 data file  : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/tr

In [ ]:
# ============================================================
# Cell 4: Node RMSF evaluation for MethodA / MethodB / MethodC
# Requires Cell 1 (Loader) and Cell 2 (Kabsch) to be already executed
# ============================================================

import os
import pandas as pd
import torch

def evaluate_node_rmsf(method, system, rep_id, device="cuda"):
    """
    Computes the per-node Root Mean Square Fluctuation (RMSF) across
    the entire trajectory for both True and Predicted coordinates.
    """

    # Load using your common loader
    model, data_cpu, paths, arch, device_obj = load_model_and_data(
        method=method,
        system=system,
        rep_id=rep_id,
        device=device,
        analysis_subdir="RMSF",
    )

    out_dir = paths.analysis_dir
    os.makedirs(out_dir, exist_ok=True)

    n_frames = len(data_cpu)
    if n_frames < 2:
        raise ValueError("Need at least 2 frames for prediction.")

    # We use Frame 0 as our reference structure for global alignment
    ref = data_cpu[0].x[:, :3].float().to(device_obj)

    true_aligned_coords = []
    pred_aligned_coords = []

    model.eval()

    with torch.no_grad():
        for t in range(n_frames - 1):
            d0 = to_device_graph(data_cpu[t], device_obj)
            d1 = to_device_graph(data_cpu[t + 1], device_obj)

            # Predict next frame
            pred = model(d0.x, d0.edge_index, d0.edge_attr)
            true_next = d1.x[:, :3]

            # Align both True and Pred to Frame 0 to eliminate global translation/rotation
            aligned_true = kabsch_align(true_next, ref)
            aligned_pred = kabsch_align(pred, ref)

            true_aligned_coords.append(aligned_true)
            pred_aligned_coords.append(aligned_pred)

    # Stack into tensors of shape [T, N, 3] where T=time, N=nodes, 3=xyz
    true_tensor = torch.stack(true_aligned_coords)
    pred_tensor = torch.stack(pred_aligned_coords)

    # Calculate mean positions over time: shape [N, 3]
    true_mean = true_tensor.mean(dim=0)
    pred_mean = pred_tensor.mean(dim=0)

    # Calculate RMSF for each node: shape [N]
    # Formula: sqrt( mean( sum( (coord - mean)^2 ) ) )
    rmsf_true = torch.sqrt(((true_tensor - true_mean)**2).sum(dim=-1).mean(dim=0))
    rmsf_pred = torch.sqrt(((pred_tensor - pred_mean)**2).sum(dim=-1).mean(dim=0))

    # Move to CPU for pandas
    rmsf_true = rmsf_true.cpu().numpy()
    rmsf_pred = rmsf_pred.cpu().numpy()

    # Construct DataFrame
    num_nodes = len(rmsf_true)
    df = pd.DataFrame({
        "node_idx": range(num_nodes),
        "rmsf_true": rmsf_true,
        "rmsf_pred": rmsf_pred,
        "absolute_error": abs(rmsf_pred - rmsf_true)
    })

    # Save to text file
    txt_path = os.path.join(out_dir, "rmsf_per_node.txt")
    df.to_csv(txt_path, sep="\t", index=False)

    print(f"DONE RMSF: {paths.method} {system.upper()} rep{rep_id}")
    print(f"Saved to: {txt_path}")

    return df

def run_rmsf_all_methods(methods=("A", "B", "C"), systems=("WT", "NAM"), reps=(1, 2, 3), device="cuda"):
    """
    Runs RMSF calculations for all methods/systems/replicas and
    compiles a global summary of the mean absolute error.
    """
    summary_rows = []

    for method in methods:
        for system in systems:
            for rep in reps:
                df = evaluate_node_rmsf(
                    method=method,
                    system=system,
                    rep_id=rep,
                    device=device
                )

                # Append summary metrics for this replica (averaging error across all nodes)
                summary_rows.append({
                    "method": method,
                    "system": system,
                    "rep_id": rep,
                    "mean_rmsf_true": df["rmsf_true"].mean(),
                    "mean_rmsf_pred": df["rmsf_pred"].mean(),
                    "mean_absolute_error": df["absolute_error"].mean()
                })

    # Save a global comparison summary
    comparison_dir = os.path.join(base_root, "RMSF_COMPARISON")
    os.makedirs(comparison_dir, exist_ok=True)

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(comparison_dir, "rmsf_summary_all_methods.csv")
    summary_df.to_csv(summary_csv, index=False)

    print("\nALL RMSF DONE")
    print("Comparison output:", comparison_dir)

    return summary_df

In [ ]:
rmsf_summary = run_rmsf_all_methods(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    device="cuda"
)

print(rmsf_summary)

Loaded:
 method     : MethodA
 system     : WT
 replica    : 1
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgegat_dynamics_best_WT_rep1.pt
 data file  : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/trajectory_graphs_phi_psi_rmsf_WT_rep1.pt
 analysis   : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/RMSF/WT_rep1
 arch       : Arch(in_features=6, hidden_dim=128, out_features=3, heads=4, edge_dim=3, conv_bias=True, model_bias=True)
DONE RMSF: MethodA WT rep1
Saved to: /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/RMSF/WT_rep1/rmsf_per_node.txt
Loaded:
 method     : MethodA
 system     : WT
 replica    : 2
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgegat_dynamics_best_WT_rep2.pt
 data file  : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/trajectory_graphs_phi_psi_rmsf_WT_rep2.pt
 analysis

In [ ]:
# ============================================================
# Edge faithfulness with K50 half-advantage metric
# No AUC
# Requires:
#   - load_model_and_data()
#   - kabsch_rmsd()
#   - to_device_graph()
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt


def get_all_layer_edge_importance(model):
    a1 = model.gat1.get_last_alpha().sum(dim=1)
    a2 = model.gat2.get_last_alpha().sum(dim=1)
    a3 = model.gat3.get_last_alpha().sum(dim=1)
    a4 = model.gat4.get_last_alpha().sum(dim=1)
    return a1 + a2 + a3 + a4


def make_edge_mask(edge_imp, k_frac):
    E = edge_imp.numel()
    k = max(1, int(round(k_frac * E)))
    _, idx = torch.topk(edge_imp, k=k, largest=True)
    mask = torch.ones(E, device=edge_imp.device, dtype=edge_imp.dtype)
    mask[idx] = 0.0
    return mask


def make_random_edge_mask(E, k_frac, device, dtype, rng):
    k = max(1, int(round(k_frac * E)))
    idx = rng.choice(E, size=k, replace=False)
    mask = torch.ones(E, device=device, dtype=dtype)
    mask[idx] = 0.0
    return mask


def make_blocks(n_frames, block_size):
    blocks = []
    for start in range(0, n_frames - 1, block_size):
        end = min(n_frames - 1, start + block_size)
        if end > start:
            blocks.append((start, end))
    return blocks


def compute_k50_half_advantage(k_pct, advantage):
    """
    K50 = K value after the maximum advantage where advantage falls to 50% of Amax.
    Uses linear interpolation.

    Returns:
      k50, Amax, half_Amax, K_peak, status
    """
    k = np.asarray(k_pct, dtype=float)
    y = np.asarray(advantage, dtype=float)

    if len(k) < 2 or np.all(~np.isfinite(y)):
        return np.nan, np.nan, np.nan, np.nan, "invalid"

    y = np.nan_to_num(y, nan=-np.inf)

    Amax = float(np.max(y))
    peak_idx = int(np.argmax(y))
    K_peak = float(k[peak_idx])

    if Amax <= 0:
        return np.nan, Amax, 0.5 * Amax, K_peak, "no_positive_advantage"

    half = 0.5 * Amax

    # Search only after peak
    for i in range(peak_idx, len(k) - 1):
        y1, y2 = y[i], y[i + 1]
        k1, k2 = k[i], k[i + 1]

        if y1 >= half and y2 <= half:
            if y1 == y2:
                return float(k2), Amax, half, K_peak, "exact_or_flat_crossing"

            frac = (half - y1) / (y2 - y1)
            k50 = k1 + frac * (k2 - k1)
            return float(k50), Amax, half, K_peak, "interpolated"

    return np.nan, Amax, half, K_peak, "not_reached_within_K_range"


def edge_faithfulness_one_replica_k50(
    method,
    system,
    rep_id,
    k_values=[1, 2, 5, 10, 15, 20, 30, 40, 50],
    block_size=500,
    n_random=5,
    device="cuda",
    seed=123,
    make_plots=True,
):
    model, data_cpu, paths, arch, device_obj = load_model_and_data(
        method=method,
        system=system,
        rep_id=rep_id,
        device=device,
        analysis_subdir="EDGE_FAITHFULNESS",
    )

    out_dir = paths.analysis_dir
    os.makedirs(out_dir, exist_ok=True)

    n_frames = len(data_cpu)
    blocks = make_blocks(n_frames, block_size)
    rng = np.random.default_rng(seed + rep_id)

    frame_rows = []
    block_rows = []

    model.eval()

    for k_pct in k_values:
        k_frac = k_pct / 100.0

        for bi, (bs, be) in enumerate(blocks):
            block_records = []

            for t in range(bs, be):
                d0 = to_device_graph(data_cpu[t], device_obj)
                d1 = to_device_graph(data_cpu[t + 1], device_obj)
                true_next = d1.x[:, :3]

                # Baseline
                model.clear_masks()
                with torch.no_grad():
                    pred_base = model(d0.x, d0.edge_index, d0.edge_attr)
                    base_rmsd = kabsch_rmsd(pred_base, true_next).item()

                # Get edge importance
                model.clear_masks()
                with torch.no_grad():
                    _ = model(d0.x, d0.edge_index, d0.edge_attr)
                    edge_imp = get_all_layer_edge_importance(model)

                E = edge_imp.numel()

                # Top-K attention mask
                top_mask = make_edge_mask(edge_imp, k_frac)
                model.set_masks_all_layers(top_mask)

                with torch.no_grad():
                    pred_top = model(d0.x, d0.edge_index, d0.edge_attr)
                    top_rmsd = kabsch_rmsd(pred_top, true_next).item()

                delta_top = top_rmsd - base_rmsd

                # Random-K masks
                random_rmsds = []
                for _ in range(n_random):
                    rand_mask = make_random_edge_mask(
                        E,
                        k_frac,
                        edge_imp.device,
                        edge_imp.dtype,
                        rng,
                    )
                    model.set_masks_all_layers(rand_mask)

                    with torch.no_grad():
                        pred_rand = model(d0.x, d0.edge_index, d0.edge_attr)
                        rand_rmsd = kabsch_rmsd(pred_rand, true_next).item()

                    random_rmsds.append(rand_rmsd)

                model.clear_masks()

                rand_mean = float(np.mean(random_rmsds))
                rand_sd = float(np.std(random_rmsds, ddof=1)) if len(random_rmsds) > 1 else 0.0

                delta_random = rand_mean - base_rmsd
                advantage = delta_top - delta_random

                rec = {
                    "method": paths.method,
                    "system": system.upper(),
                    "rep_id": rep_id,
                    "frame": t,
                    "target_frame": t + 1,
                    "block_id": bi,
                    "K_pct": k_pct,
                    "K_frac": k_frac,
                    "n_edges": E,
                    "base_rmsd": base_rmsd,
                    "top_rmsd": top_rmsd,
                    "random_rmsd_mean": rand_mean,
                    "random_rmsd_sd": rand_sd,
                    "delta_top": delta_top,
                    "delta_random": delta_random,
                    "faithfulness_advantage": advantage,
                }

                frame_rows.append(rec)
                block_records.append(rec)

            bdf = pd.DataFrame(block_records)

            block_rows.append({
                "method": paths.method,
                "system": system.upper(),
                "rep_id": rep_id,
                "block_id": bi,
                "frame_start": bs,
                "frame_end": be,
                "K_pct": k_pct,
                "K_frac": k_frac,
                "base_rmsd_mean": bdf["base_rmsd"].mean(),
                "top_rmsd_mean": bdf["top_rmsd"].mean(),
                "random_rmsd_mean": bdf["random_rmsd_mean"].mean(),
                "delta_top_mean": bdf["delta_top"].mean(),
                "delta_random_mean": bdf["delta_random"].mean(),
                "faithfulness_advantage_mean": bdf["faithfulness_advantage"].mean(),
                "faithfulness_advantage_sd_frames": bdf["faithfulness_advantage"].std(ddof=1),
                "n_frames_block": len(bdf),
            })

        print(f"[DONE] {paths.method} {system.upper()} rep{rep_id} K={k_pct}%")

    frame_df = pd.DataFrame(frame_rows)
    block_df = pd.DataFrame(block_rows)

    frame_df.to_csv(os.path.join(out_dir, "edge_faithfulness_framewise.csv"), index=False)
    block_df.to_csv(os.path.join(out_dir, "edge_faithfulness_blockwise.csv"), index=False)

    # Replica-level curve: mean across blocks
    curve_rows = []

    for k_pct, g in block_df.groupby("K_pct"):
        curve_rows.append({
            "method": paths.method,
            "system": system.upper(),
            "rep_id": rep_id,
            "K_pct": float(k_pct),
            "K_frac": float(g["K_frac"].iloc[0]),

            "delta_top_mean": g["delta_top_mean"].mean(),
            "delta_top_sd_blocks": g["delta_top_mean"].std(ddof=1),

            "delta_random_mean": g["delta_random_mean"].mean(),
            "delta_random_sd_blocks": g["delta_random_mean"].std(ddof=1),

            "faithfulness_advantage_mean": g["faithfulness_advantage_mean"].mean(),
            "faithfulness_advantage_sd_blocks": g["faithfulness_advantage_mean"].std(ddof=1),

            "n_blocks": len(g),
        })

    curve_df = pd.DataFrame(curve_rows).sort_values("K_pct")
    curve_df.to_csv(os.path.join(out_dir, "edge_faithfulness_curve_by_K.csv"), index=False)

    # K50 from replica-level curve
    k50, Amax, half_Amax, K_peak, status = compute_k50_half_advantage(
        curve_df["K_pct"].values,
        curve_df["faithfulness_advantage_mean"].values,
    )

    k50_df = pd.DataFrame([{
        "method": paths.method,
        "system": system.upper(),
        "rep_id": rep_id,
        "K50_half_advantage": k50,
        "Amax_advantage": Amax,
        "half_Amax": half_Amax,
        "K_peak": K_peak,
        "K50_status": status,
        "n_random": n_random,
        "block_size": block_size,
        "K_values": ",".join(map(str, k_values)),
    }])

    k50_df.to_csv(os.path.join(out_dir, "edge_faithfulness_K50_summary.csv"), index=False)

    with open(os.path.join(out_dir, "summary.txt"), "w") as f:
        f.write(f"Edge faithfulness K50: {paths.method} {system.upper()} rep{rep_id}\n")
        f.write(f"Model: {paths.model_file}\n")
        f.write(f"Data : {paths.data_file}\n\n")
        f.write("Definitions:\n")
        f.write("delta_top = RMSD(top-K masked) - RMSD(baseline)\n")
        f.write("delta_random = RMSD(random-K masked) - RMSD(baseline)\n")
        f.write("faithfulness_advantage = delta_top - delta_random\n")
        f.write("K50 = K after peak where advantage falls to 50% of maximum advantage\n\n")
        f.write(f"Amax_advantage: {Amax}\n")
        f.write(f"half_Amax: {half_Amax}\n")
        f.write(f"K_peak: {K_peak}\n")
        f.write(f"K50_half_advantage: {k50}\n")
        f.write(f"K50_status: {status}\n")

    if make_plots:
        x = curve_df["K_pct"].values
        y = curve_df["faithfulness_advantage_mean"].values
        yerr = curve_df["faithfulness_advantage_sd_blocks"].values

        plt.figure(figsize=(7, 4.8))
        plt.errorbar(x, y, yerr=yerr, marker="o", capsize=3, label="Advantage")

        if np.isfinite(Amax):
            plt.axhline(half_Amax, linestyle="--", linewidth=1, label="50% Amax")
        if np.isfinite(k50):
            plt.axvline(k50, linestyle="--", linewidth=1, label=f"K50={k50:.2f}%")

        plt.axhline(0, linewidth=1)
        plt.xlabel("Masked top-attention edges (%)")
        plt.ylabel("Faithfulness advantage ΔRMSD")
        plt.title(f"{paths.method} {system.upper()} rep{rep_id}: edge K50")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "edge_faithfulness_K50_curve.png"), dpi=300)
        plt.close()

    print(f"EDGE FAITHFULNESS K50 DONE: {paths.method} {system.upper()} rep{rep_id}")
    return k50_df, curve_df


def run_edge_faithfulness_k50_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    k_values=[1, 2, 5, 10, 15, 20, 30, 40, 50],
    block_size=500,
    n_random=5,
    device="cuda",
):
    all_k50 = []
    all_curves = []

    for method in methods:
        for system in systems:
            for rep in reps:
                k50_df, curve_df = edge_faithfulness_one_replica_k50(
                    method=method,
                    system=system,
                    rep_id=rep,
                    k_values=k_values,
                    block_size=block_size,
                    n_random=n_random,
                    device=device,
                    seed=123,
                    make_plots=True,
                )
                all_k50.append(k50_df)
                all_curves.append(curve_df)

    k50_all = pd.concat(all_k50, ignore_index=True)
    curves_all = pd.concat(all_curves, ignore_index=True)

    comparison_dir = os.path.join(base_root, "EDGE_FAITHFULNESS_K50_COMPARISON")
    os.makedirs(comparison_dir, exist_ok=True)

    k50_all.to_csv(
        os.path.join(comparison_dir, "edge_faithfulness_K50_all_methods_replicas.csv"),
        index=False,
    )

    curves_all.to_csv(
        os.path.join(comparison_dir, "edge_faithfulness_curves_all_methods_replicas.csv"),
        index=False,
    )

    # System-level K50 summary across replicas
    k50_system = (
        k50_all.groupby(["method", "system"])
        .agg(
            K50_mean=("K50_half_advantage", "mean"),
            K50_sd=("K50_half_advantage", "std"),
            Amax_mean=("Amax_advantage", "mean"),
            Amax_sd=("Amax_advantage", "std"),
            K_peak_mean=("K_peak", "mean"),
            K_peak_sd=("K_peak", "std"),
            n_reps=("rep_id", "count"),
        )
        .reset_index()
    )

    k50_system.to_csv(
        os.path.join(comparison_dir, "edge_faithfulness_K50_by_method_system.csv"),
        index=False,
    )

    # System-level curves across replicas
    curve_summary = (
        curves_all.groupby(["method", "system", "K_pct"])
        .agg(
            advantage_mean=("faithfulness_advantage_mean", "mean"),
            advantage_sd_reps=("faithfulness_advantage_mean", "std"),
            delta_top_mean=("delta_top_mean", "mean"),
            delta_top_sd_reps=("delta_top_mean", "std"),
            delta_random_mean=("delta_random_mean", "mean"),
            delta_random_sd_reps=("delta_random_mean", "std"),
            n_reps=("rep_id", "count"),
        )
        .reset_index()
    )

    curve_summary.to_csv(
        os.path.join(comparison_dir, "edge_faithfulness_curve_by_method_system.csv"),
        index=False,
    )

    # Plot system-level curves
    for method in sorted(curve_summary["method"].unique()):
        for system in sorted(curve_summary["system"].unique()):
            d = curve_summary[
                (curve_summary["method"] == method) &
                (curve_summary["system"] == system)
            ].sort_values("K_pct")

            if d.empty:
                continue

            plt.figure(figsize=(7, 4.8))
            plt.errorbar(
                d["K_pct"],
                d["advantage_mean"],
                yerr=d["advantage_sd_reps"],
                marker="o",
                capsize=3,
            )
            plt.axhline(0, linewidth=1)
            plt.xlabel("Masked top-attention edges (%)")
            plt.ylabel("Faithfulness advantage ΔRMSD")
            plt.title(f"{method} {system}: system edge faithfulness")
            plt.tight_layout()
            plt.savefig(
                os.path.join(comparison_dir, f"{method}_{system}_system_edge_faithfulness_curve.png"),
                dpi=300,
            )
            plt.close()

    # Plot K50 comparison
    plt.figure(figsize=(8, 4.8))
    labels = []
    means = []
    sds = []

    for _, row in k50_system.iterrows():
        labels.append(f"{row['method']}_{row['system']}")
        means.append(row["K50_mean"])
        sds.append(row["K50_sd"])

    x = np.arange(len(labels))
    plt.bar(x, means)
    plt.errorbar(x, means, yerr=sds, fmt="none", capsize=3)
    plt.xticks(x, labels, rotation=45, ha="right")
    plt.ylabel("K50 half-advantage (%)")
    plt.title("Edge faithfulness K50 comparison")
    plt.tight_layout()
    plt.savefig(os.path.join(comparison_dir, "edge_faithfulness_K50_comparison.png"), dpi=300)
    plt.close()

    print("\nALL EDGE FAITHFULNESS K50 DONE")
    print("Comparison output:", comparison_dir)

    return k50_all, k50_system, curves_all, curve_summary

In [ ]:
edge_k50_all, edge_k50_system, edge_curves_all, edge_curve_summary = run_edge_faithfulness_k50_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    k_values=[1, 2, 5, 10, 15, 20, 30, 40, 50],
    block_size=500,
    n_random=5,
    device="cuda",
)

Loaded:
 method     : MethodA
 system     : WT
 replica    : 1
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgegat_dynamics_best_WT_rep1.pt
 data file  : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/trajectory_graphs_phi_psi_rmsf_WT_rep1.pt
 analysis   : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/EDGE_FAITHFULNESS/WT_rep1
 arch       : Arch(in_features=6, hidden_dim=128, out_features=3, heads=4, edge_dim=3, conv_bias=True, model_bias=True)
[DONE] MethodA WT rep1 K=1%
[DONE] MethodA WT rep1 K=2%
[DONE] MethodA WT rep1 K=5%
[DONE] MethodA WT rep1 K=10%
[DONE] MethodA WT rep1 K=15%
[DONE] MethodA WT rep1 K=20%
[DONE] MethodA WT rep1 K=30%
[DONE] MethodA WT rep1 K=40%
[DONE] MethodA WT rep1 K=50%
EDGE FAITHFULNESS K50 DONE: MethodA WT rep1
Loaded:
 method     : MethodA
 system     : WT
 replica    : 2
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/e

In [ ]:
# ============================================================
# Node faithfulness with Amax and K_peak
# Requires:
#   - load_model_and_data()
#   - kabsch_rmsd()
#   - to_device_graph()
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt


def get_all_layer_edge_importance(model):
    a1 = model.gat1.get_last_alpha().sum(dim=1)
    a2 = model.gat2.get_last_alpha().sum(dim=1)
    a3 = model.gat3.get_last_alpha().sum(dim=1)
    a4 = model.gat4.get_last_alpha().sum(dim=1)
    return a1 + a2 + a3 + a4


def get_node_importance_from_edge_attention(model, edge_index, num_nodes):
    """
    Node importance = sum of incident all-layer/all-head edge attention.
    """
    edge_imp = get_all_layer_edge_importance(model)

    src, dst = edge_index
    node_imp = torch.zeros(num_nodes, device=edge_imp.device, dtype=edge_imp.dtype)

    node_imp.index_add_(0, src, edge_imp)
    node_imp.index_add_(0, dst, edge_imp)

    return node_imp


def make_top_node_mask(node_imp, k_frac):
    """
    Node mask: 0 for top-K important nodes, 1 for others.
    """
    N = node_imp.numel()
    k = max(1, int(round(k_frac * N)))

    _, idx = torch.topk(node_imp, k=k, largest=True)

    mask = torch.ones(N, device=node_imp.device, dtype=node_imp.dtype)
    mask[idx] = 0.0

    return mask


def make_random_node_mask(N, k_frac, device, dtype, rng):
    k = max(1, int(round(k_frac * N)))
    idx = rng.choice(N, size=k, replace=False)

    mask = torch.ones(N, device=device, dtype=dtype)
    mask[idx] = 0.0

    return mask


def edge_mask_from_node_mask(node_mask, edge_index):
    """
    Edge remains active only if both endpoint nodes are active.
    """
    src, dst = edge_index
    return node_mask[src] * node_mask[dst]


def make_blocks(n_frames, block_size):
    blocks = []
    for start in range(0, n_frames - 1, block_size):
        end = min(n_frames - 1, start + block_size)
        if end > start:
            blocks.append((start, end))
    return blocks


def compute_amax_kpeak(k_pct, advantage):
    """
    Amax = maximum faithfulness advantage
    K_peak = K where Amax occurs
    """
    k = np.asarray(k_pct, dtype=float)
    y = np.asarray(advantage, dtype=float)

    if len(k) == 0 or np.all(~np.isfinite(y)):
        return np.nan, np.nan, "invalid"

    y_clean = np.nan_to_num(y, nan=-np.inf)
    idx = int(np.argmax(y_clean))

    Amax = float(y_clean[idx])
    K_peak = float(k[idx])

    if Amax <= 0:
        return Amax, K_peak, "no_positive_advantage"

    return Amax, K_peak, "ok"


def node_faithfulness_one_replica_amax(
    method,
    system,
    rep_id,
    k_values=[1, 2, 5, 10, 15, 20, 30, 40, 50],
    block_size=500,
    n_random=5,
    device="cuda",
    seed=123,
    make_plots=True,
):
    model, data_cpu, paths, arch, device_obj = load_model_and_data(
        method=method,
        system=system,
        rep_id=rep_id,
        device=device,
        analysis_subdir="NODE_FAITHFULNESS",
    )

    out_dir = paths.analysis_dir
    os.makedirs(out_dir, exist_ok=True)

    n_frames = len(data_cpu)
    blocks = make_blocks(n_frames, block_size)
    rng = np.random.default_rng(seed + rep_id)

    frame_rows = []
    block_rows = []

    model.eval()

    for k_pct in k_values:
        k_frac = k_pct / 100.0

        for bi, (bs, be) in enumerate(blocks):
            block_records = []

            for t in range(bs, be):
                d0 = to_device_graph(data_cpu[t], device_obj)
                d1 = to_device_graph(data_cpu[t + 1], device_obj)
                true_next = d1.x[:, :3]

                # Baseline
                model.clear_masks()
                with torch.no_grad():
                    pred_base = model(d0.x, d0.edge_index, d0.edge_attr)
                    base_rmsd = kabsch_rmsd(pred_base, true_next).item()

                # Get node importance from unmasked attention
                model.clear_masks()
                with torch.no_grad():
                    _ = model(d0.x, d0.edge_index, d0.edge_attr)
                    node_imp = get_node_importance_from_edge_attention(
                        model,
                        d0.edge_index,
                        num_nodes=d0.x.shape[0],
                    )

                N = node_imp.numel()

                # Top-K node mask -> edge alpha mask
                top_node_mask = make_top_node_mask(node_imp, k_frac)
                top_edge_mask = edge_mask_from_node_mask(top_node_mask, d0.edge_index)

                model.set_masks_all_layers(top_edge_mask)

                with torch.no_grad():
                    pred_top = model(d0.x, d0.edge_index, d0.edge_attr)
                    top_rmsd = kabsch_rmsd(pred_top, true_next).item()

                delta_top = top_rmsd - base_rmsd

                # Random node masks
                random_rmsds = []

                for _ in range(n_random):
                    rand_node_mask = make_random_node_mask(
                        N,
                        k_frac,
                        node_imp.device,
                        node_imp.dtype,
                        rng,
                    )
                    rand_edge_mask = edge_mask_from_node_mask(rand_node_mask, d0.edge_index)

                    model.set_masks_all_layers(rand_edge_mask)

                    with torch.no_grad():
                        pred_rand = model(d0.x, d0.edge_index, d0.edge_attr)
                        rand_rmsd = kabsch_rmsd(pred_rand, true_next).item()

                    random_rmsds.append(rand_rmsd)

                model.clear_masks()

                rand_mean = float(np.mean(random_rmsds))
                rand_sd = float(np.std(random_rmsds, ddof=1)) if len(random_rmsds) > 1 else 0.0

                delta_random = rand_mean - base_rmsd
                advantage = delta_top - delta_random

                rec = {
                    "method": paths.method,
                    "system": system.upper(),
                    "rep_id": rep_id,
                    "frame": t,
                    "target_frame": t + 1,
                    "block_id": bi,
                    "K_pct": k_pct,
                    "K_frac": k_frac,
                    "n_nodes": N,
                    "base_rmsd": base_rmsd,
                    "top_rmsd": top_rmsd,
                    "random_rmsd_mean": rand_mean,
                    "random_rmsd_sd": rand_sd,
                    "delta_top": delta_top,
                    "delta_random": delta_random,
                    "faithfulness_advantage": advantage,
                }

                frame_rows.append(rec)
                block_records.append(rec)

            bdf = pd.DataFrame(block_records)

            block_rows.append({
                "method": paths.method,
                "system": system.upper(),
                "rep_id": rep_id,
                "block_id": bi,
                "frame_start": bs,
                "frame_end": be,
                "K_pct": k_pct,
                "K_frac": k_frac,
                "base_rmsd_mean": bdf["base_rmsd"].mean(),
                "top_rmsd_mean": bdf["top_rmsd"].mean(),
                "random_rmsd_mean": bdf["random_rmsd_mean"].mean(),
                "delta_top_mean": bdf["delta_top"].mean(),
                "delta_random_mean": bdf["delta_random"].mean(),
                "faithfulness_advantage_mean": bdf["faithfulness_advantage"].mean(),
                "faithfulness_advantage_sd_frames": bdf["faithfulness_advantage"].std(ddof=1),
                "n_frames_block": len(bdf),
            })

        print(f"[DONE] {paths.method} {system.upper()} rep{rep_id} K={k_pct}%")

    frame_df = pd.DataFrame(frame_rows)
    block_df = pd.DataFrame(block_rows)

    frame_df.to_csv(os.path.join(out_dir, "node_faithfulness_framewise.csv"), index=False)
    block_df.to_csv(os.path.join(out_dir, "node_faithfulness_blockwise.csv"), index=False)

    # Replica-level curve: mean across blocks
    curve_rows = []

    for k_pct, g in block_df.groupby("K_pct"):
        curve_rows.append({
            "method": paths.method,
            "system": system.upper(),
            "rep_id": rep_id,
            "K_pct": float(k_pct),
            "K_frac": float(g["K_frac"].iloc[0]),

            "delta_top_mean": g["delta_top_mean"].mean(),
            "delta_top_sd_blocks": g["delta_top_mean"].std(ddof=1),

            "delta_random_mean": g["delta_random_mean"].mean(),
            "delta_random_sd_blocks": g["delta_random_mean"].std(ddof=1),

            "faithfulness_advantage_mean": g["faithfulness_advantage_mean"].mean(),
            "faithfulness_advantage_sd_blocks": g["faithfulness_advantage_mean"].std(ddof=1),

            "n_blocks": len(g),
        })

    curve_df = pd.DataFrame(curve_rows).sort_values("K_pct")
    curve_df.to_csv(os.path.join(out_dir, "node_faithfulness_curve_by_K.csv"), index=False)

    # Amax and K_peak
    Amax, K_peak, status = compute_amax_kpeak(
        curve_df["K_pct"].values,
        curve_df["faithfulness_advantage_mean"].values,
    )

    amax_df = pd.DataFrame([{
        "method": paths.method,
        "system": system.upper(),
        "rep_id": rep_id,
        "Amax_advantage": Amax,
        "K_peak": K_peak,
        "Amax_status": status,
        "n_random": n_random,
        "block_size": block_size,
        "K_values": ",".join(map(str, k_values)),
    }])

    amax_df.to_csv(os.path.join(out_dir, "node_faithfulness_Amax_Kpeak_summary.csv"), index=False)

    with open(os.path.join(out_dir, "summary.txt"), "w") as f:
        f.write(f"Node faithfulness Amax/Kpeak: {paths.method} {system.upper()} rep{rep_id}\n")
        f.write(f"Model: {paths.model_file}\n")
        f.write(f"Data : {paths.data_file}\n\n")
        f.write("Definitions:\n")
        f.write("delta_top = RMSD(top-K node masked) - RMSD(baseline)\n")
        f.write("delta_random = RMSD(random-K node masked) - RMSD(baseline)\n")
        f.write("faithfulness_advantage = delta_top - delta_random\n")
        f.write("Amax = maximum faithfulness advantage over K\n")
        f.write("K_peak = K where Amax occurs\n\n")
        f.write(f"Amax_advantage: {Amax}\n")
        f.write(f"K_peak: {K_peak}\n")
        f.write(f"Amax_status: {status}\n")

    if make_plots:
        x = curve_df["K_pct"].values
        y = curve_df["faithfulness_advantage_mean"].values
        yerr = curve_df["faithfulness_advantage_sd_blocks"].values

        plt.figure(figsize=(7, 4.8))
        plt.errorbar(x, y, yerr=yerr, marker="o", capsize=3, label="Advantage")

        if np.isfinite(K_peak):
            plt.axvline(K_peak, linestyle="--", linewidth=1, label=f"K_peak={K_peak:.1f}%")
        if np.isfinite(Amax):
            plt.axhline(Amax, linestyle="--", linewidth=1, label=f"Amax={Amax:.3f}")

        plt.axhline(0, linewidth=1)
        plt.xlabel("Masked top-attention nodes (%)")
        plt.ylabel("Node faithfulness advantage ΔRMSD")
        plt.title(f"{paths.method} {system.upper()} rep{rep_id}: node Amax/Kpeak")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "node_faithfulness_Amax_Kpeak_curve.png"), dpi=300)
        plt.close()

    print(f"NODE FAITHFULNESS DONE: {paths.method} {system.upper()} rep{rep_id}")
    return amax_df, curve_df


def run_node_faithfulness_amax_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    k_values=[1, 2, 5, 10, 15, 20, 30, 40, 50],
    block_size=500,
    n_random=5,
    device="cuda",
):
    all_amax = []
    all_curves = []

    for method in methods:
        for system in systems:
            for rep in reps:
                amax_df, curve_df = node_faithfulness_one_replica_amax(
                    method=method,
                    system=system,
                    rep_id=rep,
                    k_values=k_values,
                    block_size=block_size,
                    n_random=n_random,
                    device=device,
                    seed=123,
                    make_plots=True,
                )
                all_amax.append(amax_df)
                all_curves.append(curve_df)

    amax_all = pd.concat(all_amax, ignore_index=True)
    curves_all = pd.concat(all_curves, ignore_index=True)

    comparison_dir = os.path.join(base_root, "NODE_FAITHFULNESS_AMAX_COMPARISON")
    os.makedirs(comparison_dir, exist_ok=True)

    amax_all.to_csv(
        os.path.join(comparison_dir, "node_faithfulness_Amax_Kpeak_all_methods_replicas.csv"),
        index=False,
    )

    curves_all.to_csv(
        os.path.join(comparison_dir, "node_faithfulness_curves_all_methods_replicas.csv"),
        index=False,
    )

    # System-level summary across replicas
    amax_system = (
        amax_all.groupby(["method", "system"])
        .agg(
            Amax_mean=("Amax_advantage", "mean"),
            Amax_sd=("Amax_advantage", "std"),
            K_peak_mean=("K_peak", "mean"),
            K_peak_sd=("K_peak", "std"),
            n_reps=("rep_id", "count"),
        )
        .reset_index()
    )

    amax_system.to_csv(
        os.path.join(comparison_dir, "node_faithfulness_Amax_Kpeak_by_method_system.csv"),
        index=False,
    )

    # System-level curves across replicas
    curve_summary = (
        curves_all.groupby(["method", "system", "K_pct"])
        .agg(
            advantage_mean=("faithfulness_advantage_mean", "mean"),
            advantage_sd_reps=("faithfulness_advantage_mean", "std"),
            delta_top_mean=("delta_top_mean", "mean"),
            delta_top_sd_reps=("delta_top_mean", "std"),
            delta_random_mean=("delta_random_mean", "mean"),
            delta_random_sd_reps=("delta_random_mean", "std"),
            n_reps=("rep_id", "count"),
        )
        .reset_index()
    )

    curve_summary.to_csv(
        os.path.join(comparison_dir, "node_faithfulness_curve_by_method_system.csv"),
        index=False,
    )

    # Plot system-level curves
    for method in sorted(curve_summary["method"].unique()):
        for system in sorted(curve_summary["system"].unique()):
            d = curve_summary[
                (curve_summary["method"] == method) &
                (curve_summary["system"] == system)
            ].sort_values("K_pct")

            if d.empty:
                continue

            plt.figure(figsize=(7, 4.8))
            plt.errorbar(
                d["K_pct"],
                d["advantage_mean"],
                yerr=d["advantage_sd_reps"],
                marker="o",
                capsize=3,
            )
            plt.axhline(0, linewidth=1)
            plt.xlabel("Masked top-attention nodes (%)")
            plt.ylabel("Node faithfulness advantage ΔRMSD")
            plt.title(f"{method} {system}: system node faithfulness")
            plt.tight_layout()
            plt.savefig(
                os.path.join(comparison_dir, f"{method}_{system}_system_node_faithfulness_curve.png"),
                dpi=300,
            )
            plt.close()

    # Plot Amax comparison
    plt.figure(figsize=(8, 4.8))
    labels = []
    means = []
    sds = []

    for _, row in amax_system.iterrows():
        labels.append(f"{row['method']}_{row['system']}")
        means.append(row["Amax_mean"])
        sds.append(row["Amax_sd"])

    x = np.arange(len(labels))
    plt.bar(x, means)
    plt.errorbar(x, means, yerr=sds, fmt="none", capsize=3)
    plt.xticks(x, labels, rotation=45, ha="right")
    plt.ylabel("Node Amax advantage")
    plt.title("Node faithfulness Amax comparison")
    plt.tight_layout()
    plt.savefig(os.path.join(comparison_dir, "node_faithfulness_Amax_comparison.png"), dpi=300)
    plt.close()

    # Plot K_peak comparison
    plt.figure(figsize=(8, 4.8))
    labels = []
    means = []
    sds = []

    for _, row in amax_system.iterrows():
        labels.append(f"{row['method']}_{row['system']}")
        means.append(row["K_peak_mean"])
        sds.append(row["K_peak_sd"])

    x = np.arange(len(labels))
    plt.bar(x, means)
    plt.errorbar(x, means, yerr=sds, fmt="none", capsize=3)
    plt.xticks(x, labels, rotation=45, ha="right")
    plt.ylabel("K_peak (%)")
    plt.title("Node faithfulness K_peak comparison")
    plt.tight_layout()
    plt.savefig(os.path.join(comparison_dir, "node_faithfulness_Kpeak_comparison.png"), dpi=300)
    plt.close()

    print("\nALL NODE FAITHFULNESS DONE")
    print("Comparison output:", comparison_dir)

    return amax_all, amax_system, curves_all, curve_summary

In [ ]:
node_amax_all, node_amax_system, node_curves_all, node_curve_summary = run_node_faithfulness_amax_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    k_values=[1, 2, 5, 10, 15, 20, 30, 40, 50],
    block_size=500,
    n_random=5,
    device="cuda",
)

Loaded:
 method     : MethodA
 system     : WT
 replica    : 1
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgegat_dynamics_best_WT_rep1.pt
 data file  : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/trajectory_graphs_phi_psi_rmsf_WT_rep1.pt
 analysis   : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/NODE_FAITHFULNESS/WT_rep1
 arch       : Arch(in_features=6, hidden_dim=128, out_features=3, heads=4, edge_dim=3, conv_bias=True, model_bias=True)
[DONE] MethodA WT rep1 K=1%
[DONE] MethodA WT rep1 K=2%
[DONE] MethodA WT rep1 K=5%
[DONE] MethodA WT rep1 K=10%
[DONE] MethodA WT rep1 K=15%
[DONE] MethodA WT rep1 K=20%
[DONE] MethodA WT rep1 K=30%
[DONE] MethodA WT rep1 K=40%
[DONE] MethodA WT rep1 K=50%
NODE FAITHFULNESS DONE: MethodA WT rep1
Loaded:
 method     : MethodA
 system     : WT
 replica    : 2
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgeg

In [ ]:
# ============================================================
# Cell 4A: Node weights by block for MethodA / MethodB / MethodC
# Requires:
#   - common model loader cell
#   - to_device_graph()
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt


def extract_edge_importance_variant(model, variant="all_sum"):
    """
    Returns edge importance [E] from cached attention.
    Current default: all_sum.
    Future-compatible variants can be added here.
    """
    a1 = model.gat1.get_last_alpha()  # [E,H]
    a2 = model.gat2.get_last_alpha()
    a3 = model.gat3.get_last_alpha()
    a4 = model.gat4.get_last_alpha()  # [E,1]

    if variant == "all_sum":
        return a1.sum(dim=1) + a2.sum(dim=1) + a3.sum(dim=1) + a4.sum(dim=1)

    elif variant == "all_mean":
        return torch.cat([a1, a2, a3, a4], dim=1).mean(dim=1)

    elif variant == "layer1_mean":
        return a1.mean(dim=1)

    elif variant == "layer2_mean":
        return a2.mean(dim=1)

    elif variant == "layer3_mean":
        return a3.mean(dim=1)

    elif variant == "layer4_mean":
        return a4.mean(dim=1)

    else:
        raise ValueError(f"Unknown variant: {variant}")


def edge_to_node_weight(edge_imp, edge_index, num_nodes):
    """
    Node weight = sum of incident edge importance.
    """
    src, dst = edge_index
    node_w = torch.zeros(num_nodes, device=edge_imp.device, dtype=edge_imp.dtype)

    node_w.index_add_(0, src, edge_imp)
    node_w.index_add_(0, dst, edge_imp)

    return node_w


def make_blocks_nodeweights(n_frames, block_size):
    blocks = []
    for start in range(0, n_frames, block_size):
        end = min(n_frames, start + block_size)
        if end > start:
            blocks.append((start, end))
    return blocks


def compute_node_weights_one_replica(
    method,
    system,
    rep_id,
    variant="all_sum",
    block_size=500,
    device="cuda",
    make_plot=True,
):
    model, data_cpu, paths, arch, device_obj = load_model_and_data(
        method=method,
        system=system,
        rep_id=rep_id,
        device=device,
        analysis_subdir=os.path.join("NODE_WEIGHTS", variant),
    )

    out_dir = paths.analysis_dir
    os.makedirs(out_dir, exist_ok=True)

    n_frames = len(data_cpu)
    blocks = make_blocks_nodeweights(n_frames, block_size)

    frame_rows = []
    block_rows = []

    model.eval()

    with torch.no_grad():
        for bi, (bs, be) in enumerate(blocks):
            block_node_values = []

            for t in range(bs, be):
                d = to_device_graph(data_cpu[t], device_obj)

                model.clear_masks()
                _ = model(d.x, d.edge_index, d.edge_attr)

                edge_imp = extract_edge_importance_variant(model, variant=variant)
                node_w = edge_to_node_weight(
                    edge_imp=edge_imp,
                    edge_index=d.edge_index,
                    num_nodes=d.x.shape[0],
                )

                node_w_np = node_w.detach().cpu().numpy()
                block_node_values.append(node_w_np)

                for node_id, val in enumerate(node_w_np):
                    frame_rows.append({
                        "method": paths.method,
                        "system": system.upper(),
                        "rep_id": rep_id,
                        "variant": variant,
                        "frame": t,
                        "block_id": bi,
                        "node_id": node_id,
                        "residue": node_id + 1,
                        "node_weight": float(val),
                    })

            block_arr = np.asarray(block_node_values, dtype=float)  # [frames, nodes]
            block_mean = block_arr.mean(axis=0)
            block_sd = block_arr.std(axis=0, ddof=1) if block_arr.shape[0] > 1 else np.zeros_like(block_mean)

            for node_id, val in enumerate(block_mean):
                block_rows.append({
                    "method": paths.method,
                    "system": system.upper(),
                    "rep_id": rep_id,
                    "variant": variant,
                    "block_id": bi,
                    "frame_start": bs,
                    "frame_end": be,
                    "node_id": node_id,
                    "residue": node_id + 1,
                    "node_weight_block_mean": float(val),
                    "node_weight_block_sd_frames": float(block_sd[node_id]),
                    "n_frames_block": int(block_arr.shape[0]),
                })

            print(f"[DONE] {paths.method} {system.upper()} rep{rep_id} block {bi+1}/{len(blocks)}")

    frame_df = pd.DataFrame(frame_rows)
    block_df = pd.DataFrame(block_rows)

    frame_df.to_csv(os.path.join(out_dir, "node_weights_framewise.csv"), index=False)
    block_df.to_csv(os.path.join(out_dir, "node_weights_blockwise.csv"), index=False)

    # Replica summary across blocks
    summary_rows = []

    for node_id, g in block_df.groupby("node_id"):
        vals = g["node_weight_block_mean"].values.astype(float)

        summary_rows.append({
            "method": paths.method,
            "system": system.upper(),
            "rep_id": rep_id,
            "variant": variant,
            "node_id": int(node_id),
            "residue": int(node_id) + 1,
            "node_weight_mean": float(vals.mean()),
            "node_weight_sd_blocks": float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
            "node_weight_sem_blocks": float(vals.std(ddof=1) / np.sqrt(len(vals))) if len(vals) > 1 else 0.0,
            "n_blocks": int(len(vals)),
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(out_dir, "node_weights_summary.csv"), index=False)

    with open(os.path.join(out_dir, "summary.txt"), "w") as f:
        f.write(f"Node weights by block\n")
        f.write(f"Method: {paths.method}\n")
        f.write(f"System: {system.upper()}\n")
        f.write(f"Replica: {rep_id}\n")
        f.write(f"Variant: {variant}\n")
        f.write(f"Block size: {block_size}\n")
        f.write(f"Frames: {n_frames}\n")
        f.write(f"Blocks: {len(blocks)}\n")
        f.write(f"Node weight definition: sum of incident edge importance from {variant} attention.\n")

    if make_plot:
        d = summary_df.sort_values("residue")

        plt.figure(figsize=(11, 4.8))
        plt.plot(d["residue"], d["node_weight_mean"])
        plt.xlabel("Residue")
        plt.ylabel("Node weight")
        plt.title(f"{paths.method} {system.upper()} rep{rep_id}: node weight ({variant})")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "node_weights_plot.png"), dpi=300)
        plt.close()

    print(f"\nNODE WEIGHTS DONE: {paths.method} {system.upper()} rep{rep_id}")
    print("Output:", out_dir)

    return block_df, summary_df


def run_node_weights_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    variant="all_sum",
    block_size=500,
    device="cuda",
):
    all_summaries = []

    for method in methods:
        for system in systems:
            for rep in reps:
                _, summary_df = compute_node_weights_one_replica(
                    method=method,
                    system=system,
                    rep_id=rep,
                    variant=variant,
                    block_size=block_size,
                    device=device,
                    make_plot=True,
                )
                all_summaries.append(summary_df)

    all_summary = pd.concat(all_summaries, ignore_index=True)

    out_dir = os.path.join(base_root, "NODE_WEIGHTS_COMPARISON", variant)
    os.makedirs(out_dir, exist_ok=True)

    all_summary.to_csv(
        os.path.join(out_dir, "node_weights_all_methods_replicas.csv"),
        index=False,
    )

    print("\nALL NODE WEIGHTS DONE")
    print("Comparison output:", out_dir)

    return all_summary

In [ ]:
node_weight_summary_all = run_node_weights_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    variant="all_sum",
    block_size=500,
    device="cuda",
)

Loaded:
 method     : MethodA
 system     : WT
 replica    : 1
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgegat_dynamics_best_WT_rep1.pt
 data file  : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/trajectory_graphs_phi_psi_rmsf_WT_rep1.pt
 analysis   : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/NODE_WEIGHTS/all_sum/WT_rep1
 arch       : Arch(in_features=6, hidden_dim=128, out_features=3, heads=4, edge_dim=3, conv_bias=True, model_bias=True)
[DONE] MethodA WT rep1 block 1/8
[DONE] MethodA WT rep1 block 2/8
[DONE] MethodA WT rep1 block 3/8
[DONE] MethodA WT rep1 block 4/8
[DONE] MethodA WT rep1 block 5/8
[DONE] MethodA WT rep1 block 6/8
[DONE] MethodA WT rep1 block 7/8
[DONE] MethodA WT rep1 block 8/8

NODE WEIGHTS DONE: MethodA WT rep1
Output: /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/NODE_WEIGHTS/all_sum/WT_rep1
Loaded:
 method     : MethodA
 sys

In [ ]:
# ============================================================
# Cell 4A-2: System-level aggregation of node weights
# Requires node_weights_blockwise.csv from Cell 4A
# ============================================================

import os
import numpy as np
import pandas as pd


def load_node_weight_replica_tensor(path):
    df = pd.read_csv(path)

    required = ["block_id", "node_id", "node_weight_block_mean"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path} missing columns: {missing}")

    blocks = np.sort(df["block_id"].unique())
    nodes = np.sort(df["node_id"].unique())

    X = np.full((len(blocks), len(nodes)), np.nan, dtype=float)

    bmap = {int(b): i for i, b in enumerate(blocks)}
    nmap = {int(n): i for i, n in enumerate(nodes)}

    for _, row in df.iterrows():
        bi = bmap[int(row["block_id"])]
        ni = nmap[int(row["node_id"])]
        X[bi, ni] = float(row["node_weight_block_mean"])

    if np.isnan(X).any():
        raise RuntimeError(f"NaNs found after loading {path}")

    return nodes, X


def system_boot_mean_node_weights(Xr, rng):
    """
    Xr = list of replica arrays, each [blocks, nodes]
    """
    R = len(Xr)
    rep_idx = rng.integers(0, R, size=R)

    acc = None

    for ri in rep_idx:
        X = Xr[ri]
        B = X.shape[0]
        blk_idx = rng.integers(0, B, size=B)
        mu = X[blk_idx, :].mean(axis=0)

        if acc is None:
            acc = np.zeros_like(mu)

        acc += mu

    return acc / len(rep_idx)


def bootstrap_system_node_weights(Xr, n_boot=2000, seed=808):
    rng = np.random.default_rng(seed)

    N = Xr[0].shape[1]
    boot = np.empty((n_boot, N), dtype=float)

    for b in range(n_boot):
        boot[b, :] = system_boot_mean_node_weights(Xr, rng)

    mean = boot.mean(axis=0)
    low = np.quantile(boot, 0.025, axis=0)
    high = np.quantile(boot, 0.975, axis=0)
    std = boot.std(axis=0, ddof=1)
    sem = std / np.sqrt(n_boot)

    return mean, low, high, std, sem


def aggregate_node_weights_one_system(
    method,
    system,
    reps=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=808,
):
    method_dir = METHOD_DIRS[method]
    base_dir = os.path.join(base_root, method_dir)

    Xr = []
    nodes_all = []
    input_paths = []

    for rep in reps:
        path = os.path.join(
            base_dir,
            "ANALYSIS",
            "NODE_WEIGHTS",
            variant,
            f"{system.upper()}_rep{rep}",
            "node_weights_blockwise.csv",
        )

        if not os.path.exists(path):
            raise FileNotFoundError(path)

        nodes, X = load_node_weight_replica_tensor(path)
        Xr.append(X)
        nodes_all.append(nodes)
        input_paths.append(path)

    nodes = nodes_all[0]
    for n in nodes_all[1:]:
        if not np.array_equal(nodes, n):
            raise RuntimeError(f"Node mismatch across replicas for {method_dir} {system}")

    mean, low, high, std, sem = bootstrap_system_node_weights(
        Xr,
        n_boot=n_boot,
        seed=seed,
    )

    out_dir = os.path.join(
        base_dir,
        "ANALYSIS",
        "NODE_WEIGHTS",
        variant,
        "SYSTEM",
        system.upper(),
    )
    os.makedirs(out_dir, exist_ok=True)

    rows = []
    for i, node_id in enumerate(nodes):
        rows.append({
            "method": method_dir,
            "system": system.upper(),
            "variant": variant,
            "node_id": int(node_id),
            "residue": int(node_id) + 1,
            "node_weight_mean": float(mean[i]),
            "node_weight_ci_low": float(low[i]),
            "node_weight_ci_high": float(high[i]),
            "node_weight_std_bootstrap": float(std[i]),
            "node_weight_sem_bootstrap": float(sem[i]),
        })

    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(out_dir, "system_node_weights.csv"), index=False)

    with open(os.path.join(out_dir, "summary.txt"), "w") as f:
        f.write(f"System-level node weights\n")
        f.write(f"Method: {method_dir}\n")
        f.write(f"System: {system.upper()}\n")
        f.write(f"Variant: {variant}\n")
        f.write(f"Replicas: {list(reps)}\n")
        f.write(f"Bootstrap: n_boot={n_boot}, seed={seed}\n")
        f.write("Inputs:\n")
        for p in input_paths:
            f.write(f"  {p}\n")

    print(f"DONE system node weights: {method_dir} {system.upper()}")
    return out


def run_system_node_weights_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=808,
):
    all_rows = []

    for method in methods:
        for system in systems:
            df = aggregate_node_weights_one_system(
                method=method,
                system=system,
                reps=reps,
                variant=variant,
                n_boot=n_boot,
                seed=seed,
            )
            all_rows.append(df)

    all_df = pd.concat(all_rows, ignore_index=True)

    out_dir = os.path.join(base_root, "NODE_WEIGHTS_COMPARISON", variant)
    os.makedirs(out_dir, exist_ok=True)

    all_df.to_csv(
        os.path.join(out_dir, "system_node_weights_all_methods.csv"),
        index=False,
    )

    print("\nALL SYSTEM NODE WEIGHTS DONE")
    print("Output:", out_dir)

    return all_df

In [ ]:
system_node_weights_all = run_system_node_weights_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=808,
)

DONE system node weights: MethodA WT
DONE system node weights: MethodA NAM
DONE system node weights: MethodB WT
DONE system node weights: MethodB NAM
DONE system node weights: MethodC WT
DONE system node weights: MethodC NAM

ALL SYSTEM NODE WEIGHTS DONE
Output: /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/NODE_WEIGHTS_COMPARISON/all_sum


In [ ]:
# ============================================================
# Cell 4B: Descriptor centralities by block
# Requires:
#   - common model loader
#   - to_device_graph()
#   - node_weights_blockwise.csv from Cell 4A
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import networkx as nx
import matplotlib.pyplot as plt


GRAPH_DESCRIPTORS = ["strength", "betweenness", "closeness", "pagerank", "eigenvector"]
ALL_DESCRIPTORS = ["node_weight", "strength", "betweenness", "closeness", "pagerank", "eigenvector", "MVA"]


def build_attention_graph(num_nodes, edge_index, edge_weight, eps=1e-8):
    G = nx.Graph()
    G.add_nodes_from(range(num_nodes))

    src = edge_index[0].detach().cpu().numpy()
    dst = edge_index[1].detach().cpu().numpy()
    w = edge_weight.detach().cpu().numpy()

    edge_acc = {}
    edge_count = {}

    for u, v, val in zip(src, dst, w):
        u, v = int(u), int(v)
        key = (u, v) if u <= v else (v, u)
        edge_acc[key] = edge_acc.get(key, 0.0) + float(val)
        edge_count[key] = edge_count.get(key, 0) + 1

    for (u, v), val in edge_acc.items():
        wt = val / edge_count[(u, v)]
        if wt > 0:
            G.add_edge(u, v, weight=wt, distance=1.0 / (wt + eps))

    return G


def compute_graph_descriptors(G, num_nodes):
    strength = np.zeros(num_nodes, dtype=float)
    for n in range(num_nodes):
        strength[n] = sum(attr["weight"] for _, _, attr in G.edges(n, data=True))

    betweenness = nx.betweenness_centrality(G, weight="distance", normalized=True)
    closeness = nx.closeness_centrality(G, distance="distance")
    pagerank = nx.pagerank(G, weight="weight")

    try:
        eigenvector = nx.eigenvector_centrality(G, weight="weight", max_iter=2000)
    except Exception:
        eigenvector = {n: 0.0 for n in G.nodes()}

    out = {
        "strength": strength,
        "betweenness": np.array([betweenness.get(i, 0.0) for i in range(num_nodes)]),
        "closeness": np.array([closeness.get(i, 0.0) for i in range(num_nodes)]),
        "pagerank": np.array([pagerank.get(i, 0.0) for i in range(num_nodes)]),
        "eigenvector": np.array([eigenvector.get(i, 0.0) for i in range(num_nodes)]),
    }

    return out


def load_node_weight_block_file(method, system, rep_id, variant="all_sum"):
    method_dir = METHOD_DIRS[method]
    base_dir = os.path.join(base_root, method_dir)

    path = os.path.join(
        base_dir,
        "ANALYSIS",
        "NODE_WEIGHTS",
        variant,
        f"{system.upper()}_rep{rep_id}",
        "node_weights_blockwise.csv",
    )

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)

    required = ["block_id", "node_id", "node_weight_block_mean"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path} missing columns: {missing}")

    return df


def get_node_weight_for_block(nw_df, block_id, num_nodes):
    sub = nw_df[nw_df["block_id"] == block_id].copy()
    arr = np.zeros(num_nodes, dtype=float)

    for _, row in sub.iterrows():
        nid = int(row["node_id"])
        if nid < num_nodes:
            arr[nid] = float(row["node_weight_block_mean"])

    return arr


def compute_descriptors_one_replica(
    method,
    system,
    rep_id,
    variant="all_sum",
    block_size=500,
    device="cuda",
    eps=1e-8,
    make_plot=True,
):
    model, data_cpu, paths, arch, device_obj = load_model_and_data(
        method=method,
        system=system,
        rep_id=rep_id,
        device=device,
        analysis_subdir=os.path.join("DESCRIPTOR_PIPELINE", variant, "REPLICA_BLOCK"),
    )

    out_dir = paths.analysis_dir
    os.makedirs(out_dir, exist_ok=True)

    nw_df = load_node_weight_block_file(method, system, rep_id, variant=variant)

    n_frames = len(data_cpu)
    blocks = make_blocks_nodeweights(n_frames, block_size)

    rows = []

    model.eval()

    with torch.no_grad():
        for bi, (bs, be) in enumerate(blocks):
            edge_sum = {}
            edge_count = {}
            num_nodes = None
            ref_edge_index = None

            for t in range(bs, be):
                d = to_device_graph(data_cpu[t], device_obj)
                num_nodes = d.x.shape[0]
                ref_edge_index = d.edge_index

                model.clear_masks()
                _ = model(d.x, d.edge_index, d.edge_attr)

                edge_imp = extract_edge_importance_variant(model, variant=variant)

                src = d.edge_index[0]
                dst = d.edge_index[1]

                for e in range(edge_imp.numel()):
                    u = int(src[e].item())
                    v = int(dst[e].item())
                    key = (u, v) if u <= v else (v, u)
                    val = float(edge_imp[e].item())

                    edge_sum[key] = edge_sum.get(key, 0.0) + val
                    edge_count[key] = edge_count.get(key, 0) + 1

            # Mean edge attention over frames in block
            edge_mean = {
                key: edge_sum[key] / edge_count[key]
                for key in edge_sum
            }

            # Build graph
            G = nx.Graph()
            G.add_nodes_from(range(num_nodes))

            for (u, v), wt in edge_mean.items():
                if wt > 0:
                    G.add_edge(u, v, weight=wt, distance=1.0 / (wt + eps))

            desc = compute_graph_descriptors(G, num_nodes)
            node_weight = get_node_weight_for_block(nw_df, bi, num_nodes)
            MVA = node_weight * desc["strength"]

            for node_id in range(num_nodes):
                rows.append({
                    "method": paths.method,
                    "system": system.upper(),
                    "rep_id": rep_id,
                    "variant": variant,
                    "block_id": bi,
                    "frame_start": bs,
                    "frame_end": be,
                    "node_id": node_id,
                    "residue": node_id + 1,
                    "node_weight": float(node_weight[node_id]),
                    "strength": float(desc["strength"][node_id]),
                    "betweenness": float(desc["betweenness"][node_id]),
                    "closeness": float(desc["closeness"][node_id]),
                    "pagerank": float(desc["pagerank"][node_id]),
                    "eigenvector": float(desc["eigenvector"][node_id]),
                    "MVA": float(MVA[node_id]),
                })

            print(f"[DONE] {paths.method} {system.upper()} rep{rep_id} descriptor block {bi+1}/{len(blocks)}")

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out_dir, "descriptors_by_block.csv"), index=False)

    summary_rows = []
    for node_id, g in df.groupby("node_id"):
        row = {
            "method": paths.method,
            "system": system.upper(),
            "rep_id": rep_id,
            "variant": variant,
            "node_id": int(node_id),
            "residue": int(node_id) + 1,
        }

        for desc in ALL_DESCRIPTORS:
            vals = g[desc].values.astype(float)
            row[f"{desc}_mean"] = float(vals.mean())
            row[f"{desc}_sd_blocks"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
            row[f"{desc}_sem_blocks"] = float(vals.std(ddof=1) / np.sqrt(len(vals))) if len(vals) > 1 else 0.0

        row["n_blocks"] = int(len(g))
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(out_dir, "descriptors_summary.csv"), index=False)

    if make_plot:
        for desc in ALL_DESCRIPTORS:
            d = summary_df.sort_values("residue")
            plt.figure(figsize=(11, 4.5))
            plt.plot(d["residue"], d[f"{desc}_mean"])
            plt.xlabel("Residue")
            plt.ylabel(desc)
            plt.title(f"{paths.method} {system.upper()} rep{rep_id}: {desc} ({variant})")
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"{desc}_plot.png"), dpi=300)
            plt.close()

    with open(os.path.join(out_dir, "summary.txt"), "w") as f:
        f.write("Descriptor centralities by block\n")
        f.write(f"Method: {paths.method}\n")
        f.write(f"System: {system.upper()}\n")
        f.write(f"Replica: {rep_id}\n")
        f.write(f"Variant: {variant}\n")
        f.write(f"Block size: {block_size}\n")
        f.write("Descriptors: node_weight, strength, betweenness, closeness, pagerank, eigenvector, MVA\n")
        f.write("MVA = node_weight × strength\n")

    print(f"\nDESCRIPTORS DONE: {paths.method} {system.upper()} rep{rep_id}")
    print("Output:", out_dir)

    return df, summary_df


def run_descriptors_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    variant="all_sum",
    block_size=500,
    device="cuda",
):
    all_summaries = []

    for method in methods:
        for system in systems:
            for rep in reps:
                _, summary_df = compute_descriptors_one_replica(
                    method=method,
                    system=system,
                    rep_id=rep,
                    variant=variant,
                    block_size=block_size,
                    device=device,
                    make_plot=True,
                )
                all_summaries.append(summary_df)

    all_summary = pd.concat(all_summaries, ignore_index=True)

    out_dir = os.path.join(base_root, "DESCRIPTOR_COMPARISON", variant)
    os.makedirs(out_dir, exist_ok=True)

    all_summary.to_csv(
        os.path.join(out_dir, "descriptor_replica_summaries_all_methods.csv"),
        index=False,
    )

    print("\nALL DESCRIPTORS DONE")
    print("Output:", out_dir)

    return all_summary

In [ ]:
descriptor_summary_all = run_descriptors_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    variant="all_sum",
    block_size=500,
    device="cuda",
)

Loaded:
 method     : MethodA
 system     : WT
 replica    : 1
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgegat_dynamics_best_WT_rep1.pt
 data file  : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/trajectory_graphs_phi_psi_rmsf_WT_rep1.pt
 analysis   : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/DESCRIPTOR_PIPELINE/all_sum/REPLICA_BLOCK/WT_rep1
 arch       : Arch(in_features=6, hidden_dim=128, out_features=3, heads=4, edge_dim=3, conv_bias=True, model_bias=True)
[DONE] MethodA WT rep1 descriptor block 1/8
[DONE] MethodA WT rep1 descriptor block 2/8
[DONE] MethodA WT rep1 descriptor block 3/8
[DONE] MethodA WT rep1 descriptor block 4/8
[DONE] MethodA WT rep1 descriptor block 5/8
[DONE] MethodA WT rep1 descriptor block 6/8
[DONE] MethodA WT rep1 descriptor block 7/8
[DONE] MethodA WT rep1 descriptor block 8/8

DESCRIPTORS DONE: MethodA WT rep1
Output: /content/drive/MyDrive/ABE_RF_Dif

In [ ]:
# ============================================================
# Cell 4C: System-level descriptor aggregation
# Combines replica block descriptors using hierarchical bootstrap
# ============================================================

import os
import numpy as np
import pandas as pd

ALL_DESCRIPTORS = [
    "node_weight",
    "strength",
    "betweenness",
    "closeness",
    "pagerank",
    "eigenvector",
    "MVA",
]


def load_descriptor_replica_tensor(path, descriptors=ALL_DESCRIPTORS):
    df = pd.read_csv(path)

    required = ["block_id", "node_id"] + descriptors
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path} missing columns: {missing}")

    blocks = np.sort(df["block_id"].unique())
    nodes = np.sort(df["node_id"].unique())

    X = np.full((len(blocks), len(nodes), len(descriptors)), np.nan)

    bmap = {int(b): i for i, b in enumerate(blocks)}
    nmap = {int(n): i for i, n in enumerate(nodes)}

    for _, row in df.iterrows():
        bi = bmap[int(row["block_id"])]
        ni = nmap[int(row["node_id"])]
        X[bi, ni, :] = row[descriptors].to_numpy(dtype=float)

    if np.isnan(X).any():
        raise RuntimeError(f"NaNs found after loading {path}")

    return nodes, X


def system_boot_mean_descriptor(Xr, rng):
    """
    Xr = list of replica tensors [blocks, nodes, descriptors]
    """
    R = len(Xr)
    rep_idx = rng.integers(0, R, size=R)

    acc = None

    for ri in rep_idx:
        X = Xr[ri]
        B = X.shape[0]
        blk_idx = rng.integers(0, B, size=B)
        mu = X[blk_idx, :, :].mean(axis=0)

        if acc is None:
            acc = np.zeros_like(mu)

        acc += mu

    return acc / len(rep_idx)


def bootstrap_system_descriptors(Xr, n_boot=2000, seed=909):
    rng = np.random.default_rng(seed)

    N = Xr[0].shape[1]
    M = Xr[0].shape[2]

    boot = np.empty((n_boot, N, M), dtype=float)

    for b in range(n_boot):
        boot[b, :, :] = system_boot_mean_descriptor(Xr, rng)

        if (b + 1) % max(1, n_boot // 10) == 0:
            print(f"[bootstrap] {b+1}/{n_boot}")

    mean = boot.mean(axis=0)
    low = np.quantile(boot, 0.025, axis=0)
    high = np.quantile(boot, 0.975, axis=0)
    std = boot.std(axis=0, ddof=1)
    sem = std / np.sqrt(n_boot)

    return mean, low, high, std, sem


def aggregate_descriptors_one_system(
    method,
    system,
    reps=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=909,
):
    method_dir = METHOD_DIRS[method]
    base_dir = os.path.join(base_root, method_dir)

    Xr = []
    nodes_all = []
    input_paths = []

    for rep in reps:
        path = os.path.join(
            base_dir,
            "ANALYSIS",
            "DESCRIPTOR_PIPELINE",
            variant,
            "REPLICA_BLOCK",
            f"{system.upper()}_rep{rep}",
            "descriptors_by_block.csv",
        )

        if not os.path.exists(path):
            raise FileNotFoundError(path)

        nodes, X = load_descriptor_replica_tensor(path)
        Xr.append(X)
        nodes_all.append(nodes)
        input_paths.append(path)

    nodes = nodes_all[0]
    for n in nodes_all[1:]:
        if not np.array_equal(nodes, n):
            raise RuntimeError(f"Node mismatch across replicas for {method_dir} {system}")

    mean, low, high, std, sem = bootstrap_system_descriptors(
        Xr,
        n_boot=n_boot,
        seed=seed,
    )

    out_dir = os.path.join(
        base_dir,
        "ANALYSIS",
        "DESCRIPTOR_PIPELINE",
        variant,
        "SYSTEM",
        system.upper(),
    )
    os.makedirs(out_dir, exist_ok=True)

    rows = []

    for ni, node_id in enumerate(nodes):
        row = {
            "method": method_dir,
            "system": system.upper(),
            "variant": variant,
            "node_id": int(node_id),
            "residue": int(node_id) + 1,
        }

        for mi, desc in enumerate(ALL_DESCRIPTORS):
            row[f"{desc}_mean"] = float(mean[ni, mi])
            row[f"{desc}_ci_low"] = float(low[ni, mi])
            row[f"{desc}_ci_high"] = float(high[ni, mi])
            row[f"{desc}_std_bootstrap"] = float(std[ni, mi])
            row[f"{desc}_sem_bootstrap"] = float(sem[ni, mi])

        rows.append(row)

    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(out_dir, "system_descriptors.csv"), index=False)

    with open(os.path.join(out_dir, "summary.txt"), "w") as f:
        f.write("System-level descriptor aggregation\n")
        f.write(f"Method: {method_dir}\n")
        f.write(f"System: {system.upper()}\n")
        f.write(f"Variant: {variant}\n")
        f.write(f"Replicas: {list(reps)}\n")
        f.write(f"Bootstrap: n_boot={n_boot}, seed={seed}\n")
        f.write("Descriptors: " + ", ".join(ALL_DESCRIPTORS) + "\n")
        f.write("Inputs:\n")
        for p in input_paths:
            f.write(f"  {p}\n")

    print(f"DONE system descriptors: {method_dir} {system.upper()}")
    return out


def run_system_descriptors_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=909,
):
    all_rows = []

    for method in methods:
        for system in systems:
            df = aggregate_descriptors_one_system(
                method=method,
                system=system,
                reps=reps,
                variant=variant,
                n_boot=n_boot,
                seed=seed,
            )
            all_rows.append(df)

    all_df = pd.concat(all_rows, ignore_index=True)

    out_dir = os.path.join(base_root, "DESCRIPTOR_COMPARISON", variant)
    os.makedirs(out_dir, exist_ok=True)

    all_df.to_csv(
        os.path.join(out_dir, "system_descriptors_all_methods.csv"),
        index=False,
    )

    print("\nALL SYSTEM DESCRIPTORS DONE")
    print("Output:", out_dir)

    return all_df

In [ ]:
system_descriptors_all = run_system_descriptors_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=909,
)

[bootstrap] 200/2000
[bootstrap] 400/2000
[bootstrap] 600/2000
[bootstrap] 800/2000
[bootstrap] 1000/2000
[bootstrap] 1200/2000
[bootstrap] 1400/2000
[bootstrap] 1600/2000
[bootstrap] 1800/2000
[bootstrap] 2000/2000
DONE system descriptors: MethodA WT
[bootstrap] 200/2000
[bootstrap] 400/2000
[bootstrap] 600/2000
[bootstrap] 800/2000
[bootstrap] 1000/2000
[bootstrap] 1200/2000
[bootstrap] 1400/2000
[bootstrap] 1600/2000
[bootstrap] 1800/2000
[bootstrap] 2000/2000
DONE system descriptors: MethodA NAM
[bootstrap] 200/2000
[bootstrap] 400/2000
[bootstrap] 600/2000
[bootstrap] 800/2000
[bootstrap] 1000/2000
[bootstrap] 1200/2000
[bootstrap] 1400/2000
[bootstrap] 1600/2000
[bootstrap] 1800/2000
[bootstrap] 2000/2000
DONE system descriptors: MethodB WT
[bootstrap] 200/2000
[bootstrap] 400/2000
[bootstrap] 600/2000
[bootstrap] 800/2000
[bootstrap] 1000/2000
[bootstrap] 1200/2000
[bootstrap] 1400/2000
[bootstrap] 1600/2000
[bootstrap] 1800/2000
[bootstrap] 2000/2000
DONE system descriptors: Me

In [ ]:
# ============================================================
# Cell 4D: Region-level system descriptors
# Single-system compatible
# Uses replica/block tensors and aggregates regions inside bootstrap
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ALL_DESCRIPTORS = [
    "node_weight",
    "strength",
    "betweenness",
    "closeness",
    "pagerank",
    "eigenvector",
    "MVA",
]

REGIONS = {
    "Loop": list(range(5, 8)) + [9],
    "Ful": list(range(10, 23)),
    "AS": [8] + list(range(23, 31)) + [32, 76, 84],
    "Bur": [31, 33, 34, 77, 83, 85],
    "Elb": list(range(35, 43)),
    "Fl_ext": list(range(43, 47)) + list(range(55, 59)),
    "Flp": list(range(47, 55)),
    "UpC": list(range(59, 63)) + list(range(73, 76)),
    "LoC": list(range(63, 73)),
    "80s": list(range(78, 83)),
    "Hlx": list(range(86, 95)),
    "Ter": list(range(1, 5)) + list(range(95, 104)) + list(range(194, 199)),
    "Loop′": list(range(104, 107)) + [108],
    "Ful′": list(range(109, 122)),
    "AS′": [107] + list(range(122, 130)) + [131, 175, 183],
    "Bur′": [130, 132, 133, 176, 182, 184],
    "Elb′": list(range(134, 142)),
    "Fl_ext′": list(range(142, 146)) + list(range(154, 158)),
    "Flp′": list(range(146, 154)),
    "UpC′": list(range(158, 162)) + list(range(172, 175)),
    "LoC′": list(range(162, 172)),
    "80s′": list(range(177, 182)),
    "Hlx′": list(range(185, 194)),
}


def get_region_indices_from_nodes(nodes):
    residue_to_index = {int(node) + 1: i for i, node in enumerate(nodes)}
    reg_idx = {}

    for region, residues in REGIONS.items():
        idx = [residue_to_index[r] for r in residues if r in residue_to_index]
        if idx:
            reg_idx[region] = idx

    return reg_idx


def system_region_boot_mean(Xr, reg_idx, rng):
    """
    Xr: list of [blocks, nodes, descriptors]
    returns: [regions, descriptors]
    """
    R = len(Xr)
    rep_idx = rng.integers(0, R, size=R)

    region_names = list(reg_idx.keys())
    acc = None

    for ri in rep_idx:
        X = Xr[ri]
        B = X.shape[0]
        blk_idx = rng.integers(0, B, size=B)

        mu_nodes = X[blk_idx, :, :].mean(axis=0)  # [nodes, descriptors]

        mu_regions = np.zeros((len(region_names), X.shape[2]), dtype=float)

        for r_i, reg in enumerate(region_names):
            idx = reg_idx[reg]
            mu_regions[r_i, :] = mu_nodes[idx, :].mean(axis=0)

        if acc is None:
            acc = np.zeros_like(mu_regions)

        acc += mu_regions

    return acc / len(rep_idx)


def bootstrap_region_system_descriptors(Xr, reg_idx, n_boot=2000, seed=1001):
    rng = np.random.default_rng(seed)

    region_names = list(reg_idx.keys())
    M = Xr[0].shape[2]

    boot = np.empty((n_boot, len(region_names), M), dtype=float)

    for b in range(n_boot):
        boot[b, :, :] = system_region_boot_mean(Xr, reg_idx, rng)

        if (b + 1) % max(1, n_boot // 10) == 0:
            print(f"[region bootstrap] {b+1}/{n_boot}")

    mean = boot.mean(axis=0)
    low = np.quantile(boot, 0.025, axis=0)
    high = np.quantile(boot, 0.975, axis=0)
    std = boot.std(axis=0, ddof=1)
    sem = std / np.sqrt(n_boot)

    return region_names, mean, low, high, std, sem


def region_system_descriptors_one_system(
    method,
    system,
    reps=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=1001,
    make_plots=True,
):
    method_dir = METHOD_DIRS[method]
    base_dir = os.path.join(base_root, method_dir)

    Xr = []
    nodes_all = []
    input_paths = []

    for rep in reps:
        path = os.path.join(
            base_dir,
            "ANALYSIS",
            "DESCRIPTOR_PIPELINE",
            variant,
            "REPLICA_BLOCK",
            f"{system.upper()}_rep{rep}",
            "descriptors_by_block.csv",
        )

        if not os.path.exists(path):
            raise FileNotFoundError(path)

        nodes, X = load_descriptor_replica_tensor(path, descriptors=ALL_DESCRIPTORS)
        Xr.append(X)
        nodes_all.append(nodes)
        input_paths.append(path)

    nodes = nodes_all[0]
    for n in nodes_all[1:]:
        if not np.array_equal(nodes, n):
            raise RuntimeError(f"Node mismatch across replicas for {method_dir} {system}")

    reg_idx = get_region_indices_from_nodes(nodes)

    region_names, mean, low, high, std, sem = bootstrap_region_system_descriptors(
        Xr,
        reg_idx,
        n_boot=n_boot,
        seed=seed,
    )

    out_dir = os.path.join(
        base_dir,
        "ANALYSIS",
        "DESCRIPTOR_PIPELINE",
        variant,
        "REGION_SYSTEM",
        system.upper(),
    )
    os.makedirs(out_dir, exist_ok=True)

    rows = []

    for ri, region in enumerate(region_names):
        for mi, desc in enumerate(ALL_DESCRIPTORS):
            rows.append({
                "method": method_dir,
                "system": system.upper(),
                "variant": variant,
                "region": region,
                "descriptor": desc,
                "n_residues_region": len(reg_idx[region]),
                "residues_used": ",".join(map(str, REGIONS[region])),
                "system_mean": float(mean[ri, mi]),
                "system_ci_low": float(low[ri, mi]),
                "system_ci_high": float(high[ri, mi]),
                "system_std_bootstrap": float(std[ri, mi]),
                "system_sem_bootstrap": float(sem[ri, mi]),
            })

    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(out_dir, "region_system_descriptors.csv"), index=False)

    with open(os.path.join(out_dir, "summary.txt"), "w") as f:
        f.write("Region-level system descriptor aggregation\n")
        f.write(f"Method: {method_dir}\n")
        f.write(f"System: {system.upper()}\n")
        f.write(f"Variant: {variant}\n")
        f.write(f"Replicas: {list(reps)}\n")
        f.write(f"Bootstrap: n_boot={n_boot}, seed={seed}\n")
        f.write("Region values were computed inside each bootstrap draw.\n")
        f.write("Inputs:\n")
        for p in input_paths:
            f.write(f"  {p}\n")

    if make_plots:
        for desc in ALL_DESCRIPTORS:
            d = out[out["descriptor"] == desc].copy()
            x = np.arange(len(d))
            mean_vals = d["system_mean"].values
            low_vals = d["system_ci_low"].values
            high_vals = d["system_ci_high"].values

            yerr = np.vstack([
                np.maximum(0, mean_vals - low_vals),
                np.maximum(0, high_vals - mean_vals),
            ])

            plt.figure(figsize=(12, 4.8))
            plt.bar(x, mean_vals)
            plt.errorbar(x, mean_vals, yerr=yerr, fmt="none", capsize=3)
            plt.xticks(x, d["region"].values, rotation=45, ha="right")
            plt.ylabel(desc)
            plt.title(f"{method_dir} {system.upper()} region {desc}")
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"region_system_{desc}.png"), dpi=300)
            plt.close()

    print(f"DONE region system descriptors: {method_dir} {system.upper()}")
    return out


def run_region_system_descriptors_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=1001,
):
    all_rows = []

    for method in methods:
        for system in systems:
            df = region_system_descriptors_one_system(
                method=method,
                system=system,
                reps=reps,
                variant=variant,
                n_boot=n_boot,
                seed=seed,
                make_plots=True,
            )
            all_rows.append(df)

    all_df = pd.concat(all_rows, ignore_index=True)

    out_dir = os.path.join(base_root, "DESCRIPTOR_COMPARISON", variant)
    os.makedirs(out_dir, exist_ok=True)

    all_df.to_csv(
        os.path.join(out_dir, "region_system_descriptors_all_methods.csv"),
        index=False,
    )

    print("\nALL REGION SYSTEM DESCRIPTORS DONE")
    print("Output:", out_dir)

    return all_df

In [ ]:
region_system_descriptors_all = run_region_system_descriptors_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=1001,
)

[region bootstrap] 200/2000
[region bootstrap] 400/2000
[region bootstrap] 600/2000
[region bootstrap] 800/2000
[region bootstrap] 1000/2000
[region bootstrap] 1200/2000
[region bootstrap] 1400/2000
[region bootstrap] 1600/2000
[region bootstrap] 1800/2000
[region bootstrap] 2000/2000
DONE region system descriptors: MethodA WT
[region bootstrap] 200/2000
[region bootstrap] 400/2000
[region bootstrap] 600/2000
[region bootstrap] 800/2000
[region bootstrap] 1000/2000
[region bootstrap] 1200/2000
[region bootstrap] 1400/2000
[region bootstrap] 1600/2000
[region bootstrap] 1800/2000
[region bootstrap] 2000/2000
DONE region system descriptors: MethodA NAM
[region bootstrap] 200/2000
[region bootstrap] 400/2000
[region bootstrap] 600/2000
[region bootstrap] 800/2000
[region bootstrap] 1000/2000
[region bootstrap] 1200/2000
[region bootstrap] 1400/2000
[region bootstrap] 1600/2000
[region bootstrap] 1800/2000
[region bootstrap] 2000/2000
DONE region system descriptors: MethodB WT
[region boot

In [ ]:
# ============================================================
# Cell 4E: Residue-level WT−NAM descriptor delta
# Optional two-system comparison step
# Delta = WT - NAM
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ALL_DESCRIPTORS = [
    "node_weight",
    "strength",
    "betweenness",
    "closeness",
    "pagerank",
    "eigenvector",
    "MVA",
]


def load_descriptor_system_replicas(method, system, reps=(1, 2, 3), variant="all_sum"):
    method_dir = METHOD_DIRS[method]
    base_dir = os.path.join(base_root, method_dir)

    Xr = []
    nodes_all = []

    for rep in reps:
        path = os.path.join(
            base_dir,
            "ANALYSIS",
            "DESCRIPTOR_PIPELINE",
            variant,
            "REPLICA_BLOCK",
            f"{system.upper()}_rep{rep}",
            "descriptors_by_block.csv",
        )

        if not os.path.exists(path):
            raise FileNotFoundError(path)

        nodes, X = load_descriptor_replica_tensor(path, descriptors=ALL_DESCRIPTORS)
        Xr.append(X)
        nodes_all.append(nodes)

    nodes = nodes_all[0]
    for n in nodes_all[1:]:
        if not np.array_equal(nodes, n):
            raise RuntimeError(f"Node mismatch for {method_dir} {system}")

    return nodes, Xr


def system_boot_mean_descriptor_delta(Xr, rng):
    R = len(Xr)
    rep_idx = rng.integers(0, R, size=R)

    acc = None

    for ri in rep_idx:
        X = Xr[ri]
        B = X.shape[0]
        blk_idx = rng.integers(0, B, size=B)

        mu = X[blk_idx, :, :].mean(axis=0)

        if acc is None:
            acc = np.zeros_like(mu)

        acc += mu

    return acc / len(rep_idx)


def bootstrap_residue_delta(wt_Xr, nam_Xr, n_boot=2000, seed=1111):
    rng = np.random.default_rng(seed)

    N = wt_Xr[0].shape[1]
    M = wt_Xr[0].shape[2]

    boot = np.empty((n_boot, N, M), dtype=float)

    for b in range(n_boot):
        wt_mu = system_boot_mean_descriptor_delta(wt_Xr, rng)
        nam_mu = system_boot_mean_descriptor_delta(nam_Xr, rng)
        boot[b, :, :] = wt_mu - nam_mu

        if (b + 1) % max(1, n_boot // 10) == 0:
            print(f"[delta bootstrap] {b+1}/{n_boot}")

    mean = boot.mean(axis=0)
    low = np.quantile(boot, 0.025, axis=0)
    high = np.quantile(boot, 0.975, axis=0)
    std = boot.std(axis=0, ddof=1)
    sem = std / np.sqrt(n_boot)

    return mean, low, high, std, sem


def compute_residue_delta_one_method(
    method,
    system_a="WT",
    system_b="NAM",
    reps_a=(1, 2, 3),
    reps_b=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=1111,
    make_plots=True,
):
    """
    Computes residue-level delta = system_a - system_b.
    Default: WT - NAM.
    """

    method_dir = METHOD_DIRS[method]
    base_dir = os.path.join(base_root, method_dir)

    nodes_a, Xa = load_descriptor_system_replicas(
        method, system_a, reps=reps_a, variant=variant
    )
    nodes_b, Xb = load_descriptor_system_replicas(
        method, system_b, reps=reps_b, variant=variant
    )

    if not np.array_equal(nodes_a, nodes_b):
        raise RuntimeError(f"Node mismatch between {system_a} and {system_b} for {method_dir}")

    mean, low, high, std, sem = bootstrap_residue_delta(
        Xa,
        Xb,
        n_boot=n_boot,
        seed=seed,
    )

    out_dir = os.path.join(
        base_dir,
        "ANALYSIS",
        "DESCRIPTOR_PIPELINE",
        variant,
        "DELTA_RESIDUE",
        f"{system_a.upper()}_minus_{system_b.upper()}",
    )
    os.makedirs(out_dir, exist_ok=True)

    rows = []

    for ni, node_id in enumerate(nodes_a):
        row = {
            "method": method_dir,
            "variant": variant,
            "comparison": f"{system_a.upper()}-{system_b.upper()}",
            "node_id": int(node_id),
            "residue": int(node_id) + 1,
        }

        for mi, desc in enumerate(ALL_DESCRIPTORS):
            row[f"delta_{desc}_mean"] = float(mean[ni, mi])
            row[f"delta_{desc}_ci_low"] = float(low[ni, mi])
            row[f"delta_{desc}_ci_high"] = float(high[ni, mi])
            row[f"delta_{desc}_std_bootstrap"] = float(std[ni, mi])
            row[f"delta_{desc}_sem_bootstrap"] = float(sem[ni, mi])
            row[f"delta_{desc}_significant"] = bool(
                (low[ni, mi] > 0) or (high[ni, mi] < 0)
            )

        rows.append(row)

    delta_df = pd.DataFrame(rows)
    delta_df.to_csv(os.path.join(out_dir, "delta_residue_descriptors.csv"), index=False)

    # Long-format delta
    long_rows = []
    for _, row in delta_df.iterrows():
        for desc in ALL_DESCRIPTORS:
            long_rows.append({
                "method": row["method"],
                "variant": variant,
                "comparison": row["comparison"],
                "node_id": int(row["node_id"]),
                "residue": int(row["residue"]),
                "descriptor": desc,
                "delta_mean": float(row[f"delta_{desc}_mean"]),
                "delta_ci_low": float(row[f"delta_{desc}_ci_low"]),
                "delta_ci_high": float(row[f"delta_{desc}_ci_high"]),
                "delta_std_bootstrap": float(row[f"delta_{desc}_std_bootstrap"]),
                "delta_sem_bootstrap": float(row[f"delta_{desc}_sem_bootstrap"]),
                "significant": bool(row[f"delta_{desc}_significant"]),
            })

    long_df = pd.DataFrame(long_rows)
    long_df.to_csv(os.path.join(out_dir, "delta_residue_descriptors_long.csv"), index=False)

    # Descriptor significance summary
    summary_rows = []

    for desc in ALL_DESCRIPTORS:
        mean_col = f"delta_{desc}_mean"
        lo_col = f"delta_{desc}_ci_low"
        hi_col = f"delta_{desc}_ci_high"
        std_col = f"delta_{desc}_std_bootstrap"
        sig_col = f"delta_{desc}_significant"

        d = delta_df.copy()
        d["abs_delta"] = d[mean_col].abs()
        d["ci_width"] = d[hi_col] - d[lo_col]
        sig = d[d[sig_col] == True].copy()

        summary_rows.append({
            "method": method_dir,
            "variant": variant,
            "comparison": f"{system_a.upper()}-{system_b.upper()}",
            "descriptor": desc,
            "n_significant_residues": int(len(sig)),
            "fraction_significant_residues": float(len(sig) / len(d)),
            "median_abs_delta_all_residues": float(d["abs_delta"].median()),
            "mean_abs_delta_all_residues": float(d["abs_delta"].mean()),
            "median_ci_width_all_residues": float(d["ci_width"].median()),
            "median_std_bootstrap_all_residues": float(d[std_col].median()),
            "median_abs_delta_significant": float(sig["abs_delta"].median()) if len(sig) else np.nan,
            "mean_abs_delta_significant": float(sig["abs_delta"].mean()) if len(sig) else np.nan,
            "median_ci_width_significant": float(sig["ci_width"].median()) if len(sig) else np.nan,
            "median_std_bootstrap_significant": float(sig[std_col].median()) if len(sig) else np.nan,
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(out_dir, "delta_residue_descriptor_summary.csv"), index=False)

    if make_plots:
        plot_dir = os.path.join(out_dir, "PLOTS")
        os.makedirs(plot_dir, exist_ok=True)

        for desc in ALL_DESCRIPTORS:
            mean_col = f"delta_{desc}_mean"
            lo_col = f"delta_{desc}_ci_low"
            hi_col = f"delta_{desc}_ci_high"
            sig_col = f"delta_{desc}_significant"

            d = delta_df.sort_values("residue").copy()

            x = d["residue"].values
            y = d[mean_col].values
            low_y = d[lo_col].values
            high_y = d[hi_col].values
            sig = d[sig_col].values.astype(bool)

            yerr = np.vstack([
                np.maximum(0, y - low_y),
                np.maximum(0, high_y - y),
            ])

            plt.figure(figsize=(12, 4.8))

            idx_ns = np.where(~sig)[0]
            idx_s = np.where(sig)[0]

            plt.bar(x[idx_ns], y[idx_ns], alpha=0.45, label="Not significant")
            plt.bar(x[idx_s], y[idx_s], alpha=0.95, label="Significant")
            plt.errorbar(x, y, yerr=yerr, fmt="none", capsize=2, linewidth=0.8)

            plt.axhline(0, linewidth=1)
            plt.xlabel("Residue")
            plt.ylabel(f"Δ {desc} ({system_a.upper()} − {system_b.upper()})")
            plt.title(f"{method_dir}: residue-level Δ {desc}")
            plt.legend()
            plt.tight_layout()
            plt.savefig(os.path.join(plot_dir, f"delta_residue_{desc}.png"), dpi=300)
            plt.close()

        # significance count bar
        s = summary_df.sort_values("n_significant_residues", ascending=False)

        plt.figure(figsize=(8, 4.8))
        x = np.arange(len(s))
        plt.bar(x, s["n_significant_residues"].values)
        plt.xticks(x, s["descriptor"].values, rotation=45, ha="right")
        plt.ylabel("Number of significant residues")
        plt.title(f"{method_dir}: significant residues by descriptor")
        plt.tight_layout()
        plt.savefig(os.path.join(plot_dir, "descriptor_significant_residues_bar.png"), dpi=300)
        plt.close()

    with open(os.path.join(out_dir, "summary.txt"), "w") as f:
        f.write("Residue-level descriptor delta\n")
        f.write(f"Method: {method_dir}\n")
        f.write(f"Variant: {variant}\n")
        f.write(f"Comparison: {system_a.upper()} - {system_b.upper()}\n")
        f.write(f"Bootstrap: n_boot={n_boot}, seed={seed}\n")
        f.write("Delta computed inside hierarchical bootstrap over replicas and blocks.\n")

    print(f"DONE residue delta: {method_dir} {system_a.upper()} - {system_b.upper()}")
    return delta_df, summary_df


def run_residue_delta_all_methods(
    methods=("A", "B", "C"),
    system_a="WT",
    system_b="NAM",
    reps_a=(1, 2, 3),
    reps_b=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=1111,
):
    all_delta = []
    all_summary = []

    for method in methods:
        delta_df, summary_df = compute_residue_delta_one_method(
            method=method,
            system_a=system_a,
            system_b=system_b,
            reps_a=reps_a,
            reps_b=reps_b,
            variant=variant,
            n_boot=n_boot,
            seed=seed,
            make_plots=True,
        )
        all_delta.append(delta_df)
        all_summary.append(summary_df)

    delta_all = pd.concat(all_delta, ignore_index=True)
    summary_all = pd.concat(all_summary, ignore_index=True)

    out_dir = os.path.join(
        base_root,
        "DESCRIPTOR_COMPARISON",
        variant,
        "DELTA_RESIDUE",
        f"{system_a.upper()}_minus_{system_b.upper()}",
    )
    os.makedirs(out_dir, exist_ok=True)

    delta_all.to_csv(os.path.join(out_dir, "delta_residue_descriptors_all_methods.csv"), index=False)
    summary_all.to_csv(os.path.join(out_dir, "delta_residue_descriptor_summary_all_methods.csv"), index=False)

    print("\nALL RESIDUE DELTAS DONE")
    print("Output:", out_dir)

    return delta_all, summary_all

In [ ]:
delta_residue_all, delta_residue_summary_all = run_residue_delta_all_methods(
    methods=("A", "B", "C"),
    system_a="WT",
    system_b="NAM",
    reps_a=(1, 2, 3),
    reps_b=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=1111,
)

[delta bootstrap] 200/2000
[delta bootstrap] 400/2000
[delta bootstrap] 600/2000
[delta bootstrap] 800/2000
[delta bootstrap] 1000/2000
[delta bootstrap] 1200/2000
[delta bootstrap] 1400/2000
[delta bootstrap] 1600/2000
[delta bootstrap] 1800/2000
[delta bootstrap] 2000/2000
DONE residue delta: MethodA WT - NAM
[delta bootstrap] 200/2000
[delta bootstrap] 400/2000
[delta bootstrap] 600/2000
[delta bootstrap] 800/2000
[delta bootstrap] 1000/2000
[delta bootstrap] 1200/2000
[delta bootstrap] 1400/2000
[delta bootstrap] 1600/2000
[delta bootstrap] 1800/2000
[delta bootstrap] 2000/2000
DONE residue delta: MethodB WT - NAM
[delta bootstrap] 200/2000
[delta bootstrap] 400/2000
[delta bootstrap] 600/2000
[delta bootstrap] 800/2000
[delta bootstrap] 1000/2000
[delta bootstrap] 1200/2000
[delta bootstrap] 1400/2000
[delta bootstrap] 1600/2000
[delta bootstrap] 1800/2000
[delta bootstrap] 2000/2000
DONE residue delta: MethodC WT - NAM

ALL RESIDUE DELTAS DONE
Output: /content/drive/MyDrive/ABE_R

In [ ]:
# ============================================================
# Cell 4F: Region-level WT−NAM descriptor delta
# Delta = WT - NAM
# Region averaging is done inside bootstrap
# Also counts significant residues per region from residue-level delta file
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ALL_DESCRIPTORS = [
    "node_weight",
    "strength",
    "betweenness",
    "closeness",
    "pagerank",
    "eigenvector",
    "MVA",
]


def count_sig_residues_in_regions(delta_df):
    info = {}

    for region, residues in REGIONS.items():
        sub = delta_df[delta_df["residue"].isin(residues)].copy()
        info[region] = {}

        for desc in ALL_DESCRIPTORS:
            sig_col = f"delta_{desc}_significant"
            sig_sub = sub[sub[sig_col] == True]
            sig_res = sorted(sig_sub["residue"].astype(int).tolist())

            info[region][desc] = {
                "n_significant_residues_in_region": int(len(sig_res)),
                "fraction_significant_residues_in_region": float(len(sig_res) / len(sub)) if len(sub) else np.nan,
                "significant_residues": ",".join(map(str, sig_res)),
            }

    return info


def region_delta_one_method(
    method,
    system_a="WT",
    system_b="NAM",
    reps_a=(1, 2, 3),
    reps_b=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=1212,
    make_plots=True,
):
    method_dir = METHOD_DIRS[method]
    base_dir = os.path.join(base_root, method_dir)

    nodes_a, Xa = load_descriptor_system_replicas(method, system_a, reps=reps_a, variant=variant)
    nodes_b, Xb = load_descriptor_system_replicas(method, system_b, reps=reps_b, variant=variant)

    if not np.array_equal(nodes_a, nodes_b):
        raise RuntimeError(f"Node mismatch between {system_a} and {system_b} for {method_dir}")

    reg_idx = get_region_indices_from_nodes(nodes_a)
    region_names = list(reg_idx.keys())

    # Load residue-level delta to count significant residues per region
    residue_delta_path = os.path.join(
        base_dir,
        "ANALYSIS",
        "DESCRIPTOR_PIPELINE",
        variant,
        "DELTA_RESIDUE",
        f"{system_a.upper()}_minus_{system_b.upper()}",
        "delta_residue_descriptors.csv",
    )

    if not os.path.exists(residue_delta_path):
        raise FileNotFoundError(residue_delta_path)

    residue_delta_df = pd.read_csv(residue_delta_path)
    sig_info = count_sig_residues_in_regions(residue_delta_df)

    rng = np.random.default_rng(seed)

    boot = np.empty((n_boot, len(region_names), len(ALL_DESCRIPTORS)), dtype=float)

    for b in range(n_boot):
        mu_a = system_boot_mean_descriptor_delta(Xa, rng)
        mu_b = system_boot_mean_descriptor_delta(Xb, rng)
        delta_nodes = mu_a - mu_b

        for ri, region in enumerate(region_names):
            idx = reg_idx[region]
            boot[b, ri, :] = delta_nodes[idx, :].mean(axis=0)

        if (b + 1) % max(1, n_boot // 10) == 0:
            print(f"[region delta bootstrap] {b+1}/{n_boot}")

    mean = boot.mean(axis=0)
    low = np.quantile(boot, 0.025, axis=0)
    high = np.quantile(boot, 0.975, axis=0)
    std = boot.std(axis=0, ddof=1)
    sem = std / np.sqrt(n_boot)

    out_dir = os.path.join(
        base_dir,
        "ANALYSIS",
        "DESCRIPTOR_PIPELINE",
        variant,
        "DELTA_REGION",
        f"{system_a.upper()}_minus_{system_b.upper()}",
    )
    os.makedirs(out_dir, exist_ok=True)

    rows = []

    for ri, region in enumerate(region_names):
        for mi, desc in enumerate(ALL_DESCRIPTORS):
            rinfo = sig_info[region][desc]

            rows.append({
                "method": method_dir,
                "variant": variant,
                "comparison": f"{system_a.upper()}-{system_b.upper()}",
                "region": region,
                "descriptor": desc,
                "n_residues_region": len(reg_idx[region]),
                "residues_used": ",".join(map(str, REGIONS[region])),

                "n_significant_residues_in_region": rinfo["n_significant_residues_in_region"],
                "fraction_significant_residues_in_region": rinfo["fraction_significant_residues_in_region"],
                "significant_residues": rinfo["significant_residues"],

                "delta_mean": float(mean[ri, mi]),
                "delta_ci_low": float(low[ri, mi]),
                "delta_ci_high": float(high[ri, mi]),
                "delta_std_bootstrap": float(std[ri, mi]),
                "delta_sem_bootstrap": float(sem[ri, mi]),
                "region_significant": bool((low[ri, mi] > 0) or (high[ri, mi] < 0)),
            })

    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(out_dir, "region_delta_descriptors.csv"), index=False)

    # Summary by descriptor
    summary_rows = []

    for desc in ALL_DESCRIPTORS:
        sub = out[out["descriptor"] == desc].copy()
        reg_sig = sub[sub["region_significant"] == True]
        res_sig_regions = sub[sub["n_significant_residues_in_region"] > 0]

        summary_rows.append({
            "method": method_dir,
            "variant": variant,
            "comparison": f"{system_a.upper()}-{system_b.upper()}",
            "descriptor": desc,

            "n_significant_regions": int(len(reg_sig)),
            "fraction_significant_regions": float(len(reg_sig) / len(sub)) if len(sub) else np.nan,
            "region_significant_regions": ",".join(reg_sig["region"].tolist()),

            "total_significant_residues_across_regions": int(sub["n_significant_residues_in_region"].sum()),
            "n_regions_with_any_significant_residue": int(len(res_sig_regions)),
            "regions_with_any_significant_residue": ",".join(res_sig_regions["region"].tolist()),

            "median_fraction_significant_residues_per_region": float(sub["fraction_significant_residues_in_region"].median()),
            "mean_fraction_significant_residues_per_region": float(sub["fraction_significant_residues_in_region"].mean()),

            "median_abs_delta_all_regions": float(sub["delta_mean"].abs().median()),
            "mean_abs_delta_all_regions": float(sub["delta_mean"].abs().mean()),
            "median_ci_width_all_regions": float((sub["delta_ci_high"] - sub["delta_ci_low"]).median()),
            "median_std_bootstrap_all_regions": float(sub["delta_std_bootstrap"].median()),
        })

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(os.path.join(out_dir, "region_delta_descriptor_summary.csv"), index=False)

    if make_plots:
        plot_dir = os.path.join(out_dir, "PLOTS")
        os.makedirs(plot_dir, exist_ok=True)

        for desc in ALL_DESCRIPTORS:
            d = out[out["descriptor"] == desc].copy()

            x = np.arange(len(d))
            y = d["delta_mean"].values
            low_y = d["delta_ci_low"].values
            high_y = d["delta_ci_high"].values
            sig = d["region_significant"].values.astype(bool)

            yerr = np.vstack([
                np.maximum(0, y - low_y),
                np.maximum(0, high_y - y),
            ])

            plt.figure(figsize=(12, 4.8))

            idx_ns = np.where(~sig)[0]
            idx_s = np.where(sig)[0]

            plt.bar(x[idx_ns], y[idx_ns], alpha=0.45, label="Region not significant")
            plt.bar(x[idx_s], y[idx_s], alpha=0.95, label="Region significant")
            plt.errorbar(x, y, yerr=yerr, fmt="none", capsize=3)

            plt.axhline(0, linewidth=1)
            plt.xticks(x, d["region"].values, rotation=45, ha="right")
            plt.ylabel(f"Region Δ {desc}")
            plt.title(f"{method_dir}: region Δ {desc} ({system_a.upper()} − {system_b.upper()})")
            plt.legend()
            plt.tight_layout()
            plt.savefig(os.path.join(plot_dir, f"region_delta_{desc}.png"), dpi=300)
            plt.close()

        # significant region count
        s = summary.sort_values("n_significant_regions", ascending=False)
        plt.figure(figsize=(8, 4.8))
        x = np.arange(len(s))
        plt.bar(x, s["n_significant_regions"].values)
        plt.xticks(x, s["descriptor"].values, rotation=45, ha="right")
        plt.ylabel("Number of significant regions")
        plt.title(f"{method_dir}: significant regions by descriptor")
        plt.tight_layout()
        plt.savefig(os.path.join(plot_dir, "descriptor_significant_regions_bar.png"), dpi=300)
        plt.close()

        # regions containing significant residues
        s2 = summary.sort_values("n_regions_with_any_significant_residue", ascending=False)
        plt.figure(figsize=(8, 4.8))
        x = np.arange(len(s2))
        plt.bar(x, s2["n_regions_with_any_significant_residue"].values)
        plt.xticks(x, s2["descriptor"].values, rotation=45, ha="right")
        plt.ylabel("Regions with ≥1 significant residue")
        plt.title(f"{method_dir}: regions with residue-level signal")
        plt.tight_layout()
        plt.savefig(os.path.join(plot_dir, "descriptor_regions_with_significant_residues_bar.png"), dpi=300)
        plt.close()

    with open(os.path.join(out_dir, "summary.txt"), "w") as f:
        f.write("Region-level descriptor delta\n")
        f.write(f"Method: {method_dir}\n")
        f.write(f"Variant: {variant}\n")
        f.write(f"Comparison: {system_a.upper()} - {system_b.upper()}\n")
        f.write(f"Bootstrap: n_boot={n_boot}, seed={seed}\n")
        f.write("Region averaging was performed inside each bootstrap draw.\n")
        f.write("Residue-level significant residue counts were taken from DELTA_RESIDUE output.\n")

    print(f"DONE region delta: {method_dir} {system_a.upper()} - {system_b.upper()}")
    return out, summary


def run_region_delta_all_methods(
    methods=("A", "B", "C"),
    system_a="WT",
    system_b="NAM",
    reps_a=(1, 2, 3),
    reps_b=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=1212,
):
    all_region = []
    all_summary = []

    for method in methods:
        region_df, summary_df = region_delta_one_method(
            method=method,
            system_a=system_a,
            system_b=system_b,
            reps_a=reps_a,
            reps_b=reps_b,
            variant=variant,
            n_boot=n_boot,
            seed=seed,
            make_plots=True,
        )

        all_region.append(region_df)
        all_summary.append(summary_df)

    region_all = pd.concat(all_region, ignore_index=True)
    summary_all = pd.concat(all_summary, ignore_index=True)

    out_dir = os.path.join(
        base_root,
        "DESCRIPTOR_COMPARISON",
        variant,
        "DELTA_REGION",
        f"{system_a.upper()}_minus_{system_b.upper()}",
    )
    os.makedirs(out_dir, exist_ok=True)

    region_all.to_csv(os.path.join(out_dir, "region_delta_descriptors_all_methods.csv"), index=False)
    summary_all.to_csv(os.path.join(out_dir, "region_delta_descriptor_summary_all_methods.csv"), index=False)

    print("\nALL REGION DELTAS DONE")
    print("Output:", out_dir)

    return region_all, summary_all

In [ ]:
region_delta_all, region_delta_summary_all = run_region_delta_all_methods(
    methods=("A", "B", "C"),
    system_a="WT",
    system_b="NAM",
    reps_a=(1, 2, 3),
    reps_b=(1, 2, 3),
    variant="all_sum",
    n_boot=2000,
    seed=1212,
)

[region delta bootstrap] 200/2000
[region delta bootstrap] 400/2000
[region delta bootstrap] 600/2000
[region delta bootstrap] 800/2000
[region delta bootstrap] 1000/2000
[region delta bootstrap] 1200/2000
[region delta bootstrap] 1400/2000
[region delta bootstrap] 1600/2000
[region delta bootstrap] 1800/2000
[region delta bootstrap] 2000/2000
DONE region delta: MethodA WT - NAM
[region delta bootstrap] 200/2000
[region delta bootstrap] 400/2000
[region delta bootstrap] 600/2000
[region delta bootstrap] 800/2000
[region delta bootstrap] 1000/2000
[region delta bootstrap] 1200/2000
[region delta bootstrap] 1400/2000
[region delta bootstrap] 1600/2000
[region delta bootstrap] 1800/2000
[region delta bootstrap] 2000/2000
DONE region delta: MethodB WT - NAM
[region delta bootstrap] 200/2000
[region delta bootstrap] 400/2000
[region delta bootstrap] 600/2000
[region delta bootstrap] 800/2000
[region delta bootstrap] 1000/2000
[region delta bootstrap] 1200/2000
[region delta bootstrap] 1400/

In [ ]:
# ============================================================
# Cell 3: Edge faithfulness for MethodA / MethodB / MethodC
# Requires Cell 1 common loader
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt


def get_all_layer_edge_importance(model):
    """
    all_sum attention importance:
    sum over heads + sum over layers.
    Returns [E].
    """
    a1 = model.gat1.get_last_alpha().sum(dim=1)
    a2 = model.gat2.get_last_alpha().sum(dim=1)
    a3 = model.gat3.get_last_alpha().sum(dim=1)
    a4 = model.gat4.get_last_alpha().sum(dim=1)
    return a1 + a2 + a3 + a4


def make_edge_mask(edge_imp, k_frac):
    E = edge_imp.numel()
    k = max(1, int(round(k_frac * E)))
    _, idx = torch.topk(edge_imp, k=k, largest=True)

    mask = torch.ones(E, device=edge_imp.device, dtype=edge_imp.dtype)
    mask[idx] = 0.0
    return mask


def make_random_edge_mask(E, k_frac, device, dtype, rng):
    k = max(1, int(round(k_frac * E)))
    idx = rng.choice(E, size=k, replace=False)

    mask = torch.ones(E, device=device, dtype=dtype)
    mask[idx] = 0.0
    return mask


def make_blocks(n_frames, block_size):
    blocks = []
    for start in range(0, n_frames - 1, block_size):
        end = min(n_frames - 1, start + block_size)
        if end > start:
            blocks.append((start, end))
    return blocks


def edge_faithfulness_one_replica(
    method,
    system,
    rep_id,
    k_values=range(1, 51),
    block_size=500,
    n_random=20,
    device="cuda",
    seed=123,
    make_plots=True,
):
    """
    Runs edge faithfulness for one method/system/replica.

    For each frame:
      baseline RMSD
      top-K edge masked RMSD
      random-K edge masked RMSD

    Saved:
      edge_faithfulness_framewise.csv
      edge_faithfulness_blockwise.csv
      edge_faithfulness_summary_by_K.csv
      edge_faithfulness_auc_summary.csv
    """

    model, data_cpu, paths, arch, device_obj = load_model_and_data(
        method=method,
        system=system,
        rep_id=rep_id,
        device=device,
        analysis_subdir="EDGE_FAITHFULNESS",
    )

    out_dir = paths.analysis_dir
    os.makedirs(out_dir, exist_ok=True)

    n_frames = len(data_cpu)
    blocks = make_blocks(n_frames, block_size)

    rng = np.random.default_rng(seed + rep_id)

    frame_rows = []
    block_rows = []

    model.eval()

    for k_pct in k_values:
        k_frac = k_pct / 100.0

        for bi, (bs, be) in enumerate(blocks):
            block_records = []

            for t in range(bs, be):
                d0 = to_device_graph(data_cpu[t], device_obj)
                d1 = to_device_graph(data_cpu[t + 1], device_obj)

                true_next = d1.x[:, :3]

                # baseline
                model.clear_masks()
                with torch.no_grad():
                    pred_base = model(d0.x, d0.edge_index, d0.edge_attr)
                    base_rmsd = kabsch_rmsd(pred_base, true_next).item()

                # get alpha importance from unmasked pass
                model.clear_masks()
                with torch.no_grad():
                    _ = model(d0.x, d0.edge_index, d0.edge_attr)
                    edge_imp = get_all_layer_edge_importance(model)

                E = edge_imp.numel()

                # top-K mask
                top_mask = make_edge_mask(edge_imp, k_frac)
                model.set_masks_all_layers(top_mask)

                with torch.no_grad():
                    pred_top = model(d0.x, d0.edge_index, d0.edge_attr)
                    top_rmsd = kabsch_rmsd(pred_top, true_next).item()

                delta_top = top_rmsd - base_rmsd

                # random masks
                random_rmsds = []
                for rr in range(n_random):
                    rand_mask = make_random_edge_mask(
                        E,
                        k_frac,
                        edge_imp.device,
                        edge_imp.dtype,
                        rng,
                    )
                    model.set_masks_all_layers(rand_mask)

                    with torch.no_grad():
                        pred_rand = model(d0.x, d0.edge_index, d0.edge_attr)
                        rand_rmsd = kabsch_rmsd(pred_rand, true_next).item()

                    random_rmsds.append(rand_rmsd)

                model.clear_masks()

                rand_mean = float(np.mean(random_rmsds))
                rand_sd = float(np.std(random_rmsds, ddof=1)) if len(random_rmsds) > 1 else 0.0
                delta_rand = rand_mean - base_rmsd
                advantage = delta_top - delta_rand

                record = {
                    "method": paths.method,
                    "system": system.upper(),
                    "rep_id": rep_id,
                    "frame": t,
                    "target_frame": t + 1,
                    "block_id": bi,
                    "K_pct": k_pct,
                    "K_frac": k_frac,
                    "n_edges": E,
                    "base_rmsd": base_rmsd,
                    "top_rmsd": top_rmsd,
                    "random_rmsd_mean": rand_mean,
                    "random_rmsd_sd": rand_sd,
                    "delta_top": delta_top,
                    "delta_random": delta_rand,
                    "faithfulness_advantage": advantage,
                }

                frame_rows.append(record)
                block_records.append(record)

            bdf = pd.DataFrame(block_records)

            block_rows.append({
                "method": paths.method,
                "system": system.upper(),
                "rep_id": rep_id,
                "block_id": bi,
                "frame_start": bs,
                "frame_end": be,
                "K_pct": k_pct,
                "K_frac": k_frac,
                "base_rmsd_mean": bdf["base_rmsd"].mean(),
                "top_rmsd_mean": bdf["top_rmsd"].mean(),
                "random_rmsd_mean": bdf["random_rmsd_mean"].mean(),
                "delta_top_mean": bdf["delta_top"].mean(),
                "delta_random_mean": bdf["delta_random"].mean(),
                "faithfulness_advantage_mean": bdf["faithfulness_advantage"].mean(),
                "faithfulness_advantage_sd_frames": bdf["faithfulness_advantage"].std(ddof=1),
                "n_frames_block": len(bdf),
            })

        print(f"[DONE] {paths.method} {system.upper()} rep{rep_id} K={k_pct}%")

    frame_df = pd.DataFrame(frame_rows)
    block_df = pd.DataFrame(block_rows)

    frame_df.to_csv(os.path.join(out_dir, "edge_faithfulness_framewise.csv"), index=False)
    block_df.to_csv(os.path.join(out_dir, "edge_faithfulness_blockwise.csv"), index=False)

    # summary by K across blocks
    summary_rows = []

    for k_pct, g in block_df.groupby("K_pct"):
        k_frac = g["K_frac"].iloc[0]

        summary_rows.append({
            "method": paths.method,
            "system": system.upper(),
            "rep_id": rep_id,
            "K_pct": k_pct,
            "K_frac": k_frac,

            "base_rmsd_mean": g["base_rmsd_mean"].mean(),
            "base_rmsd_sd_blocks": g["base_rmsd_mean"].std(ddof=1),

            "top_rmsd_mean": g["top_rmsd_mean"].mean(),
            "top_rmsd_sd_blocks": g["top_rmsd_mean"].std(ddof=1),

            "random_rmsd_mean": g["random_rmsd_mean"].mean(),
            "random_rmsd_sd_blocks": g["random_rmsd_mean"].std(ddof=1),

            "delta_top_mean": g["delta_top_mean"].mean(),
            "delta_top_sd_blocks": g["delta_top_mean"].std(ddof=1),

            "delta_random_mean": g["delta_random_mean"].mean(),
            "delta_random_sd_blocks": g["delta_random_mean"].std(ddof=1),

            "faithfulness_advantage_mean": g["faithfulness_advantage_mean"].mean(),
            "faithfulness_advantage_sd_blocks": g["faithfulness_advantage_mean"].std(ddof=1),

            "n_blocks": len(g),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("K_pct")
    summary_df.to_csv(os.path.join(out_dir, "edge_faithfulness_summary_by_K.csv"), index=False)

    # AUC over K_frac
    x = summary_df["K_frac"].values.astype(float)

    auc_delta_top = float(np.trapz(summary_df["delta_top_mean"].values, x))
    auc_delta_random = float(np.trapz(summary_df["delta_random_mean"].values, x))
    auc_advantage = float(np.trapz(summary_df["faithfulness_advantage_mean"].values, x))

    auc_df = pd.DataFrame([{
        "method": paths.method,
        "system": system.upper(),
        "rep_id": rep_id,
        "auc_delta_top": auc_delta_top,
        "auc_delta_random": auc_delta_random,
        "auc_advantage_top_minus_random": auc_advantage,
        "K_min": min(k_values),
        "K_max": max(k_values),
        "n_random": n_random,
        "block_size": block_size,
    }])

    auc_df.to_csv(os.path.join(out_dir, "edge_faithfulness_auc_summary.csv"), index=False)

    with open(os.path.join(out_dir, "summary.txt"), "w") as f:
        f.write(f"Edge faithfulness: {paths.method} {system.upper()} rep{rep_id}\n")
        f.write(f"Model: {paths.model_file}\n")
        f.write(f"Data : {paths.data_file}\n")
        f.write(f"K range: {min(k_values)}-{max(k_values)}%\n")
        f.write(f"Block size: {block_size}\n")
        f.write(f"Random masks per frame: {n_random}\n\n")
        f.write("Definitions:\n")
        f.write("delta_top = RMSD(top-K masked) - RMSD(baseline)\n")
        f.write("delta_random = RMSD(random-K masked) - RMSD(baseline)\n")
        f.write("faithfulness_advantage = delta_top - delta_random\n\n")
        f.write(f"AUC delta_top: {auc_delta_top}\n")
        f.write(f"AUC delta_random: {auc_delta_random}\n")
        f.write(f"AUC advantage: {auc_advantage}\n")

    if make_plots:
        plt.figure(figsize=(7, 4.8))
        plt.plot(summary_df["K_pct"], summary_df["delta_top_mean"], marker="o", label="Top-K")
        plt.plot(summary_df["K_pct"], summary_df["delta_random_mean"], marker="o", label="Random-K")
        plt.xlabel("Masked edges (%)")
        plt.ylabel("Δ RMSD")
        plt.title(f"{paths.method} {system.upper()} rep{rep_id}: edge faithfulness")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "edge_faithfulness_delta_curve.png"), dpi=300)
        plt.close()

        plt.figure(figsize=(7, 4.8))
        plt.plot(summary_df["K_pct"], summary_df["faithfulness_advantage_mean"], marker="o")
        plt.axhline(0, linewidth=1)
        plt.xlabel("Masked edges (%)")
        plt.ylabel("Top-K − Random-K Δ RMSD")
        plt.title(f"{paths.method} {system.upper()} rep{rep_id}: edge faithfulness advantage")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "edge_faithfulness_advantage_curve.png"), dpi=300)
        plt.close()

    print(f"EDGE FAITHFULNESS DONE: {paths.method} {system.upper()} rep{rep_id}")
    return auc_df, summary_df


def run_edge_faithfulness_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    k_values=range(1, 51),
    block_size=500,
    n_random=20,
    device="cuda",
):
    all_auc = []

    for method in methods:
        for system in systems:
            for rep in reps:
                auc_df, _ = edge_faithfulness_one_replica(
                    method=method,
                    system=system,
                    rep_id=rep,
                    k_values=k_values,
                    block_size=block_size,
                    n_random=n_random,
                    device=device,
                    seed=123,
                    make_plots=True,
                )
                all_auc.append(auc_df)

    auc_all = pd.concat(all_auc, ignore_index=True)

    comparison_dir = os.path.join(base_root, "EDGE_FAITHFULNESS_COMPARISON")
    os.makedirs(comparison_dir, exist_ok=True)

    auc_all.to_csv(
        os.path.join(comparison_dir, "edge_faithfulness_auc_all_methods_replicas.csv"),
        index=False,
    )

    system_summary = (
        auc_all.groupby(["method", "system"])
        .agg(
            auc_delta_top_mean=("auc_delta_top", "mean"),
            auc_delta_top_sd=("auc_delta_top", "std"),
            auc_delta_random_mean=("auc_delta_random", "mean"),
            auc_delta_random_sd=("auc_delta_random", "std"),
            auc_advantage_mean=("auc_advantage_top_minus_random", "mean"),
            auc_advantage_sd=("auc_advantage_top_minus_random", "std"),
            n_reps=("rep_id", "count"),
        )
        .reset_index()
    )

    system_summary.to_csv(
        os.path.join(comparison_dir, "edge_faithfulness_auc_by_method_system.csv"),
        index=False,
    )

    print("\nALL EDGE FAITHFULNESS DONE")
    print("Comparison:", comparison_dir)

    return auc_all, system_summary

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

True
0
Tesla T4


In [ ]:
edge_auc_all, edge_auc_summary = run_edge_faithfulness_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    k_values=[1, 2, 5, 10, 15, 20, 30, 40, 50],
    block_size=500,
    n_random=5,
    device="cuda",
)

Loaded:
 method     : MethodA
 system     : WT
 replica    : 1
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgegat_dynamics_best_WT_rep1.pt
 data file  : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/trajectory_graphs_phi_psi_rmsf_WT_rep1.pt
 analysis   : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/EDGE_FAITHFULNESS/WT_rep1
 arch       : Arch(in_features=6, hidden_dim=128, out_features=3, heads=4, edge_dim=3, conv_bias=True, model_bias=True)
[DONE] MethodA WT rep1 K=1%
[DONE] MethodA WT rep1 K=2%
[DONE] MethodA WT rep1 K=5%
[DONE] MethodA WT rep1 K=10%
[DONE] MethodA WT rep1 K=15%
[DONE] MethodA WT rep1 K=20%
[DONE] MethodA WT rep1 K=30%
[DONE] MethodA WT rep1 K=40%
[DONE] MethodA WT rep1 K=50%


/tmp/ipykernel_2740/2715167718.py:253: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc_delta_top = float(np.trapz(summary_df["delta_top_mean"].values, x))
/tmp/ipykernel_2740/2715167718.py:254: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc_delta_random = float(np.trapz(summary_df["delta_random_mean"].values, x))
/tmp/ipykernel_2740/2715167718.py:255: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc_advantage = float(np.trapz(summary_df["faithfulness_advantage_mean"].values, x))


EDGE FAITHFULNESS DONE: MethodA WT rep1
Loaded:
 method     : MethodA
 system     : WT
 replica    : 2
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgegat_dynamics_best_WT_rep2.pt
 data file  : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/trajectory_graphs_phi_psi_rmsf_WT_rep2.pt
 analysis   : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/EDGE_FAITHFULNESS/WT_rep2
 arch       : Arch(in_features=6, hidden_dim=128, out_features=3, heads=4, edge_dim=3, conv_bias=True, model_bias=True)
[DONE] MethodA WT rep2 K=1%
[DONE] MethodA WT rep2 K=2%
[DONE] MethodA WT rep2 K=5%
[DONE] MethodA WT rep2 K=10%
[DONE] MethodA WT rep2 K=15%
[DONE] MethodA WT rep2 K=20%
[DONE] MethodA WT rep2 K=30%
[DONE] MethodA WT rep2 K=40%
[DONE] MethodA WT rep2 K=50%
EDGE FAITHFULNESS DONE: MethodA WT rep2
Loaded:
 method     : MethodA
 system     : WT
 replica    : 3
 model file : /content/drive/MyDrive/ABE_RF_Diffusi

In [ ]:
edge_auc_all, edge_auc_summary = run_edge_faithfulness_all(
    methods=("A", "B", "C"),
    systems=("WT", "NAM"),
    reps=(1, 2, 3),
    k_values=range(1, 51),
    block_size=500,
    n_random=20,
    device="cuda",
)

Loaded:
 method     : MethodA
 system     : WT
 replica    : 1
 model file : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/edgegat_dynamics_best_WT_rep1.pt
 data file  : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/WT/trajectory_graphs_phi_psi_rmsf_WT_rep1.pt
 analysis   : /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/MethodA/ANALYSIS/EDGE_FAITHFULNESS/WT_rep1
 arch       : Arch(in_features=6, hidden_dim=128, out_features=3, heads=4, edge_dim=3, conv_bias=True, model_bias=True)
[DONE] MethodA WT rep1 K=1%
[DONE] MethodA WT rep1 K=2%
[DONE] MethodA WT rep1 K=3%
[DONE] MethodA WT rep1 K=4%
[DONE] MethodA WT rep1 K=5%
[DONE] MethodA WT rep1 K=6%
[DONE] MethodA WT rep1 K=7%
[DONE] MethodA WT rep1 K=8%
[DONE] MethodA WT rep1 K=9%


KeyboardInterrupt: 

In [ ]:
#To Calculate RMSD True and Predicted and saving then as text file

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax

class EdgeGATConv_MethodA(MessagePassing):
    def __init__(self, in_channels, out_channels, edge_dim=3, heads=1, concat=True, dropout=0.0, bias=True):
        super().__init__(aggr='add', node_dim=0)  # Ensures node-wise aggregation
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.concat = concat
        self.dropout = dropout

        # Linear transformation for node features
        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)

        # Linear transformation for edge features — now 3D input
        self.edge_lin = nn.Linear(edge_dim, heads * out_channels, bias=False)

        # Attention parameters: one set per head
        self.att = nn.Parameter(torch.empty(heads, 3 * out_channels))  # [H, 3C]
        self.leaky_relu = nn.LeakyReLU(0.2)

        if bias:
            self.bias = nn.Parameter(torch.Tensor(heads * out_channels if concat else out_channels))
        else:
            self.register_parameter('bias', None)

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.edge_lin.weight)
        nn.init.xavier_uniform_(self.att)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, edge_attr):
        # x: [N, in_channels]
        # edge_attr: [E, 3]
        x = self.lin(x)  # [N, H * C]
        edge_attr = self.edge_lin(edge_attr)  # [E, H * C]
        return self.propagate(edge_index=edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr, index):
        # Reshape for multi-head attention
        x_i = x_i.view(-1, self.heads, self.out_channels)         # [E, H, C]
        x_j = x_j.view(-1, self.heads, self.out_channels)
        edge_attr = edge_attr.view(-1, self.heads, self.out_channels)

        # Concatenate: [x_i || x_j || edge_attr] → [E, H, 3C]
        cat = torch.cat([x_i, x_j, edge_attr], dim=-1)

        # Compute attention scores
        alpha = torch.einsum("ehc,hc->eh", cat, self.att)  # [E, H]
        alpha = self.leaky_relu(alpha)
        alpha = softmax(alpha, index)  # Normalize across incoming edges
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        return x_j * alpha.unsqueeze(-1)  # Apply attention to messages

    def update(self, aggr_out):
        # Combine attention heads
        if self.concat:
            aggr_out = aggr_out.view(-1, self.heads * self.out_channels)  # [N, H*C]
        else:
            aggr_out = aggr_out.mean(dim=1)  # [N, C]

        if self.bias is not None:
            aggr_out += self.bias

        return aggr_out


class EdgeGATDynamics(nn.Module):
    def __init__(self, in_features=6, hidden_dim=128, out_features=3, heads=4):
        super().__init__()
        self.gat1 = EdgeGATConv_MethodA(in_features, hidden_dim, edge_dim=3, heads=heads, concat=True)
        self.gat2 = EdgeGATConv_MethodA(hidden_dim * heads, hidden_dim, edge_dim=3, heads=heads, concat=True)
        self.gat3 = EdgeGATConv_MethodA(hidden_dim * heads, hidden_dim, edge_dim=3, heads=heads, concat=True)
        self.gat4 = EdgeGATConv_MethodA(hidden_dim * heads, hidden_dim, edge_dim=3, heads=1, concat=False)
        self.fc = nn.Linear(hidden_dim, out_features)
        self.bias = nn.Parameter(torch.ones(1))

    def forward(self, x, edge_index, edge_attr):
        x = self.gat1(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.gat2(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.gat3(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.gat4(x, edge_index, edge_attr)
        out = self.fc(x)
        return out * self.bias


model_path = os.path.join(output_dir, "edgegat_dynamics_best_NAM_rep3.pt")
data_file = os.path.join(output_dir, 'trajectory_graphs_phi_psi_rmsf_NAM_rep3.pt')

import os
import torch

# Set the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = EdgeGATDynamics(in_features=6, hidden_dim=128, out_features=3, heads=4).to(device)

# Load the data
data_list_cpu = torch.load(data_file, weights_only=False)
data_list = [d.to(device) for d in data_list_cpu] # Move data to the correct device

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print("✅ Model loaded successfully on device:", device)

def kabsch_align(P, Q):
    """
    Aligns P (pred) to Q (true) using Kabsch algorithm.
    Returns: P_aligned [N, 3]
    """
    # Subtract centroids
    P_mean = P.mean(dim=0, keepdim=True)
    Q_mean = Q.mean(dim=0, keepdim=True)

    P_centered = P - P_mean
    Q_centered = Q - Q_mean

    # Compute covariance matrix
    C = torch.matmul(P_centered.T, Q_centered)

    # SVD
    V, S, Wt = torch.linalg.svd(C)
    d = torch.det(torch.matmul(V, Wt))
    D = torch.diag(torch.tensor([1., 1., d], device=P.device))

    # Rotation
    U = torch.matmul(torch.matmul(V, D), Wt)

    # Rotate and re-center
    P_rot = torch.matmul(P_centered, U)

    return P_rot + Q_mean

    def compute_kabsch_rmsd(P, Q):
    P_aligned = kabsch_align(P, Q)
    return torch.sqrt(((P_aligned - Q) ** 2).sum(dim=1).mean()).item()

    import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

kabsch_rmsds = []

model.eval()
with torch.no_grad():
    for i in tqdm(range(len(data_list) - 1)):
        d0 = data_list[i]
        d1 = data_list[i + 1]

        pred = model(d0.x, d0.edge_index, d0.edge_attr)
        true = d1.x[:, :3]

        rmsd_kabsch = compute_kabsch_rmsd(pred, true)
        kabsch_rmsds.append(rmsd_kabsch)

# Convert to numpy
kabsch_rmsds = np.array(kabsch_rmsds)
frame_indices = np.arange(len(kabsch_rmsds))

#Save text output for each replica

# Store traditional RMSD for both predicted and true trajectories
pred_rmsds = []
true_rmsds = []

model.eval()
with torch.no_grad():
    # First frame as reference
    ref_frame = data_list[0].x[:, :3]

    for i in tqdm(range(len(data_list) - 1)):
        d0 = data_list[i]
        d1 = data_list[i + 1]

        pred = model(d0.x, d0.edge_index, d0.edge_attr)
        true = d1.x[:, :3]

        pred_rmsds.append(compute_kabsch_rmsd(pred, ref_frame))
        true_rmsds.append(compute_kabsch_rmsd(true, ref_frame))

# Convert to numpy
pred_rmsds = np.array(pred_rmsds)
true_rmsds = np.array(true_rmsds)
frame_indices = np.arange(len(pred_rmsds))

mean_true = true_rmsds.mean()
mean_pred = pred_rmsds.mean()
#Save text output for each replica



In [ ]:
import os
os.makedirs("/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/EDGE_FAITHFULNESS", exist_ok=True)

In [ ]:
%%writefile /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/EDGE_FAITHFULNESS/edge_faithfulness_coherent.py
# ====== START SCRIPT ======
# (PASTE THE FULL SCRIPT I GAVE YOU LAST — UNCHANGED)
# ====== END SCRIPT ======

#!/usr/bin/env python3
"""
edge_faithfulness_coherent.py  (ROBUST + COHERENT PATHS)

Your folder layout:
  {base_dir}/WT/trajectory_graphs_phi_psi_rmsf_WT_rep3.pt
  {base_dir}/WT/edgegat_dynamics_best_WT_rep3.pt
  {base_dir}/NAM/trajectory_graphs_phi_psi_rmsf_NAM_rep2.pt
  {base_dir}/NAM/edgegat_dynamics_best_NAM_rep2.pt

This script:
- Auto-infers model architecture from the checkpoint (hidden_dim, heads, in_features, edge_dim).
- Rebuilds the exact model class (including presence/absence of per-layer bias and model-level scalar bias).
- Caches attention α (post-softmax) per layer, per forward.
- Runs edge faithfulness curves (mask top-K% edges by α importance vs random).
- Uses block means to reduce time-correlation artifacts.
- Writes CSV + TXT outputs under:
    {base_dir}/ANALYSIS/EDGE_FAITHFULNESS/{SYSTEM}_rep{rep}/

Run in Colab:
!python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/EDGE_FAITHFULNESS/edge_faithfulness_coherent.py \
  --base_dir "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR" \
  --system WT --rep_id 3 \
  --block_size 500 \
  --k_min 1 --k_max 50 --k_step 1 \
  --n_random 20 --metric both

Optional:
  --make_plots
  --device cuda   (or cpu)
"""

import os
import json
import argparse
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


# ============================================================
# Coherent path builder (system-folder layout)
# ============================================================

@dataclass
class RunPaths:
    data_file: str
    model_file: str
    out_dir: str

def build_paths(base_dir: str, system: str, rep_id: int,
                analysis_root: str = "ANALYSIS",
                analysis_subdir: str = "EDGE_FAITHFULNESS",
                tag: Optional[str] = None) -> RunPaths:
    system = system.upper()
    sys_dir = os.path.join(base_dir, system)

    data_file = os.path.join(sys_dir, f"trajectory_graphs_phi_psi_rmsf_{system}_rep{rep_id}.pt")
    model_file = os.path.join(sys_dir, f"edgegat_dynamics_best_{system}_rep{rep_id}.pt")

    suffix = f"{system}_rep{rep_id}" if tag is None else f"{system}_rep{rep_id}_{tag}"
    out_dir = os.path.join(base_dir, analysis_root, analysis_subdir, suffix)
    os.makedirs(out_dir, exist_ok=True)

    return RunPaths(data_file=data_file, model_file=model_file, out_dir=out_dir)


# ============================================================
# Geometry utilities
# ============================================================

def kabsch_rmsd(P: torch.Tensor, Q: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    """
    Kabsch RMSD between P and Q (Nx3), returns scalar tensor.
    """
    P_center = P - P.mean(dim=0, keepdim=True)
    Q_center = Q - Q.mean(dim=0, keepdim=True)

    C = P_center.T @ Q_center
    V, _, Wt = torch.linalg.svd(C)
    d = torch.sign(torch.det(V @ Wt))
    D = torch.diag(torch.tensor([1.0, 1.0, d], device=P.device, dtype=P.dtype))
    U = V @ D @ Wt

    P_rot = P_center @ U
    diff = P_rot - Q_center
    return torch.sqrt((diff * diff).sum() / (P.shape[0] + eps))

def mse_loss(P: torch.Tensor, Q: torch.Tensor) -> torch.Tensor:
    return torch.mean((P - Q) ** 2)


# ============================================================
# Robust model definition (matches train_wt.py style)
# ============================================================

class EdgeGATConv(MessagePassing):
    """
    Edge-aware attention (Meaning A): e_ij = a^T [x_i || x_j || e_ij], alpha = softmax(e_ij)
    Robust to your checkpoint:
      - att shape: [H, 3C]
      - optional per-layer bias param (keys: gat*.bias)
    Adds:
      - caches last alpha (post-softmax) in self._last_alpha: [E,H]
      - supports soft alpha mask (multiply alpha by mask; no renorm)
    """
    def __init__(self, in_channels, out_channels, edge_dim, heads, concat=True,
                 dropout=0.0, conv_bias=True):
        super().__init__(aggr='add', node_dim=0)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        self.heads = heads
        self.concat = concat
        self.dropout = dropout

        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.edge_lin = nn.Linear(edge_dim, heads * out_channels, bias=False)

        # IMPORTANT: matches your training conv: [H, 3C]
        self.att = nn.Parameter(torch.empty(heads, 3 * out_channels))
        self.leaky_relu = nn.LeakyReLU(0.2)

        if conv_bias:
            out_dim = heads * out_channels if concat else out_channels
            self.bias = nn.Parameter(torch.zeros(out_dim))
        else:
            self.register_parameter('bias', None)

        # analysis helpers
        self._last_alpha: Optional[torch.Tensor] = None  # [E,H]
        self._alpha_mask: Optional[torch.Tensor] = None  # [E] or [E,1] or [E,H]

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.edge_lin.weight)
        nn.init.xavier_uniform_(self.att)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def set_alpha_mask(self, mask: Optional[torch.Tensor]):
        self._alpha_mask = mask

    def clear_alpha_mask(self):
        self._alpha_mask = None

    def get_last_alpha(self) -> torch.Tensor:
        if self._last_alpha is None:
            raise RuntimeError("α not cached yet. Run a forward pass first.")
        return self._last_alpha

    def forward(self, x, edge_index, edge_attr):
        x = self.lin(x)                 # [N, H*C]
        edge_attr = self.edge_lin(edge_attr)  # [E, H*C]
        return self.propagate(edge_index=edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr, index):
        # reshape to [E,H,C]
        x_i = x_i.view(-1, self.heads, self.out_channels)
        x_j = x_j.view(-1, self.heads, self.out_channels)
        edge_attr = edge_attr.view(-1, self.heads, self.out_channels)

        cat = torch.cat([x_i, x_j, edge_attr], dim=-1)     # [E,H,3C]
        e = torch.einsum("ehc,hc->eh", cat, self.att)      # [E,H]
        e = self.leaky_relu(e)
        alpha = softmax(e, index)                           # [E,H]
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        self._last_alpha = alpha.detach()

        if self._alpha_mask is not None:
            m = self._alpha_mask
            if m.ndim == 1:
                m = m.view(-1, 1)  # [E,1]
            alpha = alpha * m

        return x_j * alpha.unsqueeze(-1)                    # [E,H,C]

    def update(self, aggr_out):
        # aggr_out: [N,H,C]
        if self.concat:
            out = aggr_out.reshape(-1, self.heads * self.out_channels)   # [N,H*C]
        else:
            out = aggr_out.mean(dim=1)                                   # [N,C]
        if self.bias is not None:
            out = out + self.bias
        return out


class EdgeGATDynamics(nn.Module):
    """
    4-layer EdgeGAT + Linear head.
    Robust to:
      - optional model-level scalar bias (checkpoint key "bias")
    """
    def __init__(self, in_features, hidden_dim, out_features, heads, edge_dim,
                 conv_bias: bool, model_bias: bool):
        super().__init__()
        self.gat1 = EdgeGATConv(in_features, hidden_dim, edge_dim=edge_dim, heads=heads,
                                concat=True, conv_bias=conv_bias)
        self.gat2 = EdgeGATConv(hidden_dim * heads, hidden_dim, edge_dim=edge_dim, heads=heads,
                                concat=True, conv_bias=conv_bias)
        self.gat3 = EdgeGATConv(hidden_dim * heads, hidden_dim, edge_dim=edge_dim, heads=heads,
                                concat=True, conv_bias=conv_bias)
        self.gat4 = EdgeGATConv(hidden_dim * heads, hidden_dim, edge_dim=edge_dim, heads=1,
                                concat=False, conv_bias=conv_bias)
        self.fc = nn.Linear(hidden_dim, out_features)

        if model_bias:
            # IMPORTANT: many of your checkpoints include key "bias" (scalar)
            self.bias = nn.Parameter(torch.zeros(1))
        else:
            self.register_parameter("bias", None)

    def clear_masks(self):
        self.gat1.clear_alpha_mask()
        self.gat2.clear_alpha_mask()
        self.gat3.clear_alpha_mask()
        self.gat4.clear_alpha_mask()

    def set_masks_all_layers(self, mask_vec: torch.Tensor):
        self.gat1.set_alpha_mask(mask_vec)
        self.gat2.set_alpha_mask(mask_vec)
        self.gat3.set_alpha_mask(mask_vec)
        self.gat4.set_alpha_mask(mask_vec)

    def forward(self, x, edge_index, edge_attr):
        x = self.gat1(x, edge_index, edge_attr); x = F.relu(x)
        x = self.gat2(x, edge_index, edge_attr); x = F.relu(x)
        x = self.gat3(x, edge_index, edge_attr); x = F.relu(x)
        x = self.gat4(x, edge_index, edge_attr)
        out = self.fc(x)
        if self.bias is not None:
            out = out * self.bias
        return out


# ============================================================
# Auto-infer architecture from checkpoint
# ============================================================

@dataclass
class Arch:
    in_features: int
    hidden_dim: int
    out_features: int
    heads: int
    edge_dim: int
    conv_bias: bool
    model_bias: bool

def infer_arch_from_state_dict(sd: Dict[str, torch.Tensor]) -> Arch:
    """
    Infers {in_features, hidden_dim, out_features, heads, edge_dim} from a checkpoint.
    Expected keys (from your training):
      - fc.weight: [out_features, hidden_dim]
      - gat1.att: [heads, 3*hidden_dim]
      - gat1.lin.weight: [heads*hidden_dim, in_features]
      - gat1.edge_lin.weight: [heads*hidden_dim, edge_dim]
    Also detects:
      - per-conv bias keys: 'gat1.bias' exists?
      - model bias scalar key: 'bias' exists?
    """
    required = ["fc.weight", "gat1.att", "gat1.lin.weight", "gat1.edge_lin.weight"]
    missing = [k for k in required if k not in sd]
    if missing:
        raise KeyError(f"Checkpoint missing required keys: {missing}\nAvailable keys (first 20): {list(sd.keys())[:20]}")

    fc_w = sd["fc.weight"]
    out_features = int(fc_w.shape[0])
    hidden_dim = int(fc_w.shape[1])

    att = sd["gat1.att"]
    heads = int(att.shape[0])
    # att second dim should be 3*hidden_dim
    if int(att.shape[1]) != 3 * hidden_dim:
        raise ValueError(f"gat1.att shape {tuple(att.shape)} inconsistent with hidden_dim={hidden_dim} (expected [H, {3*hidden_dim}])")

    lin_w = sd["gat1.lin.weight"]  # [heads*hidden_dim, in_features]
    if int(lin_w.shape[0]) != heads * hidden_dim:
        raise ValueError(f"gat1.lin.weight shape {tuple(lin_w.shape)} inconsistent with heads={heads}, hidden_dim={hidden_dim}")
    in_features = int(lin_w.shape[1])

    edge_w = sd["gat1.edge_lin.weight"]  # [heads*hidden_dim, edge_dim]
    edge_dim = int(edge_w.shape[1])

    conv_bias = ("gat1.bias" in sd)
    model_bias = ("bias" in sd) and (sd["bias"].numel() == 1)

    return Arch(
        in_features=in_features,
        hidden_dim=hidden_dim,
        out_features=out_features,
        heads=heads,
        edge_dim=edge_dim,
        conv_bias=conv_bias,
        model_bias=model_bias
    )


# ============================================================
# Helpers: blocks, IO
# ============================================================

def make_blocks(n_frames: int, block_size: int) -> List[Tuple[int, int]]:
    blocks = []
    for start in range(0, n_frames, block_size):
        end = min(n_frames, start + block_size)
        if end - start >= 2:
            blocks.append((start, end))
    return blocks

def save_csv(path: str, header: List[str], rows: List[List]):
    import csv
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)

def save_txt(path: str, lines: List[str]):
    with open(path, "w") as f:
        for ln in lines:
            f.write(ln.rstrip() + "\n")


# ============================================================
# Core evaluation
# ============================================================

def compute_error(model: EdgeGATDynamics, d0, d1, metric: str) -> Dict[str, float]:
    pred = model(d0.x, d0.edge_index, d0.edge_attr)
    true = d1.x[:, :3]
    out = {}
    if metric in ("rmsd", "both"):
        out["rmsd"] = float(kabsch_rmsd(pred, true).item())
    if metric in ("mse", "both"):
        out["mse"] = float(mse_loss(pred, true).item())
    return out

def get_edge_importance_from_alpha(model: EdgeGATDynamics) -> torch.Tensor:
    """
    Edge importance = sum over heads + sum over layers of post-softmax α.
    Returns [E].
    """
    a1 = model.gat1.get_last_alpha().sum(dim=1)  # [E]
    a2 = model.gat2.get_last_alpha().sum(dim=1)
    a3 = model.gat3.get_last_alpha().sum(dim=1)
    a4 = model.gat4.get_last_alpha().sum(dim=1)  # heads=1
    return a1 + a2 + a3 + a4

def topk_mask(edge_imp: torch.Tensor, k_frac: float) -> torch.Tensor:
    E = edge_imp.numel()
    k = max(1, int(round(k_frac * E)))
    _, idx = torch.topk(edge_imp, k=k, largest=True, sorted=False)
    m = torch.ones(E, device=edge_imp.device, dtype=edge_imp.dtype)
    m[idx] = 0.0
    return m

def random_mask(E: int, k_frac: float, device, dtype, rng: np.random.Generator) -> torch.Tensor:
    k = max(1, int(round(k_frac * E)))
    idx = rng.choice(E, size=k, replace=False)
    m = torch.ones(E, device=device, dtype=dtype)
    m[idx] = 0.0
    return m

def eval_block(model, data_list, start, end, k_frac, n_random, metric, seed) -> Dict[str, Dict[str, float]]:
    rng = np.random.default_rng(seed)
    metrics = ["rmsd", "mse"] if metric == "both" else [metric]

    base_vals = {m: [] for m in metrics}
    top_vals  = {m: [] for m in metrics}
    rand_vals = {m: [] for m in metrics}  # per-frame mean over random draws

    for t in range(start, end - 1):
        d0 = data_list[t]
        d1 = data_list[t + 1]

        # baseline
        model.clear_masks()
        e_base = compute_error(model, d0, d1, metric)
        for m in metrics:
            base_vals[m].append(e_base[m])

        # compute alpha (unmasked) for ranking
        model.clear_masks()
        _ = model(d0.x, d0.edge_index, d0.edge_attr)
        edge_imp = get_edge_importance_from_alpha(model)
        E = edge_imp.numel()

        # top-K mask
        m_top = topk_mask(edge_imp, k_frac)
        model.set_masks_all_layers(m_top)
        e_top = compute_error(model, d0, d1, metric)
        for m in metrics:
            top_vals[m].append(e_top[m])

        # random-K masks (average)
        rand_frame = {m: [] for m in metrics}
        for _ in range(n_random):
            m_r = random_mask(E, k_frac, edge_imp.device, edge_imp.dtype, rng)
            model.set_masks_all_layers(m_r)
            e_r = compute_error(model, d0, d1, metric)
            for m in metrics:
                rand_frame[m].append(e_r[m])

        for m in metrics:
            rand_vals[m].append(float(np.mean(rand_frame[m])))

        model.clear_masks()

    def mu_sd(a: List[float]) -> Tuple[float, float]:
        arr = np.asarray(a, dtype=float)
        return float(arr.mean()), float(arr.std(ddof=1) if arr.size > 1 else 0.0)

    out = {"base": {}, "top": {}, "rand_mu": {}, "rand_sd": {}}
    for m in metrics:
        out["base"][m] = mu_sd(base_vals[m])[0]
        out["top"][m]  = mu_sd(top_vals[m])[0]
        out["rand_mu"][m], out["rand_sd"][m] = mu_sd(rand_vals[m])
    return out


# ============================================================
# Main
# ============================================================

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base_dir", required=True)
    ap.add_argument("--system", required=True, choices=["WT", "NAM", "wt", "nam"])
    ap.add_argument("--rep_id", required=True, type=int)

    ap.add_argument("--analysis_root", default="ANALYSIS")
    ap.add_argument("--analysis_subdir", default="EDGE_FAITHFULNESS")
    ap.add_argument("--tag", default=None)

    ap.add_argument("--block_size", type=int, default=500)

    # scan up to 50% as you requested
    ap.add_argument("--k_min", type=int, default=1)
    ap.add_argument("--k_max", type=int, default=50)
    ap.add_argument("--k_step", type=int, default=1)

    ap.add_argument("--n_random", type=int, default=20)
    ap.add_argument("--metric", choices=["rmsd", "mse", "both"], default="both")
    ap.add_argument("--seed", type=int, default=123)
    ap.add_argument("--device", default=("cuda" if torch.cuda.is_available() else "cpu"))
    ap.add_argument("--make_plots", action="store_true")

    args = ap.parse_args()

    paths = build_paths(args.base_dir, args.system, args.rep_id,
                        analysis_root=args.analysis_root,
                        analysis_subdir=args.analysis_subdir,
                        tag=args.tag)

    if not os.path.exists(paths.data_file):
        raise FileNotFoundError(f"Trajectory file not found:\n{paths.data_file}")
    if not os.path.exists(paths.model_file):
        raise FileNotFoundError(f"Model file not found:\n{paths.model_file}")

    out_dir = paths.out_dir
    os.makedirs(out_dir, exist_ok=True)

    device = torch.device(args.device)

    # Load checkpoint (raw state_dict or wrapped)
    ckpt = torch.load(paths.model_file, map_location="cpu")
    state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    if not isinstance(state_dict, dict):
        raise ValueError("Loaded checkpoint is not a state_dict dict.")

    arch = infer_arch_from_state_dict(state_dict)

    # Build model matching checkpoint
    model = EdgeGATDynamics(
        in_features=arch.in_features,
        hidden_dim=arch.hidden_dim,
        out_features=arch.out_features,
        heads=arch.heads,
        edge_dim=arch.edge_dim,
        conv_bias=arch.conv_bias,
        model_bias=arch.model_bias
    ).to(device)

    # Load strictly: if mismatch now, we WANT to know
    model.load_state_dict(state_dict, strict=True)
    model.eval()

    # Save run config + inferred arch
    save_txt(os.path.join(out_dir, "run_config.txt"), [
        "Edge faithfulness (ROBUST: auto-infer arch from checkpoint)",
        json.dumps({
            **vars(args),
            "data_file": paths.data_file,
            "model_file": paths.model_file,
            "out_dir": out_dir,
            "inferred_arch": arch.__dict__
        }, indent=2)
    ])

    # Load trajectory list
    data_list = torch.load(paths.data_file, map_location="cpu", weights_only=False)
    if not isinstance(data_list, (list, tuple)) or len(data_list) < 3:
        raise ValueError("Trajectory .pt must contain a list of Data objects (len>=3).")

    n_frames = len(data_list)
    blocks = make_blocks(n_frames, args.block_size)
    if len(blocks) == 0:
        raise ValueError("No valid blocks; reduce block_size.")

    # Lazy device transfer to avoid huge GPU memory spikes
    def to_dev(d):
        d = d.to(device)
        d.x = d.x.float()
        d.edge_attr = d.edge_attr.float()
        d.edge_index = d.edge_index.long()
        return d

    class DevData:
        def __init__(self, cpu_list): self.cpu_list = cpu_list
        def __len__(self): return len(self.cpu_list)
        def __getitem__(self, i): return to_dev(self.cpu_list[i])

    dev = DevData(data_list)

    k_list = list(range(args.k_min, args.k_max + 1, args.k_step))
    k_fracs = [k / 100.0 for k in k_list]
    metrics = ["rmsd", "mse"] if args.metric == "both" else [args.metric]

    per_block_rows = []
    per_k_rows = []

    for k_pct, k_frac in zip(k_list, k_fracs):
        block_stats = []
        for bi, (bs, be) in enumerate(blocks):
            stats = eval_block(
                model=model,
                data_list=dev,
                start=bs,
                end=be,
                k_frac=k_frac,
                n_random=args.n_random,
                metric=args.metric,
                seed=args.seed + 1000 * bi + 17 * k_pct
            )
            block_stats.append(stats)

            for m in metrics:
                base = stats["base"][m]
                top  = stats["top"][m]
                rnd  = stats["rand_mu"][m]
                rndsd= stats["rand_sd"][m]
                per_block_rows.append([
                    k_pct, k_frac, bi, bs, be, m,
                    base, top, rnd, rndsd,
                    top - base, rnd - base, top - rnd
                ])

        # aggregate across blocks: mean ± SD (block-level)
        for m in metrics:
            base_b = np.array([b["base"][m] for b in block_stats], float)
            top_b  = np.array([b["top"][m] for b in block_stats], float)
            rnd_b  = np.array([b["rand_mu"][m] for b in block_stats], float)

            dtop = top_b - base_b
            drnd = rnd_b - base_b
            adv  = top_b - rnd_b

            def mu_sd(a):
                return float(a.mean()), float(a.std(ddof=1) if a.size > 1 else 0.0)

            base_mu, base_sd = mu_sd(base_b)
            top_mu,  top_sd  = mu_sd(top_b)
            rnd_mu,  rnd_sd  = mu_sd(rnd_b)
            dtop_mu, dtop_sd = mu_sd(dtop)
            drnd_mu, drnd_sd = mu_sd(drnd)
            adv_mu,  adv_sd  = mu_sd(adv)

            per_k_rows.append([
                k_pct, k_frac, m,
                base_mu, base_sd,
                top_mu, top_sd,
                rnd_mu, rnd_sd,
                dtop_mu, dtop_sd,
                drnd_mu, drnd_sd,
                adv_mu, adv_sd,
                len(blocks)
            ])

        print(f"[Done] {args.system.upper()} rep{args.rep_id}: K={k_pct}%")

    save_csv(os.path.join(out_dir, "edge_faithfulness_per_block.csv"),
             ["K_pct","K_frac","block_id","frame_start","frame_end","metric",
              "base_mean","top_mean","rand_mean","rand_sd_over_frames",
              "delta_top","delta_rand","adv_top_minus_rand"],
             per_block_rows)

    save_csv(os.path.join(out_dir, "edge_faithfulness_summary_by_K.csv"),
             ["K_pct","K_frac","metric",
              "base_mu","base_sd_blocks",
              "top_mu","top_sd_blocks",
              "rand_mu","rand_sd_blocks",
              "delta_top_mu","delta_top_sd_blocks",
              "delta_rand_mu","delta_rand_sd_blocks",
              "adv_mu","adv_sd_blocks",
              "n_blocks"],
             per_k_rows)

    save_txt(os.path.join(out_dir, "summary.txt"), [
        f"Edge faithfulness summary: {args.system.upper()} rep{args.rep_id}",
        f"Data file : {paths.data_file}",
        f"Model file: {paths.model_file}",
        f"Out dir   : {out_dir}",
        f"Inferred arch: {arch.__dict__}",
        f"Frames: {n_frames}",
        f"Block size: {args.block_size} frames; n_blocks={len(blocks)}",
        f"K scan: {args.k_min}% to {args.k_max}% step {args.k_step}%",
        f"Random draws per frame: {args.n_random}",
        f"Metrics: {args.metric}",
        "",
        "Interpretation:",
        "- adv_mu = mean_over_blocks(top - random). adv_mu > 0 => top-K masking hurts more than random.",
    ])

    if args.make_plots:
        try:
            import matplotlib.pyplot as plt
            for met in metrics:
                rows = [r for r in per_k_rows if r[2] == met]
                ks = [r[0] for r in rows]
                adv_mu = [r[13] for r in rows]
                adv_sd = [r[14] for r in rows]

                plt.figure()
                plt.errorbar(ks, adv_mu, yerr=adv_sd, fmt='-o')
                plt.xlabel("K (% edges masked)")
                plt.ylabel("Top - Random (block mean ± SD)")
                plt.title(f"{args.system.upper()} rep{args.rep_id}: edge faithfulness advantage ({met})")
                plt.grid(True)
                plt.tight_layout()
                plt.savefig(os.path.join(out_dir, f"edge_faithfulness_advantage_{met}.png"), dpi=200)
                plt.close()
        except Exception as e:
            save_txt(os.path.join(out_dir, "plot_error.txt"), [
                "Plotting failed, but CSV/TXT outputs were written.",
                repr(e)
            ])

    print(f"\nDONE. Outputs in:\n{out_dir}")


if __name__ == "__main__":
    main()

# ====== END SCRIPT ======

In [ ]:
!python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/EDGE_FAITHFULNESS/edge_faithfulness_coherent.py \
  --base_dir "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR" \
  --system WT \
  --rep_id 3 \
  --block_size 500 \
  --k_min 1 --k_max 50 --k_step 1 \
  --n_random 20 \
  --metric both \
  --make_plots \
  --device cuda

/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/EDGE_FAITHFULNESS/edge_faithfulness_coherent.py:89: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  D = torch.diag(torch.tensor([1.0, 1.0, d], device=P.device, dtype=P.dtype))
[Done] WT rep3: K=1%
[Done] WT rep3: K=2%
[Done] WT rep3: K=3%
[Done] WT rep3: K=4%
[Done] WT rep3: K=5%
[Done] WT rep3: K=6%
[Done] WT rep3: K=7%
[Done] WT rep3: K=8%
[Done] WT rep3: K=9%
[Done] WT rep3: K=10%
[Done] WT rep3: K=11%
[Done] WT rep3: K=12%
[Done] WT rep3: K=13%
[Done] WT rep3: K=14%
[Done] WT rep3: K=15%
[Done] WT rep3: K=16%
[Done] WT rep3: K=17%
[Done] WT rep3: K=18%
[Done] WT rep3: K=19%
[Done] WT rep3: K=20%
[Done] WT rep3: K=21%
[Done] WT rep3: K=22%
[Done] WT rep3: K=23%
[Done] WT rep3: K=24%
[Done] WT rep3: K=25%
[Done] WT rep3: K=26%
[Done] 

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/NODE_FAITHFULNESS", exist_ok=True)

In [ ]:
%%writefile /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/NODE_FAITHFULNESS/node_faithfulness_coherent.py
#!/usr/bin/env python3
"""
node_faithfulness_coherent.py  (ROBUST + COHERENT PATHS)

Node faithfulness analysis:
- Derive node importance from model attention α (post-softmax), aggregated across heads+layers.
- Mask top-K% nodes (and same number random nodes) by zeroing α on edges incident to masked nodes.
- Compare degradation: top-K vs random-K (faithfulness).
- Block means to reduce time correlation.
- Writes CSV/TXT under:
    {base_dir}/ANALYSIS/NODE_FAITHFULNESS/{SYSTEM}_rep{rep_id}/

Run (Colab):
!python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/NODE_FAITHFULNESS/node_faithfulness_coherent.py \
  --base_dir "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR" \
  --system WT --rep_id 3 \
  --block_size 500 \
  --k_min 1 --k_max 50 --k_step 1 \
  --n_random 20 \
  --metric both
"""

import os
import json
import argparse
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


# ============================================================
# Coherent path builder (system-folder layout)
# ============================================================

@dataclass
class RunPaths:
    data_file: str
    model_file: str
    out_dir: str

def build_paths(base_dir: str, system: str, rep_id: int,
                analysis_root: str = "ANALYSIS",
                analysis_subdir: str = "NODE_FAITHFULNESS",
                tag: Optional[str] = None) -> RunPaths:
    system = system.upper()
    sys_dir = os.path.join(base_dir, system)

    data_file = os.path.join(sys_dir, f"trajectory_graphs_phi_psi_rmsf_{system}_rep{rep_id}.pt")
    model_file = os.path.join(sys_dir, f"edgegat_dynamics_best_{system}_rep{rep_id}.pt")

    suffix = f"{system}_rep{rep_id}" if tag is None else f"{system}_rep{rep_id}_{tag}"
    out_dir = os.path.join(base_dir, analysis_root, analysis_subdir, suffix)
    os.makedirs(out_dir, exist_ok=True)

    return RunPaths(data_file=data_file, model_file=model_file, out_dir=out_dir)


# ============================================================
# Geometry utilities
# ============================================================

def kabsch_rmsd(P: torch.Tensor, Q: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    P_center = P - P.mean(dim=0, keepdim=True)
    Q_center = Q - Q.mean(dim=0, keepdim=True)

    C = P_center.T @ Q_center
    V, _, Wt = torch.linalg.svd(C)
    d = torch.sign(torch.det(V @ Wt))
    D = torch.diag(torch.tensor([1.0, 1.0, d], device=P.device, dtype=P.dtype))
    U = V @ D @ Wt

    P_rot = P_center @ U
    diff = P_rot - Q_center
    return torch.sqrt((diff * diff).sum() / (P.shape[0] + eps))

def mse_loss(P: torch.Tensor, Q: torch.Tensor) -> torch.Tensor:
    return torch.mean((P - Q) ** 2)


# ============================================================
# Robust model definition (matches your train-style checkpoint)
# ============================================================

class EdgeGATConv(MessagePassing):
    """
    Edge-aware attention (Meaning A): e_ij = a^T [x_i || x_j || e_ij], alpha = softmax(e_ij)

    Checkpoint-compatible:
      - att shape: [H, 3C]
      - optional per-layer bias key: gat*.bias

    Analysis additions:
      - caches last alpha (post-softmax): self._last_alpha [E,H]
      - supports soft alpha mask: alpha *= mask (no renorm)
    """
    def __init__(self, in_channels, out_channels, edge_dim, heads, concat=True,
                 dropout=0.0, conv_bias=True):
        super().__init__(aggr='add', node_dim=0)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        self.heads = heads
        self.concat = concat
        self.dropout = dropout

        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.edge_lin = nn.Linear(edge_dim, heads * out_channels, bias=False)

        self.att = nn.Parameter(torch.empty(heads, 3 * out_channels))  # [H, 3C]
        self.leaky_relu = nn.LeakyReLU(0.2)

        if conv_bias:
            out_dim = heads * out_channels if concat else out_channels
            self.bias = nn.Parameter(torch.zeros(out_dim))
        else:
            self.register_parameter('bias', None)

        self._last_alpha: Optional[torch.Tensor] = None  # [E,H]
        self._alpha_mask: Optional[torch.Tensor] = None  # [E] or [E,1] or [E,H]

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.edge_lin.weight)
        nn.init.xavier_uniform_(self.att)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def set_alpha_mask(self, mask: Optional[torch.Tensor]):
        self._alpha_mask = mask

    def clear_alpha_mask(self):
        self._alpha_mask = None

    def get_last_alpha(self) -> torch.Tensor:
        if self._last_alpha is None:
            raise RuntimeError("α not cached yet. Run a forward pass first.")
        return self._last_alpha

    def forward(self, x, edge_index, edge_attr):
        x = self.lin(x)                 # [N, H*C]
        edge_attr = self.edge_lin(edge_attr)  # [E, H*C]
        return self.propagate(edge_index=edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr, index):
        x_i = x_i.view(-1, self.heads, self.out_channels)       # [E,H,C]
        x_j = x_j.view(-1, self.heads, self.out_channels)
        edge_attr = edge_attr.view(-1, self.heads, self.out_channels)

        cat = torch.cat([x_i, x_j, edge_attr], dim=-1)          # [E,H,3C]
        e = torch.einsum("ehc,hc->eh", cat, self.att)           # [E,H]
        e = self.leaky_relu(e)
        alpha = softmax(e, index)                                # [E,H]
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        self._last_alpha = alpha.detach()

        if self._alpha_mask is not None:
            m = self._alpha_mask
            if m.ndim == 1:
                m = m.view(-1, 1)  # [E,1]
            alpha = alpha * m

        return x_j * alpha.unsqueeze(-1)                         # [E,H,C]

    def update(self, aggr_out):
        # aggr_out: [N,H,C]
        if self.concat:
            out = aggr_out.reshape(-1, self.heads * self.out_channels)   # [N,H*C]
        else:
            out = aggr_out.mean(dim=1)                                   # [N,C]
        if self.bias is not None:
            out = out + self.bias
        return out


class EdgeGATDynamics(nn.Module):
    """
    4-layer EdgeGAT + Linear head.
    Robust to:
      - optional model-level scalar bias key "bias"
    """
    def __init__(self, in_features, hidden_dim, out_features, heads, edge_dim,
                 conv_bias: bool, model_bias: bool):
        super().__init__()
        self.gat1 = EdgeGATConv(in_features, hidden_dim, edge_dim=edge_dim, heads=heads,
                                concat=True, conv_bias=conv_bias)
        self.gat2 = EdgeGATConv(hidden_dim * heads, hidden_dim, edge_dim=edge_dim, heads=heads,
                                concat=True, conv_bias=conv_bias)
        self.gat3 = EdgeGATConv(hidden_dim * heads, hidden_dim, edge_dim=edge_dim, heads=heads,
                                concat=True, conv_bias=conv_bias)
        self.gat4 = EdgeGATConv(hidden_dim * heads, hidden_dim, edge_dim=edge_dim, heads=1,
                                concat=False, conv_bias=conv_bias)
        self.fc = nn.Linear(hidden_dim, out_features)

        if model_bias:
            self.bias = nn.Parameter(torch.zeros(1))
        else:
            self.register_parameter("bias", None)

    def clear_masks(self):
        self.gat1.clear_alpha_mask()
        self.gat2.clear_alpha_mask()
        self.gat3.clear_alpha_mask()
        self.gat4.clear_alpha_mask()

    def set_masks_all_layers(self, mask_vec: torch.Tensor):
        self.gat1.set_alpha_mask(mask_vec)
        self.gat2.set_alpha_mask(mask_vec)
        self.gat3.set_alpha_mask(mask_vec)
        self.gat4.set_alpha_mask(mask_vec)

    def forward(self, x, edge_index, edge_attr):
        x = self.gat1(x, edge_index, edge_attr); x = F.relu(x)
        x = self.gat2(x, edge_index, edge_attr); x = F.relu(x)
        x = self.gat3(x, edge_index, edge_attr); x = F.relu(x)
        x = self.gat4(x, edge_index, edge_attr)
        out = self.fc(x)
        if self.bias is not None:
            out = out * self.bias
        return out


# ============================================================
# Auto-infer architecture from checkpoint
# ============================================================

@dataclass
class Arch:
    in_features: int
    hidden_dim: int
    out_features: int
    heads: int
    edge_dim: int
    conv_bias: bool
    model_bias: bool

def infer_arch_from_state_dict(sd: Dict[str, torch.Tensor]) -> Arch:
    required = ["fc.weight", "gat1.att", "gat1.lin.weight", "gat1.edge_lin.weight"]
    missing = [k for k in required if k not in sd]
    if missing:
        raise KeyError(f"Checkpoint missing required keys: {missing}\nAvailable keys (first 20): {list(sd.keys())[:20]}")

    fc_w = sd["fc.weight"]
    out_features = int(fc_w.shape[0])
    hidden_dim = int(fc_w.shape[1])

    att = sd["gat1.att"]
    heads = int(att.shape[0])
    if int(att.shape[1]) != 3 * hidden_dim:
        raise ValueError(f"gat1.att shape {tuple(att.shape)} inconsistent with hidden_dim={hidden_dim} (expected [H, {3*hidden_dim}])")

    lin_w = sd["gat1.lin.weight"]  # [heads*hidden_dim, in_features]
    if int(lin_w.shape[0]) != heads * hidden_dim:
        raise ValueError(f"gat1.lin.weight shape {tuple(lin_w.shape)} inconsistent with heads={heads}, hidden_dim={hidden_dim}")
    in_features = int(lin_w.shape[1])

    edge_w = sd["gat1.edge_lin.weight"]  # [heads*hidden_dim, edge_dim]
    edge_dim = int(edge_w.shape[1])

    conv_bias = ("gat1.bias" in sd)
    model_bias = ("bias" in sd) and (sd["bias"].numel() == 1)

    return Arch(
        in_features=in_features,
        hidden_dim=hidden_dim,
        out_features=out_features,
        heads=heads,
        edge_dim=edge_dim,
        conv_bias=conv_bias,
        model_bias=model_bias
    )


# ============================================================
# Helpers: blocks, IO
# ============================================================

def make_blocks(n_frames: int, block_size: int) -> List[Tuple[int, int]]:
    blocks = []
    for start in range(0, n_frames, block_size):
        end = min(n_frames, start + block_size)
        if end - start >= 2:
            blocks.append((start, end))
    return blocks

def save_csv(path: str, header: List[str], rows: List[List]):
    import csv
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)

def save_txt(path: str, lines: List[str]):
    with open(path, "w") as f:
        for ln in lines:
            f.write(ln.rstrip() + "\n")


# ============================================================
# Core evaluation
# ============================================================

def compute_error(model: EdgeGATDynamics, d0, d1, metric: str) -> Dict[str, float]:
    pred = model(d0.x, d0.edge_index, d0.edge_attr)
    true = d1.x[:, :3]
    out = {}
    if metric in ("rmsd", "both"):
        out["rmsd"] = float(kabsch_rmsd(pred, true).item())
    if metric in ("mse", "both"):
        out["mse"] = float(mse_loss(pred, true).item())
    return out

def get_edge_importance_from_alpha(model: EdgeGATDynamics) -> torch.Tensor:
    a1 = model.gat1.get_last_alpha().sum(dim=1)  # [E]
    a2 = model.gat2.get_last_alpha().sum(dim=1)
    a3 = model.gat3.get_last_alpha().sum(dim=1)
    a4 = model.gat4.get_last_alpha().sum(dim=1)  # heads=1
    return a1 + a2 + a3 + a4  # [E]

def node_importance_from_alpha(model: EdgeGATDynamics, edge_index: torch.Tensor, num_nodes: int) -> torch.Tensor:
    """
    Node importance I(v) = sum of incident edge attention mass (across layers+heads).
    """
    edge_imp = get_edge_importance_from_alpha(model)  # [E]
    src, dst = edge_index  # [E], [E]
    node_imp = torch.zeros(num_nodes, device=edge_imp.device, dtype=edge_imp.dtype)
    node_imp.index_add_(0, src, edge_imp)
    node_imp.index_add_(0, dst, edge_imp)
    return node_imp

def topk_node_mask(node_imp: torch.Tensor, k_frac: float) -> torch.Tensor:
    N = node_imp.numel()
    k = max(1, int(round(k_frac * N)))
    _, idx = torch.topk(node_imp, k=k, largest=True)
    m = torch.ones(N, device=node_imp.device, dtype=node_imp.dtype)
    m[idx] = 0.0
    return m

def random_node_mask(N: int, k_frac: float, device, dtype, rng: np.random.Generator) -> torch.Tensor:
    k = max(1, int(round(k_frac * N)))
    idx = rng.choice(N, size=k, replace=False)
    m = torch.ones(N, device=device, dtype=dtype)
    m[idx] = 0.0
    return m

def edge_mask_from_node_mask(node_mask: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
    src, dst = edge_index
    return node_mask[src] * node_mask[dst]  # [E]


def eval_block(model, data_list, start, end, k_frac, n_random, metric, seed) -> Dict[str, Dict[str, float]]:
    rng = np.random.default_rng(seed)
    metrics = ["rmsd","mse"] if metric == "both" else [metric]

    base_vals = {m: [] for m in metrics}
    top_vals  = {m: [] for m in metrics}
    rand_vals = {m: [] for m in metrics}

    for t in range(start, end - 1):
        d0 = data_list[t]
        d1 = data_list[t+1]

        # baseline
        model.clear_masks()
        e_base = compute_error(model, d0, d1, metric)
        for m in metrics:
            base_vals[m].append(e_base[m])

        # compute alpha (unmasked) for ranking
        model.clear_masks()
        _ = model(d0.x, d0.edge_index, d0.edge_attr)

        # node importance from alpha
        num_nodes = d0.x.shape[0]
        node_imp = node_importance_from_alpha(model, d0.edge_index, num_nodes=num_nodes)

        # top-K node mask -> edge mask -> apply
        node_m = topk_node_mask(node_imp, k_frac)
        edge_m = edge_mask_from_node_mask(node_m, d0.edge_index)

        model.set_masks_all_layers(edge_m)
        e_top = compute_error(model, d0, d1, metric)
        for m in metrics:
            top_vals[m].append(e_top[m])

        # random node masks
        rand_frame = {m: [] for m in metrics}
        for _ in range(n_random):
            node_mr = random_node_mask(num_nodes, k_frac, node_imp.device, node_imp.dtype, rng)
            edge_mr = edge_mask_from_node_mask(node_mr, d0.edge_index)
            model.set_masks_all_layers(edge_mr)
            e_r = compute_error(model, d0, d1, metric)
            for m in metrics:
                rand_frame[m].append(e_r[m])

        for m in metrics:
            rand_vals[m].append(float(np.mean(rand_frame[m])))

        model.clear_masks()

    def mu_sd(a: List[float]) -> Tuple[float, float]:
        arr = np.asarray(a, dtype=float)
        return float(arr.mean()), float(arr.std(ddof=1) if arr.size > 1 else 0.0)

    out = {"base": {}, "top": {}, "rand_mu": {}, "rand_sd": {}}
    for m in metrics:
        out["base"][m] = mu_sd(base_vals[m])[0]
        out["top"][m]  = mu_sd(top_vals[m])[0]
        out["rand_mu"][m], out["rand_sd"][m] = mu_sd(rand_vals[m])
    return out


# ============================================================
# Main
# ============================================================

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base_dir", required=True)
    ap.add_argument("--system", required=True, choices=["WT","NAM","wt","nam"])
    ap.add_argument("--rep_id", required=True, type=int)

    ap.add_argument("--analysis_root", default="ANALYSIS")
    ap.add_argument("--analysis_subdir", default="NODE_FAITHFULNESS")
    ap.add_argument("--tag", default=None)

    ap.add_argument("--block_size", type=int, default=500)

    ap.add_argument("--k_min", type=int, default=1)
    ap.add_argument("--k_max", type=int, default=50)
    ap.add_argument("--k_step", type=int, default=1)

    ap.add_argument("--n_random", type=int, default=20)
    ap.add_argument("--metric", choices=["rmsd","mse","both"], default="both")
    ap.add_argument("--seed", type=int, default=123)
    ap.add_argument("--device", default=("cuda" if torch.cuda.is_available() else "cpu"))
    ap.add_argument("--make_plots", action="store_true")

    args = ap.parse_args()

    paths = build_paths(args.base_dir, args.system, args.rep_id,
                        analysis_root=args.analysis_root,
                        analysis_subdir=args.analysis_subdir,
                        tag=args.tag)

    if not os.path.exists(paths.data_file):
        raise FileNotFoundError(f"Trajectory file not found:\n{paths.data_file}")
    if not os.path.exists(paths.model_file):
        raise FileNotFoundError(f"Model file not found:\n{paths.model_file}")

    out_dir = paths.out_dir
    os.makedirs(out_dir, exist_ok=True)

    device = torch.device(args.device)

    ckpt = torch.load(paths.model_file, map_location="cpu")
    state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    if not isinstance(state_dict, dict):
        raise ValueError("Loaded checkpoint is not a state_dict dict.")

    arch = infer_arch_from_state_dict(state_dict)

    model = EdgeGATDynamics(
        in_features=arch.in_features,
        hidden_dim=arch.hidden_dim,
        out_features=arch.out_features,
        heads=arch.heads,
        edge_dim=arch.edge_dim,
        conv_bias=arch.conv_bias,
        model_bias=arch.model_bias
    ).to(device)

    model.load_state_dict(state_dict, strict=True)
    model.eval()

    save_txt(os.path.join(out_dir, "run_config.txt"), [
        "Node faithfulness (ROBUST: auto-infer arch from checkpoint)",
        json.dumps({
            **vars(args),
            "data_file": paths.data_file,
            "model_file": paths.model_file,
            "out_dir": out_dir,
            "inferred_arch": arch.__dict__
        }, indent=2)
    ])

    data_list = torch.load(paths.data_file, map_location="cpu", weights_only=False)
    if not isinstance(data_list, (list, tuple)) or len(data_list) < 3:
        raise ValueError("Trajectory .pt must contain a list of Data objects (len>=3).")

    n_frames = len(data_list)
    blocks = make_blocks(n_frames, args.block_size)
    if len(blocks) == 0:
        raise ValueError("No valid blocks; reduce block_size.")

    def to_dev(d):
        d = d.to(device)
        d.x = d.x.float()
        d.edge_attr = d.edge_attr.float()
        d.edge_index = d.edge_index.long()
        return d

    class DevData:
        def __init__(self, cpu_list): self.cpu_list = cpu_list
        def __len__(self): return len(self.cpu_list)
        def __getitem__(self, i): return to_dev(self.cpu_list[i])

    dev = DevData(data_list)

    k_list = list(range(args.k_min, args.k_max + 1, args.k_step))
    k_fracs = [k / 100.0 for k in k_list]
    metrics = ["rmsd","mse"] if args.metric == "both" else [args.metric]

    per_block_rows = []
    per_k_rows = []

    for k_pct, k_frac in zip(k_list, k_fracs):
        block_stats = []
        for bi, (bs, be) in enumerate(blocks):
            stats = eval_block(
                model=model,
                data_list=dev,
                start=bs,
                end=be,
                k_frac=k_frac,
                n_random=args.n_random,
                metric=args.metric,
                seed=args.seed + 1000 * bi + 17 * k_pct
            )
            block_stats.append(stats)

            for m in metrics:
                base = stats["base"][m]
                top  = stats["top"][m]
                rnd  = stats["rand_mu"][m]
                rndsd= stats["rand_sd"][m]
                per_block_rows.append([
                    k_pct, k_frac, bi, bs, be, m,
                    base, top, rnd, rndsd,
                    top - base, rnd - base, top - rnd
                ])

        for m in metrics:
            base_b = np.array([b["base"][m] for b in block_stats], float)
            top_b  = np.array([b["top"][m] for b in block_stats], float)
            rnd_b  = np.array([b["rand_mu"][m] for b in block_stats], float)

            dtop = top_b - base_b
            drnd = rnd_b - base_b
            adv  = top_b - rnd_b

            def mu_sd(a):
                return float(a.mean()), float(a.std(ddof=1) if a.size > 1 else 0.0)

            base_mu, base_sd = mu_sd(base_b)
            top_mu,  top_sd  = mu_sd(top_b)
            rnd_mu,  rnd_sd  = mu_sd(rnd_b)
            dtop_mu, dtop_sd = mu_sd(dtop)
            drnd_mu, drnd_sd = mu_sd(drnd)
            adv_mu,  adv_sd  = mu_sd(adv)

            per_k_rows.append([
                k_pct, k_frac, m,
                base_mu, base_sd,
                top_mu, top_sd,
                rnd_mu, rnd_sd,
                dtop_mu, dtop_sd,
                drnd_mu, drnd_sd,
                adv_mu, adv_sd,
                len(blocks)
            ])

        print(f"[Done] {args.system.upper()} rep{args.rep_id}: K={k_pct}%")

    save_csv(os.path.join(out_dir, "node_faithfulness_per_block.csv"),
             ["K_pct","K_frac","block_id","frame_start","frame_end","metric",
              "base_mean","top_mean","rand_mean","rand_sd_over_frames",
              "delta_top","delta_rand","adv_top_minus_rand"],
             per_block_rows)

    save_csv(os.path.join(out_dir, "node_faithfulness_summary_by_K.csv"),
             ["K_pct","K_frac","metric",
              "base_mu","base_sd_blocks",
              "top_mu","top_sd_blocks",
              "rand_mu","rand_sd_blocks",
              "delta_top_mu","delta_top_sd_blocks",
              "delta_rand_mu","delta_rand_sd_blocks",
              "adv_mu","adv_sd_blocks",
              "n_blocks"],
             per_k_rows)

    save_txt(os.path.join(out_dir, "summary.txt"), [
        f"Node faithfulness summary: {args.system.upper()} rep{args.rep_id}",
        f"Data file : {paths.data_file}",
        f"Model file: {paths.model_file}",
        f"Out dir   : {out_dir}",
        f"Inferred arch: {arch.__dict__}",
        f"Frames: {n_frames}",
        f"Block size: {args.block_size} frames; n_blocks={len(blocks)}",
        f"K scan: {args.k_min}% to {args.k_max}% step {args.k_step}%",
        f"Random draws per frame: {args.n_random}",
        f"Metrics: {args.metric}",
        "",
        "Interpretation:",
        "- adv_mu = mean_over_blocks(top - random). adv_mu > 0 => top-K node masking hurts more than random.",
        "- Node masking is implemented by setting edge alpha mask = node_mask[src]*node_mask[dst].",
    ])

    if args.make_plots:
        try:
            import matplotlib.pyplot as plt
            for met in metrics:
                rows = [r for r in per_k_rows if r[2] == met]
                ks = [r[0] for r in rows]
                adv_mu = [r[13] for r in rows]
                adv_sd = [r[14] for r in rows]

                plt.figure()
                plt.errorbar(ks, adv_mu, yerr=adv_sd, fmt='-o')
                plt.xlabel("K (% nodes masked)")
                plt.ylabel("Top - Random (block mean ± SD)")
                plt.title(f"{args.system.upper()} rep{args.rep_id}: node faithfulness advantage ({met})")
                plt.grid(True)
                plt.tight_layout()
                plt.savefig(os.path.join(out_dir, f"node_faithfulness_advantage_{met}.png"), dpi=200)
                plt.close()
        except Exception as e:
            save_txt(os.path.join(out_dir, "plot_error.txt"), [
                "Plotting failed, but CSV/TXT outputs were written.",
                repr(e)
            ])

    print(f"\\nDONE. Outputs in:\\n{out_dir}")


if __name__ == "__main__":
    main()


Writing /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/NODE_FAITHFULNESS/node_faithfulness_coherent.py


In [ ]:
!python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/NODE_FAITHFULNESS/node_faithfulness_coherent.py \
  --base_dir "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR" \
  --system WT \
  --rep_id 3 \
  --block_size 500 \
  --k_min 1 --k_max 50 --k_step 1 \
  --n_random 20 \
  --metric both \
  --make_plots \
  --device cuda

/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/NODE_FAITHFULNESS/node_faithfulness_coherent.py:76: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  D = torch.diag(torch.tensor([1.0, 1.0, d], device=P.device, dtype=P.dtype))
[Done] WT rep3: K=1%
[Done] WT rep3: K=2%
[Done] WT rep3: K=3%
[Done] WT rep3: K=4%
[Done] WT rep3: K=5%
[Done] WT rep3: K=6%
[Done] WT rep3: K=7%
[Done] WT rep3: K=8%
[Done] WT rep3: K=9%
[Done] WT rep3: K=10%
[Done] WT rep3: K=11%
[Done] WT rep3: K=12%
[Done] WT rep3: K=13%
[Done] WT rep3: K=14%
[Done] WT rep3: K=15%
[Done] WT rep3: K=16%
[Done] WT rep3: K=17%
[Done] WT rep3: K=18%
[Done] WT rep3: K=19%
[Done] WT rep3: K=20%
[Done] WT rep3: K=21%
[Done] WT rep3: K=22%
[Done] WT rep3: K=23%
[Done] WT rep3: K=24%
[Done] WT rep3: K=25%
[Done] WT rep3: K=26%
[Done] 

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/NODE_WEIGHTS", exist_ok=True)


In [ ]:
%%writefile /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/NODE_WEIGHTS/node_attention_weights_coherent.py
#!/usr/bin/env python3
"""
node_attention_weights_coherent.py (Step 3, robust to variable edges per frame)

Extract continuous per-node attention weights (no thresholding).

Key point:
- If your graph edges change across frames (E varies), we can still compute node weights robustly
  because N is constant. We therefore DO NOT try to accumulate per-edge summaries here.

Definition (per frame t):
  edge_imp[e] = sum_{layers} sum_{heads} alpha_e(layer, head)
  node_w[v]   = sum_{incident edges e to v} edge_imp[e]   (both src and dst incidence)

We compute block means (default block_size=500 frames) and report:
- per-node mean across blocks
- per-node SD across blocks

Outputs:
  {base_dir}/ANALYSIS/NODE_WEIGHTS/{SYSTEM}_rep{rep_id}/
    - node_attention_weights_by_block.csv (optional)
    - node_attention_weights_summary.csv
    - summary.txt
    - run_config.txt
"""

import os, json, argparse
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


# ----------------------------
# Coherent path builder
# ----------------------------

@dataclass
class RunPaths:
    data_file: str
    model_file: str
    out_dir: str

def build_paths(base_dir: str, system: str, rep_id: int,
                analysis_root: str = "ANALYSIS",
                analysis_subdir: str = "NODE_WEIGHTS",
                tag: Optional[str] = None) -> RunPaths:
    system = system.upper()
    sys_dir = os.path.join(base_dir, system)

    data_file = os.path.join(sys_dir, f"trajectory_graphs_phi_psi_rmsf_{system}_rep{rep_id}.pt")
    model_file = os.path.join(sys_dir, f"edgegat_dynamics_best_{system}_rep{rep_id}.pt")

    suffix = f"{system}_rep{rep_id}" if tag is None else f"{system}_rep{rep_id}_{tag}"
    out_dir = os.path.join(base_dir, analysis_root, analysis_subdir, suffix)
    os.makedirs(out_dir, exist_ok=True)

    return RunPaths(data_file=data_file, model_file=model_file, out_dir=out_dir)


# ----------------------------
# Model (checkpoint-compatible)
# ----------------------------

class EdgeGATConv(MessagePassing):
    def __init__(self, in_channels, out_channels, edge_dim, heads, concat=True,
                 dropout=0.0, conv_bias=True):
        super().__init__(aggr='add', node_dim=0)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        self.heads = heads
        self.concat = concat
        self.dropout = dropout

        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.edge_lin = nn.Linear(edge_dim, heads * out_channels, bias=False)
        self.att = nn.Parameter(torch.empty(heads, 3 * out_channels))  # [H,3C]
        self.leaky_relu = nn.LeakyReLU(0.2)

        if conv_bias:
            out_dim = heads * out_channels if concat else out_channels
            self.bias = nn.Parameter(torch.zeros(out_dim))
        else:
            self.register_parameter("bias", None)

        self._last_alpha: Optional[torch.Tensor] = None
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.edge_lin.weight)
        nn.init.xavier_uniform_(self.att)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def get_last_alpha(self) -> torch.Tensor:
        if self._last_alpha is None:
            raise RuntimeError("α not cached yet. Run a forward pass first.")
        return self._last_alpha

    def forward(self, x, edge_index, edge_attr):
        x = self.lin(x)
        edge_attr = self.edge_lin(edge_attr)
        return self.propagate(edge_index=edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr, index):
        x_i = x_i.view(-1, self.heads, self.out_channels)
        x_j = x_j.view(-1, self.heads, self.out_channels)
        edge_attr = edge_attr.view(-1, self.heads, self.out_channels)

        cat = torch.cat([x_i, x_j, edge_attr], dim=-1)          # [E,H,3C]
        e = torch.einsum("ehc,hc->eh", cat, self.att)           # [E,H]
        e = self.leaky_relu(e)
        alpha = softmax(e, index)                                # [E,H]
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        self._last_alpha = alpha.detach()
        return x_j * alpha.unsqueeze(-1)

    def update(self, aggr_out):
        if self.concat:
            out = aggr_out.reshape(-1, self.heads * self.out_channels)
        else:
            out = aggr_out.mean(dim=1)
        if self.bias is not None:
            out = out + self.bias
        return out


class EdgeGATDynamics(nn.Module):
    def __init__(self, in_features, hidden_dim, out_features, heads, edge_dim,
                 conv_bias: bool, model_bias: bool):
        super().__init__()
        self.gat1 = EdgeGATConv(in_features, hidden_dim, edge_dim=edge_dim, heads=heads, concat=True, conv_bias=conv_bias)
        self.gat2 = EdgeGATConv(hidden_dim * heads, hidden_dim, edge_dim=edge_dim, heads=heads, concat=True, conv_bias=conv_bias)
        self.gat3 = EdgeGATConv(hidden_dim * heads, hidden_dim, edge_dim=edge_dim, heads=heads, concat=True, conv_bias=conv_bias)
        self.gat4 = EdgeGATConv(hidden_dim * heads, hidden_dim, edge_dim=edge_dim, heads=1, concat=False, conv_bias=conv_bias)
        self.fc = nn.Linear(hidden_dim, out_features)

        if model_bias:
            self.bias = nn.Parameter(torch.zeros(1))
        else:
            self.register_parameter("bias", None)

    def forward(self, x, edge_index, edge_attr):
        x = self.gat1(x, edge_index, edge_attr); x = F.relu(x)
        x = self.gat2(x, edge_index, edge_attr); x = F.relu(x)
        x = self.gat3(x, edge_index, edge_attr); x = F.relu(x)
        x = self.gat4(x, edge_index, edge_attr)
        out = self.fc(x)
        if self.bias is not None:
            out = out * self.bias
        return out


# ----------------------------
# Auto-infer architecture
# ----------------------------

@dataclass
class Arch:
    in_features: int
    hidden_dim: int
    out_features: int
    heads: int
    edge_dim: int
    conv_bias: bool
    model_bias: bool

def infer_arch_from_state_dict(sd: Dict[str, torch.Tensor]) -> Arch:
    required = ["fc.weight", "gat1.att", "gat1.lin.weight", "gat1.edge_lin.weight"]
    missing = [k for k in required if k not in sd]
    if missing:
        raise KeyError(f"Checkpoint missing keys: {missing}")

    fc_w = sd["fc.weight"]
    out_features = int(fc_w.shape[0])
    hidden_dim = int(fc_w.shape[1])

    att = sd["gat1.att"]
    heads = int(att.shape[0])
    if int(att.shape[1]) != 3 * hidden_dim:
        raise ValueError(f"gat1.att shape {tuple(att.shape)} inconsistent with hidden_dim={hidden_dim}")

    lin_w = sd["gat1.lin.weight"]
    in_features = int(lin_w.shape[1])

    edge_w = sd["gat1.edge_lin.weight"]
    edge_dim = int(edge_w.shape[1])

    conv_bias = ("gat1.bias" in sd)
    model_bias = ("bias" in sd) and (sd["bias"].numel() == 1)

    return Arch(in_features, hidden_dim, out_features, heads, edge_dim, conv_bias, model_bias)


# ----------------------------
# Helpers
# ----------------------------

def make_blocks(n_frames: int, block_size: int) -> List[Tuple[int, int]]:
    blocks = []
    for start in range(0, n_frames, block_size):
        end = min(n_frames, start + block_size)
        if end - start >= 2:
            blocks.append((start, end))
    return blocks

def save_txt(path: str, lines: List[str]):
    with open(path, "w") as f:
        for ln in lines:
            f.write(ln.rstrip() + "\n")

def save_csv(path: str, header: List[str], rows: List[List]):
    import csv
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)

def edge_importance_from_alpha(model: EdgeGATDynamics) -> torch.Tensor:
    a1 = model.gat1.get_last_alpha().sum(dim=1)
    a2 = model.gat2.get_last_alpha().sum(dim=1)
    a3 = model.gat3.get_last_alpha().sum(dim=1)
    a4 = model.gat4.get_last_alpha().sum(dim=1)
    return a1 + a2 + a3 + a4  # [E]

def node_weights_from_edge_imp(edge_imp: torch.Tensor, edge_index: torch.Tensor, num_nodes: int) -> torch.Tensor:
    src, dst = edge_index
    node_w = torch.zeros(num_nodes, device=edge_imp.device, dtype=edge_imp.dtype)
    node_w.index_add_(0, src, edge_imp)
    node_w.index_add_(0, dst, edge_imp)
    return node_w


# ----------------------------
# Main
# ----------------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base_dir", required=True)
    ap.add_argument("--system", required=True, choices=["WT","NAM","wt","nam"])
    ap.add_argument("--rep_id", required=True, type=int)

    ap.add_argument("--analysis_root", default="ANALYSIS")
    ap.add_argument("--analysis_subdir", default="NODE_WEIGHTS")
    ap.add_argument("--tag", default=None)

    ap.add_argument("--block_size", type=int, default=500)
    ap.add_argument("--device", default=("cuda" if torch.cuda.is_available() else "cpu"))

    ap.add_argument("--write_block_csv", action="store_true",
                    help="Write node_attention_weights_by_block.csv (large but useful).")

    args = ap.parse_args()

    paths = build_paths(args.base_dir, args.system, args.rep_id,
                        analysis_root=args.analysis_root,
                        analysis_subdir=args.analysis_subdir,
                        tag=args.tag)

    if not os.path.exists(paths.data_file):
        raise FileNotFoundError(f"Trajectory file not found:\n{paths.data_file}")
    if not os.path.exists(paths.model_file):
        raise FileNotFoundError(f"Model file not found:\n{paths.model_file}")

    out_dir = paths.out_dir
    os.makedirs(out_dir, exist_ok=True)

    device = torch.device(args.device)

    ckpt = torch.load(paths.model_file, map_location="cpu")
    state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    if not isinstance(state_dict, dict):
        raise ValueError("Loaded checkpoint is not a state_dict dict.")

    arch = infer_arch_from_state_dict(state_dict)

    model = EdgeGATDynamics(
        in_features=arch.in_features,
        hidden_dim=arch.hidden_dim,
        out_features=arch.out_features,
        heads=arch.heads,
        edge_dim=arch.edge_dim,
        conv_bias=arch.conv_bias,
        model_bias=arch.model_bias
    ).to(device)
    model.load_state_dict(state_dict, strict=True)
    model.eval()

    data_list = torch.load(paths.data_file, map_location="cpu", weights_only=False)
    if not isinstance(data_list, (list, tuple)) or len(data_list) < 2:
        raise ValueError("Trajectory .pt must contain a list of Data objects (len>=2).")

    n_frames = len(data_list)
    blocks = make_blocks(n_frames, args.block_size)
    if len(blocks) == 0:
        raise ValueError("No valid blocks; reduce block_size.")

    def to_dev(d):
        d = d.to(device)
        d.x = d.x.float()
        d.edge_attr = d.edge_attr.float()
        d.edge_index = d.edge_index.long()
        return d

    # N is assumed constant
    d0 = to_dev(data_list[0])
    num_nodes = int(d0.x.shape[0])

    block_node_means = []   # list of [N] arrays
    block_rows = []

    for bi, (bs, be) in enumerate(blocks):
        node_sum = torch.zeros(num_nodes, device=device)
        count = 0

        for t in range(bs, be):
            dt = to_dev(data_list[t])

            # sanity: ensure N constant
            if dt.x.shape[0] != num_nodes:
                raise RuntimeError(f"Num nodes changed at frame {t}: {dt.x.shape[0]} vs {num_nodes}")

            _ = model(dt.x, dt.edge_index, dt.edge_attr)  # caches alpha

            edge_imp = edge_importance_from_alpha(model)  # [E_t] varies per frame is OK
            node_w = node_weights_from_edge_imp(edge_imp, dt.edge_index, num_nodes)  # [N]

            node_sum += node_w
            count += 1

        node_mean = (node_sum / max(count, 1)).detach().cpu().numpy()
        block_node_means.append(node_mean)

        if args.write_block_csv:
            for v in range(num_nodes):
                block_rows.append([bi, bs, be, v, float(node_mean[v])])

        print(f"[Done] Block {bi+1}/{len(blocks)}: frames {bs}-{be-1} (count={count})")

    BN = np.stack(block_node_means, axis=0)  # [B,N]
    node_mu = BN.mean(axis=0)
    node_sd = BN.std(axis=0, ddof=1) if BN.shape[0] > 1 else np.zeros_like(node_mu)

    # write summary
    node_rows = [[v, float(node_mu[v]), float(node_sd[v]), int(len(blocks))] for v in range(num_nodes)]
    save_csv(os.path.join(out_dir, "node_attention_weights_summary.csv"),
             ["node_id","weight_mean","weight_sd_blocks","n_blocks"],
             node_rows)

    if args.write_block_csv:
        save_csv(os.path.join(out_dir, "node_attention_weights_by_block.csv"),
                 ["block_id","frame_start","frame_end","node_id","node_weight_block_mean"],
                 block_rows)

    save_txt(os.path.join(out_dir, "run_config.txt"), [
        "Node attention weights (Step 3) - robust to variable E per frame",
        json.dumps({
            **vars(args),
            "data_file": paths.data_file,
            "model_file": paths.model_file,
            "out_dir": out_dir,
            "inferred_arch": arch.__dict__,
            "n_frames": n_frames,
            "n_blocks": len(blocks),
            "num_nodes": num_nodes,
            "note": "Edge count may vary per frame; edge summaries are not computed in this step."
        }, indent=2)
    ])

    save_txt(os.path.join(out_dir, "summary.txt"), [
        f"Node weights extraction (Step 3): {args.system.upper()} rep{args.rep_id}",
        f"Data file : {paths.data_file}",
        f"Model file: {paths.model_file}",
        f"Out dir   : {out_dir}",
        f"Inferred arch: {arch.__dict__}",
        f"Frames: {n_frames}",
        f"Block size: {args.block_size}; n_blocks={len(blocks)}",
        f"Num nodes: {num_nodes}",
        "",
        "Definition:",
        "- edge_imp[e] = sum_layers sum_heads alpha_e (per frame; E can vary)",
        "- node_w[v]   = sum_{incident edges} edge_imp[e] (both src and dst)",
    ])

    print(f"\nDONE. Outputs in:\n{out_dir}")


if __name__ == "__main__":
    main()


Writing /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/NODE_WEIGHTS/node_attention_weights_coherent.py


In [ ]:
!python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/NODE_WEIGHTS/node_attention_weights_coherent.py \
  --base_dir "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR" \
  --system NAM \
  --rep_id 3 \
  --block_size 500 \
  --device cuda \
  --write_block_csv

[Done] Block 1/8: frames 0-499 (count=500)
[Done] Block 2/8: frames 500-999 (count=500)
[Done] Block 3/8: frames 1000-1499 (count=500)
[Done] Block 4/8: frames 1500-1999 (count=500)
[Done] Block 5/8: frames 2000-2499 (count=500)
[Done] Block 6/8: frames 2500-2999 (count=500)
[Done] Block 7/8: frames 3000-3499 (count=500)
[Done] Block 8/8: frames 3500-3999 (count=500)

DONE. Outputs in:
/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/NODE_WEIGHTS/NAM_rep3


In [ ]:
import os
os.makedirs("/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE", exist_ok=True)

In [ ]:
%%writefile /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/descriptor_by_block_all_sum.py

#!/usr/bin/env python3
import os, json, argparse
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx

from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


# ============================================================
# Paths
# ============================================================

@dataclass
class RunPaths:
    data_file: str
    model_file: str
    out_dir: str


def build_paths(base_dir, system, rep_id, analysis_root, out_subdir, variant):
    system = system.upper()
    data_file = os.path.join(
        base_dir, system,
        f"trajectory_graphs_phi_psi_rmsf_{system}_rep{rep_id}.pt"
    )
    model_file = os.path.join(
        base_dir, system,
        f"edgegat_dynamics_best_{system}_rep{rep_id}.pt"
    )
    out_dir = os.path.join(
        base_dir, analysis_root, out_subdir, variant, "REPLICA_BLOCK", f"{system}_rep{rep_id}"
    )
    os.makedirs(out_dir, exist_ok=True)
    return RunPaths(data_file, model_file, out_dir)


# ============================================================
# Model definition
# ============================================================

class EdgeGATConv(MessagePassing):
    def __init__(self, in_channels, out_channels, edge_dim, heads,
                 concat=True, dropout=0.0, conv_bias=True):
        super().__init__(aggr="add", node_dim=0)
        self.out_channels = out_channels
        self.heads = heads
        self.concat = concat
        self.dropout = dropout

        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.edge_lin = nn.Linear(edge_dim, heads * out_channels, bias=False)
        self.att = nn.Parameter(torch.empty(heads, 3 * out_channels))
        self.leaky_relu = nn.LeakyReLU(0.2)

        if conv_bias:
            out_dim = heads * out_channels if concat else out_channels
            self.bias = nn.Parameter(torch.zeros(out_dim))
        else:
            self.register_parameter("bias", None)

        self._last_alpha = None
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.edge_lin.weight)
        nn.init.xavier_uniform_(self.att)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def get_last_alpha(self):
        if self._last_alpha is None:
            raise RuntimeError("Alpha not cached. Run forward first.")
        return self._last_alpha

    def forward(self, x, edge_index, edge_attr):
        x = self.lin(x)
        edge_attr = self.edge_lin(edge_attr)
        return self.propagate(edge_index=edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr, index):
        x_i = x_i.view(-1, self.heads, self.out_channels)
        x_j = x_j.view(-1, self.heads, self.out_channels)
        edge_attr = edge_attr.view(-1, self.heads, self.out_channels)

        cat = torch.cat([x_i, x_j, edge_attr], dim=-1)
        e = torch.einsum("ehc,hc->eh", cat, self.att)
        e = self.leaky_relu(e)

        alpha = softmax(e, index)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        self._last_alpha = alpha.detach()
        return x_j * alpha.unsqueeze(-1)

    def update(self, aggr_out):
        if self.concat:
            out = aggr_out.reshape(-1, self.heads * self.out_channels)
        else:
            out = aggr_out.mean(dim=1)

        if self.bias is not None:
            out = out + self.bias
        return out


class EdgeGATDynamics(nn.Module):
    def __init__(self, in_features, hidden_dim, out_features, heads, edge_dim,
                 conv_bias=True, model_bias=True):
        super().__init__()

        self.gat1 = EdgeGATConv(in_features, hidden_dim, edge_dim, heads, concat=True, conv_bias=conv_bias)
        self.gat2 = EdgeGATConv(hidden_dim * heads, hidden_dim, edge_dim, heads, concat=True, conv_bias=conv_bias)
        self.gat3 = EdgeGATConv(hidden_dim * heads, hidden_dim, edge_dim, heads, concat=True, conv_bias=conv_bias)
        self.gat4 = EdgeGATConv(hidden_dim * heads, hidden_dim, edge_dim, 1, concat=False, conv_bias=conv_bias)

        self.fc = nn.Linear(hidden_dim, out_features)

        if model_bias:
            self.bias = nn.Parameter(torch.zeros(1))
        else:
            self.register_parameter("bias", None)

    def forward(self, x, edge_index, edge_attr):
        x = self.gat1(x, edge_index, edge_attr); x = F.relu(x)
        x = self.gat2(x, edge_index, edge_attr); x = F.relu(x)
        x = self.gat3(x, edge_index, edge_attr); x = F.relu(x)
        x = self.gat4(x, edge_index, edge_attr)
        out = self.fc(x)
        if self.bias is not None:
            out = out * self.bias
        return out


@dataclass
class Arch:
    in_features: int
    hidden_dim: int
    out_features: int
    heads: int
    edge_dim: int
    conv_bias: bool
    model_bias: bool


def infer_arch(sd):
    fc_w = sd["fc.weight"]
    out_features = int(fc_w.shape[0])
    hidden_dim = int(fc_w.shape[1])

    heads = int(sd["gat1.att"].shape[0])
    in_features = int(sd["gat1.lin.weight"].shape[1])
    edge_dim = int(sd["gat1.edge_lin.weight"].shape[1])

    conv_bias = "gat1.bias" in sd
    model_bias = ("bias" in sd) and (sd["bias"].numel() == 1)

    return Arch(in_features, hidden_dim, out_features, heads, edge_dim, conv_bias, model_bias)


# ============================================================
# Descriptor utilities
# ============================================================

def make_blocks(n_frames, block_size):
    blocks = []
    for start in range(0, n_frames, block_size):
        end = min(n_frames, start + block_size)
        if end - start >= 2:
            blocks.append((start, end))
    return blocks


def to_device(d, device):
    d = d.to(device)
    d.x = d.x.float()
    d.edge_attr = d.edge_attr.float()
    d.edge_index = d.edge_index.long()
    return d


def all_sum_attention(model):
    """
    all_sum = sum over all attention channels:
      gat1 heads + gat2 heads + gat3 heads + gat4 single head
    """
    a1 = model.gat1.get_last_alpha().sum(dim=1)
    a2 = model.gat2.get_last_alpha().sum(dim=1)
    a3 = model.gat3.get_last_alpha().sum(dim=1)
    a4 = model.gat4.get_last_alpha().sum(dim=1)
    return a1 + a2 + a3 + a4


def undirected_key(u, v):
    return (u, v) if u <= v else (v, u)


def build_graph(num_nodes, edge_mean, eps):
    G = nx.Graph()
    G.add_nodes_from(range(num_nodes))
    for (u, v), w in edge_mean.items():
        if w <= 0:
            continue
        G.add_edge(u, v, weight=float(w), distance=float(1.0 / (w + eps)))
    return G


def safe_eigenvector(G):
    try:
        return nx.eigenvector_centrality(G, weight="weight", max_iter=2000)
    except Exception:
        return {n: 0.0 for n in G.nodes()}


def compute_descriptors(G, num_nodes):
    # strength = attention-weighted degree
    strength = np.zeros(num_nodes, dtype=float)
    for n in G.nodes():
        strength[n] = sum(attr["weight"] for _, _, attr in G.edges(n, data=True))

    betweenness = nx.betweenness_centrality(G, weight="distance", normalized=True)
    closeness = nx.closeness_centrality(G, distance="distance")
    pagerank = nx.pagerank(G, weight="weight")
    eigenvector = safe_eigenvector(G)

    return {
        "strength": np.array([strength[i] for i in range(num_nodes)], dtype=float),
        "betweenness": np.array([betweenness.get(i, 0.0) for i in range(num_nodes)], dtype=float),
        "closeness": np.array([closeness.get(i, 0.0) for i in range(num_nodes)], dtype=float),
        "pagerank": np.array([pagerank.get(i, 0.0) for i in range(num_nodes)], dtype=float),
        "eigenvector": np.array([eigenvector.get(i, 0.0) for i in range(num_nodes)], dtype=float),
    }


def descriptors_for_block(model, data_cpu, bs, be, device, eps):
    num_nodes = int(data_cpu[bs].x.shape[0])

    edge_sum = {}
    edge_count = {}

    with torch.no_grad():
        for t in range(bs, be):
            d = to_device(data_cpu[t], device)
            _ = model(d.x, d.edge_index, d.edge_attr)

            edge_imp = all_sum_attention(model)

            src = d.edge_index[0]
            dst = d.edge_index[1]

            for e in range(edge_imp.numel()):
                u = int(src[e].item())
                v = int(dst[e].item())
                key = undirected_key(u, v)
                val = float(edge_imp[e].item())

                edge_sum[key] = edge_sum.get(key, 0.0) + val
                edge_count[key] = edge_count.get(key, 0) + 1

    edge_mean = {
        key: edge_sum[key] / max(edge_count[key], 1)
        for key in edge_sum.keys()
    }

    G = build_graph(num_nodes, edge_mean, eps)
    desc = compute_descriptors(G, num_nodes)

    return desc, len(edge_mean)


# ============================================================
# Main
# ============================================================

def main():
    ap = argparse.ArgumentParser()

    ap.add_argument("--base_dir", required=True)
    ap.add_argument("--system", required=True, choices=["WT", "NAM", "wt", "nam"])
    ap.add_argument("--rep_id", required=True, type=int)

    ap.add_argument("--analysis_root", default="ANALYSIS")
    ap.add_argument("--out_subdir", default="DESCRIPTOR_PIPELINE")
    ap.add_argument("--variant", default="all_sum")

    ap.add_argument("--block_size", type=int, default=500)
    ap.add_argument("--eps", type=float, default=1e-8)
    ap.add_argument("--device", default=("cuda" if torch.cuda.is_available() else "cpu"))

    args = ap.parse_args()

    system = args.system.upper()
    paths = build_paths(
        args.base_dir,
        system,
        args.rep_id,
        args.analysis_root,
        args.out_subdir,
        args.variant,
    )

    if not os.path.exists(paths.data_file):
        raise FileNotFoundError(paths.data_file)
    if not os.path.exists(paths.model_file):
        raise FileNotFoundError(paths.model_file)

    device = torch.device(args.device)

    ckpt = torch.load(paths.model_file, map_location="cpu")
    sd = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt

    arch = infer_arch(sd)

    model = EdgeGATDynamics(
        in_features=arch.in_features,
        hidden_dim=arch.hidden_dim,
        out_features=arch.out_features,
        heads=arch.heads,
        edge_dim=arch.edge_dim,
        conv_bias=arch.conv_bias,
        model_bias=arch.model_bias,
    ).to(device)

    model.load_state_dict(sd, strict=True)
    model.eval()

    data_cpu = torch.load(paths.data_file, map_location="cpu", weights_only=False)
    blocks = make_blocks(len(data_cpu), args.block_size)

    if len(blocks) == 0:
        raise RuntimeError("No valid blocks. Reduce block_size.")

    rows = []

    for bi, (bs, be) in enumerate(blocks):
        desc, n_edges = descriptors_for_block(
            model=model,
            data_cpu=data_cpu,
            bs=bs,
            be=be,
            device=device,
            eps=args.eps,
        )

        num_nodes = len(desc["strength"])

        for node_id in range(num_nodes):
            rows.append({
                "block_id": bi,
                "frame_start": bs,
                "frame_end": be,
                "node_id": node_id,
                "residue": node_id + 1,
                "strength": float(desc["strength"][node_id]),
                "betweenness": float(desc["betweenness"][node_id]),
                "closeness": float(desc["closeness"][node_id]),
                "pagerank": float(desc["pagerank"][node_id]),
                "eigenvector": float(desc["eigenvector"][node_id]),
                "n_edges_block": int(n_edges),
            })

        print(f"[Done] {system} rep{args.rep_id} block {bi+1}/{len(blocks)} frames {bs}-{be-1}")

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(paths.out_dir, "descriptors_by_block.csv"), index=False)

    # simple replicate summary across blocks
    summary_rows = []
    for node_id, g in df.groupby("node_id"):
        row = {
            "node_id": int(node_id),
            "residue": int(node_id) + 1,
        }
        for met in ["strength", "betweenness", "closeness", "pagerank", "eigenvector"]:
            vals = g[met].values.astype(float)
            row[f"{met}_mean"] = float(vals.mean())
            row[f"{met}_sd_blocks"] = float(vals.std(ddof=1) if len(vals) > 1 else 0.0)
            row[f"{met}_n_blocks"] = int(len(vals))
        summary_rows.append(row)

    pd.DataFrame(summary_rows).to_csv(
        os.path.join(paths.out_dir, "descriptors_summary.csv"),
        index=False,
    )

    cfg = {
        "system": system,
        "rep_id": args.rep_id,
        "variant": args.variant,
        "block_size": args.block_size,
        "eps": args.eps,
        "data_file": paths.data_file,
        "model_file": paths.model_file,
        "out_dir": paths.out_dir,
        "arch": arch.__dict__,
        "descriptor_definitions": {
            "strength": "sum of all_sum attention-weighted edges incident on node",
            "betweenness": "weighted shortest-path betweenness using distance=1/(attention+eps)",
            "closeness": "weighted closeness using distance=1/(attention+eps)",
            "pagerank": "weighted PageRank using attention as edge weight",
            "eigenvector": "weighted eigenvector centrality using attention as edge weight",
        },
    }

    with open(os.path.join(paths.out_dir, "run_config.txt"), "w") as f:
        f.write(json.dumps(cfg, indent=2))

    print("\nDONE.")
    print("Output:", paths.out_dir)


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/descriptor_by_block_all_sum.py


In [ ]:
import os

base = "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR"
script = f"{base}/ANALYSIS/DESCRIPTOR_PIPELINE/descriptor_by_block_all_sum.py"

for SYS in ["WT", "NAM"]:
    for REP in [1, 2, 3]:
        cmd = f"""
        python {script} \
          --base_dir "{base}" \
          --system {SYS} \
          --rep_id {REP} \
          --block_size 500 \
          --device cuda
        """
        print(cmd)
        os.system(cmd)


        python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/descriptor_by_block_all_sum.py           --base_dir "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR"           --system WT           --rep_id 1           --block_size 500           --device cuda
        

        python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/descriptor_by_block_all_sum.py           --base_dir "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR"           --system WT           --rep_id 2           --block_size 500           --device cuda
        

        python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/descriptor_by_block_all_sum.py           --base_dir "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR"           --system WT           --rep_id 3           --block_size 500           --device cuda
        

        python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSI

In [ ]:
%%writefile /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/aggregate_descriptor_pipeline.py

#!/usr/bin/env python3

import os
import json
import argparse
import numpy as np
import pandas as pd

METRICS = ["strength", "betweenness", "closeness", "pagerank", "eigenvector"]


# ============================================================
# Load replica block data
# ============================================================

def load_replica(csv_path):
    df = pd.read_csv(csv_path)

    needed = ["block_id", "node_id"] + METRICS
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} missing columns: {missing}")

    blocks = np.sort(df["block_id"].unique())
    nodes = np.sort(df["node_id"].unique())

    B, N, M = len(blocks), len(nodes), len(METRICS)
    X = np.full((B, N, M), np.nan)

    bmap = {b: i for i, b in enumerate(blocks)}
    nmap = {n: i for i, n in enumerate(nodes)}

    for _, row in df.iterrows():
        bi = bmap[row["block_id"]]
        ni = nmap[row["node_id"]]
        X[bi, ni, :] = row[METRICS].values

    if np.isnan(X).any():
        raise RuntimeError(f"{csv_path} contains NaNs")

    return nodes, X


# ============================================================
# Bootstrap
# ============================================================

def system_boot_mean(Xr, rng):
    R = len(Xr)
    rep_idx = rng.integers(0, R, size=R)

    acc = None

    for ri in rep_idx:
        X = Xr[ri]
        B = X.shape[0]
        blk_idx = rng.integers(0, B, size=B)
        mu = X[blk_idx].mean(axis=0)

        if acc is None:
            acc = np.zeros_like(mu)
        acc += mu

    return acc / len(rep_idx)


def bootstrap_system(Xr, n_boot, seed):
    rng = np.random.default_rng(seed)

    N = Xr[0].shape[1]
    M = Xr[0].shape[2]

    boot = np.empty((n_boot, N, M))

    for b in range(n_boot):
        boot[b] = system_boot_mean(Xr, rng)

        if (b + 1) % max(1, n_boot // 10) == 0:
            print(f"[bootstrap] {b+1}/{n_boot}")

    mean = boot.mean(axis=0)
    ci_low = np.quantile(boot, 0.025, axis=0)
    ci_high = np.quantile(boot, 0.975, axis=0)
    std = boot.std(axis=0, ddof=1)
    sem = std / np.sqrt(n_boot)

    return mean, ci_low, ci_high, std, sem


# ============================================================
# Main processing
# ============================================================

def process_system(base_dir, system, reps, args):
    Xr = []
    nodes_all = []
    paths = []

    for rep in reps:
        path = os.path.join(
            base_dir,
            args.analysis_root,
            args.subdir,
            args.variant,
            "REPLICA_BLOCK",
            f"{system}_rep{rep}",
            "descriptors_by_block.csv"
        )

        if not os.path.exists(path):
            raise FileNotFoundError(path)

        nodes, X = load_replica(path)

        Xr.append(X)
        nodes_all.append(nodes)
        paths.append(path)

    # consistency check
    nodes = nodes_all[0]
    for n in nodes_all[1:]:
        if not np.array_equal(nodes, n):
            raise RuntimeError("Node mismatch across replicas")

    # replica summary
    rep_rows = []
    for rep, X in zip(reps, Xr):
        mu = X.mean(axis=0)
        sd = X.std(axis=0, ddof=1)

        for ni, node_id in enumerate(nodes):
            row = {"rep_id": rep, "node_id": int(node_id), "residue": int(node_id)+1}
            for mi, met in enumerate(METRICS):
                row[f"{met}_mean"] = float(mu[ni, mi])
                row[f"{met}_sd_blocks"] = float(sd[ni, mi])
                row[f"{met}_n_blocks"] = int(X.shape[0])
            rep_rows.append(row)

    # bootstrap
    mean, ci_low, ci_high, std, sem = bootstrap_system(
        Xr, args.n_boot, args.seed
    )

    # system summary
    sys_rows = []
    for ni, node_id in enumerate(nodes):
        row = {"node_id": int(node_id), "residue": int(node_id)+1}
        for mi, met in enumerate(METRICS):
            row[f"{met}_mean"] = float(mean[ni, mi])
            row[f"{met}_ci_low"] = float(ci_low[ni, mi])
            row[f"{met}_ci_high"] = float(ci_high[ni, mi])
            row[f"{met}_std_bootstrap"] = float(std[ni, mi])
            row[f"{met}_sem_bootstrap"] = float(sem[ni, mi])
        sys_rows.append(row)

    out_dir = os.path.join(
        base_dir,
        args.analysis_root,
        args.subdir,
        args.variant,
        "SYSTEM",
        system
    )
    os.makedirs(out_dir, exist_ok=True)

    pd.DataFrame(rep_rows).to_csv(
        os.path.join(out_dir, "replica_descriptors.csv"),
        index=False
    )

    pd.DataFrame(sys_rows).to_csv(
        os.path.join(out_dir, "system_descriptors.csv"),
        index=False
    )

    with open(os.path.join(out_dir, "run_config.txt"), "w") as f:
        f.write(json.dumps({
            "system": system,
            "reps": reps,
            "n_boot": args.n_boot,
            "seed": args.seed,
            "inputs": paths
        }, indent=2))

    print(f"[DONE] {system} -> {out_dir}")


# ============================================================
# CLI
# ============================================================

def main():
    ap = argparse.ArgumentParser()

    ap.add_argument("--base_dir", required=True)
    ap.add_argument("--systems", default="WT,NAM")
    ap.add_argument("--reps", default="1,2,3")

    ap.add_argument("--analysis_root", default="ANALYSIS")
    ap.add_argument("--subdir", default="DESCRIPTOR_PIPELINE")
    ap.add_argument("--variant", default="all_sum")

    ap.add_argument("--n_boot", type=int, default=2000)
    ap.add_argument("--seed", type=int, default=404)

    args = ap.parse_args()

    systems = [x.strip() for x in args.systems.split(",")]
    reps = [int(x) for x in args.reps.split(",")]

    for system in systems:
        process_system(args.base_dir, system, reps, args)

    print("\nALL DONE")


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/aggregate_descriptor_pipeline.py


In [ ]:
!python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/aggregate_descriptor_pipeline.py \
  --base_dir "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR" \
  --systems "WT,NAM" \
  --reps "1,2,3" \
  --variant all_sum \
  --n_boot 2000

[bootstrap] 200/2000
[bootstrap] 400/2000
[bootstrap] 600/2000
[bootstrap] 800/2000
[bootstrap] 1000/2000
[bootstrap] 1200/2000
[bootstrap] 1400/2000
[bootstrap] 1600/2000
[bootstrap] 1800/2000
[bootstrap] 2000/2000
[DONE] WT -> /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/all_sum/SYSTEM/WT
[bootstrap] 200/2000
[bootstrap] 400/2000
[bootstrap] 600/2000
[bootstrap] 800/2000
[bootstrap] 1000/2000
[bootstrap] 1200/2000
[bootstrap] 1400/2000
[bootstrap] 1600/2000
[bootstrap] 1800/2000
[bootstrap] 2000/2000
[DONE] NAM -> /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/all_sum/SYSTEM/NAM

ALL DONE


In [ ]:
%%writefile /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/delta_descriptor_pipeline.py

#!/usr/bin/env python3

import os
import json
import argparse
import numpy as np
import pandas as pd

METRICS = ["strength", "betweenness", "closeness", "pagerank", "eigenvector"]


def load_replica(csv_path):
    df = pd.read_csv(csv_path)

    needed = ["block_id", "node_id"] + METRICS
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} missing columns: {missing}")

    blocks = np.sort(df["block_id"].unique())
    nodes = np.sort(df["node_id"].unique())

    B, N, M = len(blocks), len(nodes), len(METRICS)
    X = np.full((B, N, M), np.nan)

    bmap = {b: i for i, b in enumerate(blocks)}
    nmap = {n: i for i, n in enumerate(nodes)}

    for _, row in df.iterrows():
        bi = bmap[row["block_id"]]
        ni = nmap[row["node_id"]]
        X[bi, ni, :] = row[METRICS].values

    if np.isnan(X).any():
        raise RuntimeError(f"{csv_path} contains NaNs after pivot")

    return nodes, X


def load_system_replicas(base_dir, args, system, reps):
    Xr = []
    nodes_all = []
    paths = []

    for rep in reps:
        path = os.path.join(
            base_dir,
            args.analysis_root,
            args.subdir,
            args.variant,
            "REPLICA_BLOCK",
            f"{system}_rep{rep}",
            "descriptors_by_block.csv",
        )

        if not os.path.exists(path):
            raise FileNotFoundError(path)

        nodes, X = load_replica(path)
        Xr.append(X)
        nodes_all.append(nodes)
        paths.append(path)

    nodes = nodes_all[0]
    for n in nodes_all[1:]:
        if not np.array_equal(nodes, n):
            raise RuntimeError(f"Node mismatch inside {system}")

    return nodes, Xr, paths


def system_boot_mean(Xr, rng):
    R = len(Xr)
    rep_idx = rng.integers(0, R, size=R)

    acc = None

    for ri in rep_idx:
        X = Xr[ri]
        B = X.shape[0]
        blk_idx = rng.integers(0, B, size=B)
        mu = X[blk_idx].mean(axis=0)

        if acc is None:
            acc = np.zeros_like(mu)
        acc += mu

    return acc / len(rep_idx)


def bootstrap_delta(wt_Xr, nam_Xr, n_boot, seed):
    rng = np.random.default_rng(seed)

    N = wt_Xr[0].shape[1]
    M = wt_Xr[0].shape[2]

    boot = np.empty((n_boot, N, M))

    for b in range(n_boot):
        wt_mu = system_boot_mean(wt_Xr, rng)
        nm_mu = system_boot_mean(nam_Xr, rng)
        boot[b] = wt_mu - nm_mu

        if (b + 1) % max(1, n_boot // 10) == 0:
            print(f"[bootstrap] {b+1}/{n_boot}")

    mean = boot.mean(axis=0)
    low = np.quantile(boot, 0.025, axis=0)
    high = np.quantile(boot, 0.975, axis=0)
    std = boot.std(axis=0, ddof=1)
    sem = std / np.sqrt(n_boot)

    return mean, low, high, std, sem


def write_delta_summary(out_dir, nodes, mean, low, high, std, sem):
    rows = []

    for ni, node_id in enumerate(nodes):
        row = {
            "node_id": int(node_id),
            "residue": int(node_id) + 1,
        }

        for mi, met in enumerate(METRICS):
            row[f"delta_{met}_mean"] = float(mean[ni, mi])
            row[f"delta_{met}_ci_low"] = float(low[ni, mi])
            row[f"delta_{met}_ci_high"] = float(high[ni, mi])
            row[f"delta_{met}_std_bootstrap"] = float(std[ni, mi])
            row[f"delta_{met}_sem_bootstrap"] = float(sem[ni, mi])
            row[f"delta_{met}_significant"] = bool((low[ni, mi] > 0) or (high[ni, mi] < 0))

        rows.append(row)

    pd.DataFrame(rows).to_csv(
        os.path.join(out_dir, "delta_descriptors.csv"),
        index=False,
    )


def write_descriptor_significance_summary(out_dir):
    df = pd.read_csv(os.path.join(out_dir, "delta_descriptors.csv"))

    rows = []

    for met in METRICS:
        mean_col = f"delta_{met}_mean"
        lo_col = f"delta_{met}_ci_low"
        hi_col = f"delta_{met}_ci_high"
        std_col = f"delta_{met}_std_bootstrap"
        sig_col = f"delta_{met}_significant"

        d = df.copy()
        d["abs_delta"] = d[mean_col].abs()
        d["ci_width"] = d[hi_col] - d[lo_col]

        sig = d[d[sig_col] == True].copy()

        if len(sig) > 0:
            median_abs_sig = float(sig["abs_delta"].median())
            mean_abs_sig = float(sig["abs_delta"].mean())
            median_ci_sig = float(sig["ci_width"].median())
            median_std_sig = float(sig[std_col].median())
        else:
            median_abs_sig = np.nan
            mean_abs_sig = np.nan
            median_ci_sig = np.nan
            median_std_sig = np.nan

        rows.append({
            "descriptor": met,
            "n_significant_nodes": int(sig_col in d.columns and d[sig_col].sum()),
            "fraction_significant_nodes": float(d[sig_col].mean()),
            "median_abs_delta_significant": median_abs_sig,
            "mean_abs_delta_significant": mean_abs_sig,
            "median_ci_width_significant": median_ci_sig,
            "median_std_bootstrap_significant": median_std_sig,
            "median_abs_delta_all_nodes": float(d["abs_delta"].median()),
            "mean_abs_delta_all_nodes": float(d["abs_delta"].mean()),
            "median_ci_width_all_nodes": float(d["ci_width"].median()),
            "median_std_bootstrap_all_nodes": float(d[std_col].median()),
        })

    pd.DataFrame(rows).to_csv(
        os.path.join(out_dir, "descriptor_significance_summary.csv"),
        index=False,
    )


def main():
    ap = argparse.ArgumentParser()

    ap.add_argument("--base_dir", required=True)
    ap.add_argument("--analysis_root", default="ANALYSIS")
    ap.add_argument("--subdir", default="DESCRIPTOR_PIPELINE")
    ap.add_argument("--variant", default="all_sum")

    ap.add_argument("--wt_reps", default="1,2,3")
    ap.add_argument("--nam_reps", default="1,2,3")

    ap.add_argument("--n_boot", type=int, default=2000)
    ap.add_argument("--seed", type=int, default=505)

    args = ap.parse_args()

    wt_reps = [int(x.strip()) for x in args.wt_reps.split(",") if x.strip()]
    nam_reps = [int(x.strip()) for x in args.nam_reps.split(",") if x.strip()]

    wt_nodes, wt_Xr, wt_paths = load_system_replicas(
        args.base_dir, args, "WT", wt_reps
    )
    nam_nodes, nam_Xr, nam_paths = load_system_replicas(
        args.base_dir, args, "NAM", nam_reps
    )

    if not np.array_equal(wt_nodes, nam_nodes):
        raise RuntimeError("WT and NAM node IDs do not match")

    out_dir = os.path.join(
        args.base_dir,
        args.analysis_root,
        args.subdir,
        args.variant,
        "DELTA",
    )
    os.makedirs(out_dir, exist_ok=True)

    mean, low, high, std, sem = bootstrap_delta(
        wt_Xr,
        nam_Xr,
        n_boot=args.n_boot,
        seed=args.seed,
    )

    write_delta_summary(out_dir, wt_nodes, mean, low, high, std, sem)
    write_descriptor_significance_summary(out_dir)

    cfg = {
        "variant": args.variant,
        "wt_reps": wt_reps,
        "nam_reps": nam_reps,
        "n_boot": args.n_boot,
        "seed": args.seed,
        "metrics": METRICS,
        "wt_inputs": wt_paths,
        "nam_inputs": nam_paths,
        "note": "Delta = WT - NAM. CI/STD/SEM computed from hierarchical bootstrap samples.",
    }

    with open(os.path.join(out_dir, "run_config.txt"), "w") as f:
        f.write(json.dumps(cfg, indent=2))

    print("\nDONE.")
    print("Output:", out_dir)


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/delta_descriptor_pipeline.py


In [ ]:
!python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/delta_descriptor_pipeline.py \
  --base_dir "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR" \
  --variant all_sum \
  --wt_reps "1,2,3" \
  --nam_reps "1,2,3" \
  --n_boot 2000 \
  --seed 505

[bootstrap] 200/2000
[bootstrap] 400/2000
[bootstrap] 600/2000
[bootstrap] 800/2000
[bootstrap] 1000/2000
[bootstrap] 1200/2000
[bootstrap] 1400/2000
[bootstrap] 1600/2000
[bootstrap] 1800/2000
[bootstrap] 2000/2000

DONE.
Output: /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/all_sum/DELTA


In [ ]:
%%writefile /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/merge_all_descriptors_pipeline.py

#!/usr/bin/env python3

import os
import json
import argparse
import numpy as np
import pandas as pd


GRAPH_DESCRIPTORS = [
    "strength",
    "betweenness",
    "closeness",
    "pagerank",
    "eigenvector",
]

MERGED_DESCRIPTORS = [
    "node_weight",
    "strength",
    "betweenness",
    "closeness",
    "pagerank",
    "eigenvector",
    "MVA",
]


# ============================================================
# Load files
# ============================================================

def load_descriptor(csv_path):
    df = pd.read_csv(csv_path)

    required = ["block_id", "node_id"] + GRAPH_DESCRIPTORS
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} missing columns: {missing}")

    keep = ["block_id", "node_id", "residue"] + GRAPH_DESCRIPTORS
    keep = [c for c in keep if c in df.columns]

    return df[keep].copy()


def load_nodeweight(csv_path):
    df = pd.read_csv(csv_path)

    if "node_weight_block_mean" in df.columns:
        value_col = "node_weight_block_mean"
    elif "node_weight" in df.columns:
        value_col = "node_weight"
    elif "weight_mean" in df.columns:
        value_col = "weight_mean"
    else:
        raise ValueError(
            f"Could not find node-weight column in {csv_path}. "
            f"Columns found: {list(df.columns)}"
        )

    required = ["block_id", "node_id", value_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} missing columns: {missing}")

    return df[["block_id", "node_id", value_col]].rename(
        columns={value_col: "node_weight"}
    )


def merge_descriptor_nodeweight(desc_df, nw_df):
    df = desc_df.merge(nw_df, on=["block_id", "node_id"], how="inner")

    if len(df) == 0:
        raise RuntimeError("Merge produced empty dataframe.")

    if "residue" not in df.columns:
        df["residue"] = df["node_id"].astype(int) + 1

    df["MVA"] = df["node_weight"] * df["strength"]

    return df


# ============================================================
# Tensor construction
# ============================================================

def load_replica(base_dir, system, rep, args):
    system = system.upper()

    desc_path = os.path.join(
        base_dir,
        args.analysis_root,
        args.descriptor_subdir,
        args.variant,
        "REPLICA_BLOCK",
        f"{system}_rep{rep}",
        "descriptors_by_block.csv",
    )

    nw_path = os.path.join(
        base_dir,
        args.analysis_root,
        args.nodeweight_subdir,
        f"{system}_rep{rep}",
        "node_attention_weights_by_block.csv",
    )

    if not os.path.exists(desc_path):
        raise FileNotFoundError(desc_path)
    if not os.path.exists(nw_path):
        raise FileNotFoundError(nw_path)

    desc_df = load_descriptor(desc_path)
    nw_df = load_nodeweight(nw_path)
    merged = merge_descriptor_nodeweight(desc_df, nw_df)

    blocks = np.sort(merged["block_id"].unique())
    nodes = np.sort(merged["node_id"].unique())

    B = len(blocks)
    N = len(nodes)
    M = len(MERGED_DESCRIPTORS)

    X = np.full((B, N, M), np.nan, dtype=float)

    bmap = {int(b): i for i, b in enumerate(blocks)}
    nmap = {int(n): i for i, n in enumerate(nodes)}

    for _, row in merged.iterrows():
        bi = bmap[int(row["block_id"])]
        ni = nmap[int(row["node_id"])]
        X[bi, ni, :] = row[MERGED_DESCRIPTORS].to_numpy(dtype=float)

    if np.isnan(X).any():
        raise RuntimeError(
            f"NaNs after tensor construction for {system}_rep{rep}. "
            f"NaNs={np.isnan(X).sum()}"
        )

    return nodes, X, desc_path, nw_path


def check_nodes_match(nodes_list):
    base = nodes_list[0]
    for n in nodes_list[1:]:
        if not np.array_equal(base, n):
            raise RuntimeError("Node IDs do not match across replicas/systems.")
    return base


# ============================================================
# Bootstrap
# ============================================================

def system_boot_mean(Xr, rng):
    R = len(Xr)
    rep_idx = rng.integers(0, R, size=R)

    acc = None

    for ri in rep_idx:
        X = Xr[ri]
        B = X.shape[0]

        block_idx = rng.integers(0, B, size=B)
        mu_rep = X[block_idx, :, :].mean(axis=0)

        if acc is None:
            acc = np.zeros_like(mu_rep)

        acc += mu_rep

    return acc / len(rep_idx)


def bootstrap_system_and_delta(wt_Xr, nam_Xr, n_boot, seed):
    rng = np.random.default_rng(seed)

    N = wt_Xr[0].shape[1]
    M = wt_Xr[0].shape[2]

    boot_wt = np.empty((n_boot, N, M), dtype=float)
    boot_nam = np.empty((n_boot, N, M), dtype=float)
    boot_delta = np.empty((n_boot, N, M), dtype=float)

    for b in range(n_boot):
        wt_mu = system_boot_mean(wt_Xr, rng)
        nam_mu = system_boot_mean(nam_Xr, rng)

        boot_wt[b, :, :] = wt_mu
        boot_nam[b, :, :] = nam_mu
        boot_delta[b, :, :] = wt_mu - nam_mu

        if (b + 1) % max(1, n_boot // 10) == 0:
            print(f"[bootstrap] {b+1}/{n_boot}")

    def summarize(boot):
        mean = boot.mean(axis=0)
        low = np.quantile(boot, 0.025, axis=0)
        high = np.quantile(boot, 0.975, axis=0)
        std = boot.std(axis=0, ddof=1)
        sem = std / np.sqrt(n_boot)
        return mean, low, high, std, sem

    return summarize(boot_wt), summarize(boot_nam), summarize(boot_delta)


# ============================================================
# Output writers
# ============================================================

def write_system_output(out_path, nodes, stats, system_label):
    mean, low, high, std, sem = stats

    rows = []

    for ni, node_id in enumerate(nodes):
        row = {
            "node_id": int(node_id),
            "residue": int(node_id) + 1,
        }

        for mi, desc in enumerate(MERGED_DESCRIPTORS):
            row[f"{desc}_mean"] = float(mean[ni, mi])
            row[f"{desc}_ci_low"] = float(low[ni, mi])
            row[f"{desc}_ci_high"] = float(high[ni, mi])
            row[f"{desc}_std_bootstrap"] = float(std[ni, mi])
            row[f"{desc}_sem_bootstrap"] = float(sem[ni, mi])

        rows.append(row)

    pd.DataFrame(rows).to_csv(out_path, index=False)


def write_delta_output(out_path, nodes, stats):
    mean, low, high, std, sem = stats

    rows = []

    for ni, node_id in enumerate(nodes):
        row = {
            "node_id": int(node_id),
            "residue": int(node_id) + 1,
        }

        for mi, desc in enumerate(MERGED_DESCRIPTORS):
            row[f"delta_{desc}_mean"] = float(mean[ni, mi])
            row[f"delta_{desc}_ci_low"] = float(low[ni, mi])
            row[f"delta_{desc}_ci_high"] = float(high[ni, mi])
            row[f"delta_{desc}_std_bootstrap"] = float(std[ni, mi])
            row[f"delta_{desc}_sem_bootstrap"] = float(sem[ni, mi])
            row[f"delta_{desc}_significant"] = bool(
                (low[ni, mi] > 0) or (high[ni, mi] < 0)
            )

        rows.append(row)

    pd.DataFrame(rows).to_csv(out_path, index=False)


def write_long_delta_output(out_path, wide_csv):
    df = pd.read_csv(wide_csv)

    rows = []

    for _, row in df.iterrows():
        for desc in MERGED_DESCRIPTORS:
            rows.append({
                "node_id": int(row["node_id"]),
                "residue": int(row["residue"]),
                "descriptor": desc,
                "delta_mean": float(row[f"delta_{desc}_mean"]),
                "delta_ci_low": float(row[f"delta_{desc}_ci_low"]),
                "delta_ci_high": float(row[f"delta_{desc}_ci_high"]),
                "delta_std_bootstrap": float(row[f"delta_{desc}_std_bootstrap"]),
                "delta_sem_bootstrap": float(row[f"delta_{desc}_sem_bootstrap"]),
                "significant": bool(row[f"delta_{desc}_significant"]),
            })

    pd.DataFrame(rows).to_csv(out_path, index=False)


def write_summary(out_path, delta_wide_csv):
    df = pd.read_csv(delta_wide_csv)
    rows = []

    for desc in MERGED_DESCRIPTORS:
        mean_col = f"delta_{desc}_mean"
        lo_col = f"delta_{desc}_ci_low"
        hi_col = f"delta_{desc}_ci_high"
        std_col = f"delta_{desc}_std_bootstrap"
        sig_col = f"delta_{desc}_significant"

        d = df.copy()
        d["abs_delta"] = d[mean_col].abs()
        d["ci_width"] = d[hi_col] - d[lo_col]
        sig = d[d[sig_col] == True].copy()

        rows.append({
            "descriptor": desc,
            "n_significant_nodes": int(sig.shape[0]),
            "fraction_significant_nodes": float(sig.shape[0] / d.shape[0]),
            "median_abs_delta_all_nodes": float(d["abs_delta"].median()),
            "mean_abs_delta_all_nodes": float(d["abs_delta"].mean()),
            "median_ci_width_all_nodes": float(d["ci_width"].median()),
            "median_std_bootstrap_all_nodes": float(d[std_col].median()),
            "median_abs_delta_significant": float(sig["abs_delta"].median()) if len(sig) else np.nan,
            "mean_abs_delta_significant": float(sig["abs_delta"].mean()) if len(sig) else np.nan,
            "median_ci_width_significant": float(sig["ci_width"].median()) if len(sig) else np.nan,
            "median_std_bootstrap_significant": float(sig[std_col].median()) if len(sig) else np.nan,
        })

    pd.DataFrame(rows).to_csv(out_path, index=False)


# ============================================================
# Main
# ============================================================

def main():
    ap = argparse.ArgumentParser()

    ap.add_argument("--base_dir", required=True)
    ap.add_argument("--analysis_root", default="ANALYSIS")

    ap.add_argument("--descriptor_subdir", default="DESCRIPTOR_PIPELINE")
    ap.add_argument("--nodeweight_subdir", default="NODE_WEIGHTS")
    ap.add_argument("--variant", default="all_sum")

    ap.add_argument("--wt_reps", default="1,2,3")
    ap.add_argument("--nam_reps", default="1,2,3")

    ap.add_argument("--n_boot", type=int, default=2000)
    ap.add_argument("--seed", type=int, default=606)

    args = ap.parse_args()

    wt_reps = [int(x.strip()) for x in args.wt_reps.split(",") if x.strip()]
    nam_reps = [int(x.strip()) for x in args.nam_reps.split(",") if x.strip()]

    wt_Xr = []
    nam_Xr = []
    nodes_all = []
    input_records = []

    for rep in wt_reps:
        nodes, X, desc_path, nw_path = load_replica(args.base_dir, "WT", rep, args)
        wt_Xr.append(X)
        nodes_all.append(nodes)
        input_records.append({"system": "WT", "rep": rep, "descriptor": desc_path, "node_weight": nw_path})

    for rep in nam_reps:
        nodes, X, desc_path, nw_path = load_replica(args.base_dir, "NAM", rep, args)
        nam_Xr.append(X)
        nodes_all.append(nodes)
        input_records.append({"system": "NAM", "rep": rep, "descriptor": desc_path, "node_weight": nw_path})

    nodes = check_nodes_match(nodes_all)

    wt_stats, nam_stats, delta_stats = bootstrap_system_and_delta(
        wt_Xr,
        nam_Xr,
        n_boot=args.n_boot,
        seed=args.seed,
    )

    out_dir = os.path.join(
        args.base_dir,
        args.analysis_root,
        args.descriptor_subdir,
        args.variant,
        "MERGED_ALL_DESCRIPTORS",
    )
    os.makedirs(out_dir, exist_ok=True)

    wt_csv = os.path.join(out_dir, "WT_all_descriptors.csv")
    nam_csv = os.path.join(out_dir, "NAM_all_descriptors.csv")
    delta_csv = os.path.join(out_dir, "delta_all_descriptors.csv")
    delta_long_csv = os.path.join(out_dir, "delta_all_descriptors_long.csv")
    summary_csv = os.path.join(out_dir, "descriptor_significance_summary.csv")

    write_system_output(wt_csv, nodes, wt_stats, "WT")
    write_system_output(nam_csv, nodes, nam_stats, "NAM")
    write_delta_output(delta_csv, nodes, delta_stats)
    write_long_delta_output(delta_long_csv, delta_csv)
    write_summary(summary_csv, delta_csv)

    cfg = {
        "variant": args.variant,
        "descriptors": MERGED_DESCRIPTORS,
        "wt_reps": wt_reps,
        "nam_reps": nam_reps,
        "n_boot": args.n_boot,
        "seed": args.seed,
        "inputs": input_records,
        "definitions": {
            "node_weight": "masking/node-weight block mean from NODE_WEIGHTS outputs",
            "strength": "attention-based degree centrality: sum of all_sum attention-weighted incident edges",
            "betweenness": "weighted shortest-path betweenness using distance=1/(attention+eps)",
            "closeness": "weighted closeness using distance=1/(attention+eps)",
            "pagerank": "weighted PageRank using attention as edge weight",
            "eigenvector": "weighted eigenvector centrality using attention as edge weight",
            "MVA": "model-weighted attention = node_weight × strength",
        },
        "note": "CI/STD/SEM computed from hierarchical bootstrap over replicas and blocks.",
    }

    with open(os.path.join(out_dir, "run_config.txt"), "w") as f:
        f.write(json.dumps(cfg, indent=2))

    print("\nDONE.")
    print("Outputs written to:", out_dir)
    print("Main delta:", delta_csv)
    print("Summary:", summary_csv)


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/merge_all_descriptors_pipeline.py


In [ ]:
!python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/merge_all_descriptors_pipeline.py \
  --base_dir "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR" \
  --variant all_sum \
  --wt_reps "1,2,3" \
  --nam_reps "1,2,3" \
  --n_boot 2000 \
  --seed 606

[bootstrap] 200/2000
[bootstrap] 400/2000
[bootstrap] 600/2000
[bootstrap] 800/2000
[bootstrap] 1000/2000
[bootstrap] 1200/2000
[bootstrap] 1400/2000
[bootstrap] 1600/2000
[bootstrap] 1800/2000
[bootstrap] 2000/2000

DONE.
Outputs written to: /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/all_sum/MERGED_ALL_DESCRIPTORS
Main delta: /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/all_sum/MERGED_ALL_DESCRIPTORS/delta_all_descriptors.csv
Summary: /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/all_sum/MERGED_ALL_DESCRIPTORS/descriptor_significance_summary.csv


In [ ]:
%%writefile /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/plot_residue_delta_descriptors.py

#!/usr/bin/env python3

import os
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DESCRIPTORS = [
    "node_weight",
    "strength",
    "betweenness",
    "closeness",
    "pagerank",
    "eigenvector",
    "MVA",
]


def asymmetric_error(mean, low, high):
    lower = np.maximum(0.0, mean - low)
    upper = np.maximum(0.0, high - mean)
    return np.vstack([lower, upper])


def plot_descriptor_delta(df, desc, out_png, plot_all=True, top_n=50):
    mean_col = f"delta_{desc}_mean"
    low_col = f"delta_{desc}_ci_low"
    high_col = f"delta_{desc}_ci_high"
    sig_col = f"delta_{desc}_significant"

    d = df.copy()

    if not plot_all:
        d["_rank"] = d[mean_col].abs()
        d = d.sort_values("_rank", ascending=False).head(top_n)
        d = d.sort_values("residue")

    x = d["residue"].values.astype(int)
    mean = d[mean_col].values.astype(float)
    low = d[low_col].values.astype(float)
    high = d[high_col].values.astype(float)
    sig = d[sig_col].values.astype(bool)

    yerr = asymmetric_error(mean, low, high)

    plt.figure(figsize=(max(12, 0.045 * len(d)), 4.8))

    # non-significant
    idx_ns = np.where(~sig)[0]
    plt.bar(x[idx_ns], mean[idx_ns], alpha=0.55, label="Not significant")

    # significant
    idx_sig = np.where(sig)[0]
    plt.bar(x[idx_sig], mean[idx_sig], alpha=0.95, label="Significant")

    plt.errorbar(x, mean, yerr=yerr, fmt="none", capsize=2, linewidth=0.8)

    plt.axhline(0, linewidth=1)
    plt.xlabel("Residue")
    plt.ylabel(f"Δ {desc} (WT − NAM)")
    plt.title(f"Residue-wise Δ {desc} with 95% bootstrap CI")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def plot_significance_bar(summary_df, out_png):
    d = summary_df.copy()
    d = d.sort_values("n_significant_nodes", ascending=False)

    x = np.arange(len(d))
    y = d["n_significant_nodes"].values

    plt.figure(figsize=(9, 4.8))
    plt.bar(x, y)
    plt.xticks(x, d["descriptor"].values, rotation=45, ha="right")
    plt.ylabel("Number of significant residues")
    plt.title("Significant WT−NAM residue differences by descriptor")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def plot_fraction_significance_bar(summary_df, out_png):
    d = summary_df.copy()
    d = d.sort_values("fraction_significant_nodes", ascending=False)

    x = np.arange(len(d))
    y = d["fraction_significant_nodes"].values

    plt.figure(figsize=(9, 4.8))
    plt.bar(x, y)
    plt.xticks(x, d["descriptor"].values, rotation=45, ha="right")
    plt.ylabel("Fraction of significant residues")
    plt.title("Fraction of significant WT−NAM residue differences by descriptor")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def main():
    ap = argparse.ArgumentParser()

    ap.add_argument("--base_dir", required=True)
    ap.add_argument("--analysis_root", default="ANALYSIS")
    ap.add_argument("--subdir", default="DESCRIPTOR_PIPELINE")
    ap.add_argument("--variant", default="all_sum")

    ap.add_argument("--top_n", type=int, default=50)
    ap.add_argument("--plot_topN", action="store_true")

    args = ap.parse_args()

    in_dir = os.path.join(
        args.base_dir,
        args.analysis_root,
        args.subdir,
        args.variant,
        "MERGED_ALL_DESCRIPTORS",
    )

    delta_csv = os.path.join(in_dir, "delta_all_descriptors.csv")
    summary_csv = os.path.join(in_dir, "descriptor_significance_summary.csv")

    if not os.path.exists(delta_csv):
        raise FileNotFoundError(delta_csv)
    if not os.path.exists(summary_csv):
        raise FileNotFoundError(summary_csv)

    out_dir = os.path.join(in_dir, "PLOTS_RESIDUE")
    os.makedirs(out_dir, exist_ok=True)

    df = pd.read_csv(delta_csv)
    summary = pd.read_csv(summary_csv)

    # save sorted summary too
    summary.sort_values("n_significant_nodes", ascending=False).to_csv(
        os.path.join(out_dir, "descriptor_significance_summary_sorted.csv"),
        index=False
    )

    for desc in DESCRIPTORS:
        plot_descriptor_delta(
            df,
            desc,
            os.path.join(out_dir, f"delta_{desc}_all_residues.png"),
            plot_all=True,
            top_n=args.top_n,
        )

        if args.plot_topN:
            plot_descriptor_delta(
                df,
                desc,
                os.path.join(out_dir, f"delta_{desc}_top{args.top_n}_residue_sorted.png"),
                plot_all=False,
                top_n=args.top_n,
            )

    plot_significance_bar(
        summary,
        os.path.join(out_dir, "descriptor_significant_nodes_bar.png"),
    )

    plot_fraction_significance_bar(
        summary,
        os.path.join(out_dir, "descriptor_fraction_significant_nodes_bar.png"),
    )

    print("\nDONE.")
    print("Plots saved to:", out_dir)


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/plot_residue_delta_descriptors.py


In [ ]:
!python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/plot_residue_delta_descriptors.py \
  --base_dir "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR" \
  --variant all_sum \
  --top_n 50 \
  --plot_topN


DONE.
Plots saved to: /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/all_sum/MERGED_ALL_DESCRIPTORS/PLOTS_RESIDUE


In [ ]:
%%writefile /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/region_bootstrap_all_descriptors.py

#!/usr/bin/env python3

import os
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GRAPH_DESCRIPTORS = ["strength", "betweenness", "closeness", "pagerank", "eigenvector"]
DESCRIPTORS = ["node_weight", "strength", "betweenness", "closeness", "pagerank", "eigenvector", "MVA"]

REGIONS = {
    "Loop": list(range(5, 8)) + [9],
    "Ful": list(range(10, 23)),
    "AS": [8] + list(range(23, 31)) + [32, 76, 84],
    "Bur": [31, 33, 34, 77, 83, 85],
    "Elb": list(range(35, 43)),
    "Fl_ext": list(range(43, 47)) + list(range(55, 59)),
    "Flp": list(range(47, 55)),
    "UpC": list(range(59, 63)) + list(range(73, 76)),
    "LoC": list(range(63, 73)),
    "80s": list(range(78, 83)),
    "Hlx": list(range(86, 95)),
    "Ter": list(range(1, 5)) + list(range(95, 104)) + list(range(194, 199)),
    "Loop′": list(range(104, 107)) + [108],
    "Ful′": list(range(109, 122)),
    "AS′": [107] + list(range(122, 130)) + [131, 175, 183],
    "Bur′": [130, 132, 133, 176, 182, 184],
    "Elb′": list(range(134, 142)),
    "Fl_ext′": list(range(142, 146)) + list(range(154, 158)),
    "Flp′": list(range(146, 154)),
    "UpC′": list(range(158, 162)) + list(range(172, 175)),
    "LoC′": list(range(162, 172)),
    "80s′": list(range(177, 182)),
    "Hlx′": list(range(185, 194)),
}


def load_nodeweight(path):
    df = pd.read_csv(path)

    if "node_weight_block_mean" in df.columns:
        col = "node_weight_block_mean"
    elif "node_weight" in df.columns:
        col = "node_weight"
    elif "weight_mean" in df.columns:
        col = "weight_mean"
    else:
        raise ValueError(f"No node-weight column found in {path}. Columns: {list(df.columns)}")

    return df[["block_id", "node_id", col]].rename(columns={col: "node_weight"})


def load_replica(base_dir, system, rep, args):
    system = system.upper()

    desc_path = os.path.join(
        base_dir, args.analysis_root, args.subdir, args.variant,
        "REPLICA_BLOCK", f"{system}_rep{rep}", "descriptors_by_block.csv"
    )

    nw_path = os.path.join(
        base_dir, args.analysis_root, args.nodeweight_subdir,
        f"{system}_rep{rep}", "node_attention_weights_by_block.csv"
    )

    if not os.path.exists(desc_path):
        raise FileNotFoundError(desc_path)
    if not os.path.exists(nw_path):
        raise FileNotFoundError(nw_path)

    desc = pd.read_csv(desc_path)
    nw = load_nodeweight(nw_path)

    keep = ["block_id", "node_id", "residue"] + GRAPH_DESCRIPTORS
    keep = [c for c in keep if c in desc.columns]
    desc = desc[keep].copy()

    df = desc.merge(nw, on=["block_id", "node_id"], how="inner")

    if "residue" not in df.columns:
        df["residue"] = df["node_id"].astype(int) + 1

    df["MVA"] = df["node_weight"] * df["strength"]

    blocks = np.sort(df["block_id"].unique())
    nodes = np.sort(df["node_id"].unique())

    X = np.full((len(blocks), len(nodes), len(DESCRIPTORS)), np.nan)

    bmap = {int(b): i for i, b in enumerate(blocks)}
    nmap = {int(n): i for i, n in enumerate(nodes)}

    for _, row in df.iterrows():
        bi = bmap[int(row["block_id"])]
        ni = nmap[int(row["node_id"])]
        X[bi, ni, :] = row[DESCRIPTORS].to_numpy(dtype=float)

    if np.isnan(X).any():
        raise RuntimeError(f"NaNs after loading {system}_rep{rep}: {np.isnan(X).sum()}")

    return nodes, X


def check_nodes_match(nodes_list):
    base = nodes_list[0]
    for n in nodes_list[1:]:
        if not np.array_equal(base, n):
            raise RuntimeError("Node mismatch across systems/replicas.")
    return base


def region_indices(nodes):
    residue_to_index = {int(node) + 1: i for i, node in enumerate(nodes)}

    out = {}
    for reg, residues in REGIONS.items():
        idx = [residue_to_index[r] for r in residues if r in residue_to_index]
        if idx:
            out[reg] = idx
    return out


def count_significant_residues_by_region(delta_df):
    """
    Counts residue-level significant residues inside each region for every descriptor.
    Uses delta_all_descriptors.csv.
    """
    out = {}

    for reg, residues in REGIONS.items():
        sub = delta_df[delta_df["residue"].isin(residues)].copy()
        out[reg] = {}

        for desc in DESCRIPTORS:
            sig_col = f"delta_{desc}_significant"

            if sig_col not in sub.columns:
                raise ValueError(f"Missing column in delta file: {sig_col}")

            sig_sub = sub[sub[sig_col] == True]
            sig_residues = sorted(sig_sub["residue"].astype(int).tolist())

            out[reg][desc] = {
                "n_significant_residues_in_region": int(len(sig_residues)),
                "fraction_significant_residues_in_region": (
                    float(len(sig_residues) / len(sub)) if len(sub) > 0 else np.nan
                ),
                "significant_residues": ",".join(map(str, sig_residues)),
            }

    return out


def system_boot_mean(Xr, rng):
    R = len(Xr)
    rep_idx = rng.integers(0, R, size=R)

    acc = None
    for ri in rep_idx:
        X = Xr[ri]
        B = X.shape[0]
        blk_idx = rng.integers(0, B, size=B)
        mu = X[blk_idx, :, :].mean(axis=0)

        if acc is None:
            acc = np.zeros_like(mu)
        acc += mu

    return acc / len(rep_idx)


def plot_region(df, desc, out_png):
    d = df[df["descriptor"] == desc].copy()
    if d.empty:
        return

    x = np.arange(len(d))
    mean = d["delta_mean"].values
    low = d["delta_ci_low"].values
    high = d["delta_ci_high"].values
    sig = d["region_significant"].values.astype(bool)

    yerr = np.vstack([
        np.maximum(0, mean - low),
        np.maximum(0, high - mean),
    ])

    plt.figure(figsize=(max(11, 0.65 * len(d)), 4.8))

    idx_ns = np.where(~sig)[0]
    idx_s = np.where(sig)[0]

    plt.bar(x[idx_ns], mean[idx_ns], alpha=0.5, label="Region not significant")
    plt.bar(x[idx_s], mean[idx_s], alpha=0.95, label="Region significant")
    plt.errorbar(x, mean, yerr=yerr, fmt="none", capsize=3)

    plt.axhline(0, linewidth=1)
    plt.xticks(x, d["region"].values, rotation=45, ha="right")
    plt.ylabel(f"Region Δ {desc} (WT − NAM)")
    plt.title(f"Region-wise Δ {desc} with hierarchical bootstrap CI")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def plot_sig_regions(summary, out_png):
    d = summary.sort_values("n_significant_regions", ascending=False)

    plt.figure(figsize=(9, 4.8))
    plt.bar(np.arange(len(d)), d["n_significant_regions"].values)
    plt.xticks(np.arange(len(d)), d["descriptor"].values, rotation=45, ha="right")
    plt.ylabel("Number of significant regions")
    plt.title("Significant WT−NAM regions by descriptor")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def plot_regions_with_residue_signal(summary, out_png):
    d = summary.sort_values("n_regions_with_any_significant_residue", ascending=False)

    plt.figure(figsize=(9, 4.8))
    plt.bar(np.arange(len(d)), d["n_regions_with_any_significant_residue"].values)
    plt.xticks(np.arange(len(d)), d["descriptor"].values, rotation=45, ha="right")
    plt.ylabel("Regions containing ≥1 significant residue")
    plt.title("Regions containing residue-level WT−NAM signal")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base_dir", required=True)
    ap.add_argument("--analysis_root", default="ANALYSIS")
    ap.add_argument("--subdir", default="DESCRIPTOR_PIPELINE")
    ap.add_argument("--variant", default="all_sum")
    ap.add_argument("--nodeweight_subdir", default="NODE_WEIGHTS")
    ap.add_argument("--wt_reps", default="1,2,3")
    ap.add_argument("--nam_reps", default="1,2,3")
    ap.add_argument("--n_boot", type=int, default=2000)
    ap.add_argument("--seed", type=int, default=707)
    args = ap.parse_args()

    wt_reps = [int(x.strip()) for x in args.wt_reps.split(",") if x.strip()]
    nam_reps = [int(x.strip()) for x in args.nam_reps.split(",") if x.strip()]

    # Load residue-level delta file to count significant residues inside each region
    delta_residue_csv = os.path.join(
        args.base_dir,
        args.analysis_root,
        args.subdir,
        args.variant,
        "MERGED_ALL_DESCRIPTORS",
        "delta_all_descriptors.csv",
    )

    if not os.path.exists(delta_residue_csv):
        raise FileNotFoundError(delta_residue_csv)

    delta_residue_df = pd.read_csv(delta_residue_csv)
    sig_residue_info = count_significant_residues_by_region(delta_residue_df)

    wt_Xr, nam_Xr, nodes_all = [], [], []

    for rep in wt_reps:
        nodes, X = load_replica(args.base_dir, "WT", rep, args)
        wt_Xr.append(X)
        nodes_all.append(nodes)

    for rep in nam_reps:
        nodes, X = load_replica(args.base_dir, "NAM", rep, args)
        nam_Xr.append(X)
        nodes_all.append(nodes)

    nodes = check_nodes_match(nodes_all)
    reg_idx = region_indices(nodes)

    rng = np.random.default_rng(args.seed)

    boot = {
        reg: np.zeros((args.n_boot, len(DESCRIPTORS)), dtype=float)
        for reg in reg_idx
    }

    for b in range(args.n_boot):
        wt_mu = system_boot_mean(wt_Xr, rng)
        nam_mu = system_boot_mean(nam_Xr, rng)
        delta = wt_mu - nam_mu

        for reg, idx in reg_idx.items():
            boot[reg][b, :] = delta[idx, :].mean(axis=0)

        if (b + 1) % max(1, args.n_boot // 10) == 0:
            print(f"[bootstrap] {b+1}/{args.n_boot}")

    rows = []
    for reg in REGIONS:
        if reg not in boot:
            continue

        for mi, desc in enumerate(DESCRIPTORS):
            vals = boot[reg][:, mi]

            mean = float(vals.mean())
            low = float(np.quantile(vals, 0.025))
            high = float(np.quantile(vals, 0.975))
            std = float(vals.std(ddof=1))
            sem = float(std / np.sqrt(args.n_boot))

            res_sig = sig_residue_info[reg][desc]

            rows.append({
                "region": reg,
                "descriptor": desc,
                "n_residues_region": len(reg_idx[reg]),

                "n_significant_residues_in_region": res_sig["n_significant_residues_in_region"],
                "fraction_significant_residues_in_region": res_sig["fraction_significant_residues_in_region"],
                "significant_residues": res_sig["significant_residues"],

                "delta_mean": mean,
                "delta_ci_low": low,
                "delta_ci_high": high,
                "delta_std_bootstrap": std,
                "delta_sem_bootstrap": sem,

                "region_significant": bool((low > 0) or (high < 0)),
                "residues_used": ",".join(map(str, REGIONS[reg])),
            })

    out_dir = os.path.join(
        args.base_dir,
        args.analysis_root,
        args.subdir,
        args.variant,
        "MERGED_ALL_DESCRIPTORS",
        "REGION_BOOTSTRAP",
    )
    os.makedirs(out_dir, exist_ok=True)

    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(out_dir, "region_delta_all_descriptors.csv"), index=False)

    summary_rows = []
    for desc in DESCRIPTORS:
        sub = out[out["descriptor"] == desc].copy()

        reg_sig = sub[sub["region_significant"] == True]
        res_sig_regions = sub[sub["n_significant_residues_in_region"] > 0]

        summary_rows.append({
            "descriptor": desc,

            "n_significant_regions": int(len(reg_sig)),
            "fraction_significant_regions": float(len(reg_sig) / len(sub)) if len(sub) else np.nan,
            "region_significant_regions": ",".join(reg_sig["region"].tolist()),

            "total_significant_residues_across_regions": int(sub["n_significant_residues_in_region"].sum()),
            "n_regions_with_any_significant_residue": int(len(res_sig_regions)),
            "regions_with_any_significant_residue": ",".join(res_sig_regions["region"].tolist()),

            "median_fraction_significant_residues_per_region": float(
                sub["fraction_significant_residues_in_region"].median()
            ),
            "mean_fraction_significant_residues_per_region": float(
                sub["fraction_significant_residues_in_region"].mean()
            ),

            "median_abs_delta_all_regions": float(sub["delta_mean"].abs().median()),
            "median_ci_width_all_regions": float((sub["delta_ci_high"] - sub["delta_ci_low"]).median()),
            "median_std_bootstrap_all_regions": float(sub["delta_std_bootstrap"].median()),
        })

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(os.path.join(out_dir, "region_significance_summary.csv"), index=False)

    for desc in DESCRIPTORS:
        plot_region(out, desc, os.path.join(out_dir, f"region_delta_{desc}.png"))

    plot_sig_regions(summary, os.path.join(out_dir, "descriptor_significant_regions_bar.png"))
    plot_regions_with_residue_signal(
        summary,
        os.path.join(out_dir, "descriptor_regions_with_any_significant_residue_bar.png")
    )

    print("\nDONE.")
    print("Outputs:", out_dir)


if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/region_bootstrap_all_descriptors.py


In [ ]:
!python /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/region_bootstrap_all_descriptors.py \
  --base_dir "/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR" \
  --variant all_sum \
  --wt_reps "1,2,3" \
  --nam_reps "1,2,3" \
  --n_boot 2000 \
  --seed 707

[bootstrap] 200/2000
[bootstrap] 400/2000
[bootstrap] 600/2000
[bootstrap] 800/2000
[bootstrap] 1000/2000
[bootstrap] 1200/2000
[bootstrap] 1400/2000
[bootstrap] 1600/2000
[bootstrap] 1800/2000
[bootstrap] 2000/2000

DONE.
Outputs: /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/ANALYSIS/DESCRIPTOR_PIPELINE/all_sum/MERGED_ALL_DESCRIPTORS/REGION_BOOTSTRAP
